<a href="https://colab.research.google.com/github/Joammp/ML_Radio_Signal/blob/main/C%C3%B3pia_de_C%C3%B3pia_de_pt_86_1403.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IMPORTS

In [ ]:
!pip install scikeras
!pip install tensorflow

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np # Import numpy for potential future use

from matplotlib import pyplot as plt

import h5py
import json
from numpy import argwhere

from sklearn.model_selection import train_test_split
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
import threading # Usaremos threading para garantir a segurança da thread
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import numpy as np
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.model_selection import RandomizedSearchCV
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [ ]:
# Before creating the plot, set the desired font size and name
plt.rcParams['font.size'] = 12  # Change the default font size
plt.rcParams['font.family'] = 'serif'  # Change the default font family (e.g., 'serif', 'sans-serif', 'monospace')

# You can also change these settings for specific text elements if needed.
# For example, to change the title font size:
# plt.title('Plot Title', fontsize=16)


#IMPORT DATASET

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pinxau1000/radioml2018")

print("Path to dataset files:", path)

h5py_path = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'
modulation_classes_path = path + '/classes-fixed.json'
# Open the dataset
hdf5_file = h5py.File(h5py_path, 'r')
# Load the modulation classes. You can also copy and paste the content of classes-fixed.txt.
modulation_classes = json.load(open(modulation_classes_path, 'r'))

# Read the HDF5 groups
data = hdf5_file['X']
modulation_onehot = hdf5_file['Y']
snr = hdf5_file['Z']

100%|██████████| 18.0G/18.0G [04:11<00:00, 76.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/pinxau1000/radioml2018/versions/2


In [ ]:


# Load all data from the datasets in the HDF5 file
X = data
Y = modulation_onehot
Z = snr

# Convert one-hot encoded labels to numerical labels
y = np.argmax(Y, axis=1)

#DEFS


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEED GLOBAL — garante reprodutibilidade total entre execuções
# Cole esta célula como a PRIMEIRA célula de código do notebook
# ══════════════════════════════════════════════════════════════════════════════

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Garante operações determinísticas na GPU (pequeno custo de performance)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"✅ Seeds fixadas: SEED={SEED}")
print(f"   random     : OK")
print(f"   numpy      : OK")
print(f"   torch CPU  : OK")
print(f"   torch CUDA : OK")
print(f"   cudnn det. : OK")

✅ Seeds fixadas: SEED=42
   random     : OK
   numpy      : OK
   torch CPU  : OK
   torch CUDA : OK
   cudnn det. : OK


In [ ]:
import torch
import os
from tqdm import tqdm

def train_and_validate(model, train_loader, val_loader, criterion, optimizer,
                       scheduler=None, epochs=10, save_name="best_model"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 📁 Diretório padrão do Colab
    SAVE_DIR = "/content"
    os.makedirs(SAVE_DIR, exist_ok=True)

    best_val_acc = 0.0
    best_model_path = None
    epochs_no_improve = 0

    # 🔥 Early stopping baseado no scheduler
    if scheduler is not None and isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        early_stop_patience = scheduler.patience * 2
    else:
        early_stop_patience = 10

    print(f"🛑 Early Stopping patience: {early_stop_patience}")

    for epoch in range(epochs):

        # ── TREINO ──
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar_train = tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]")

        for inputs, labels in pbar_train:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # ── VALIDAÇÃO ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        # ── MÉTRICAS ──
        final_train_loss = train_loss / len(train_loader)
        final_train_acc  = 100. * train_correct / train_total
        final_val_loss   = val_loss / len(val_loader)
        final_val_acc    = 100. * val_correct / val_total

        current_lr = optimizer.param_groups[0]['lr']

        # ── CHECKPOINT MELHOR ──
        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            epochs_no_improve = 0

            # 🧹 remove modelo anterior
            if best_model_path and os.path.exists(best_model_path):
                os.remove(best_model_path)

            # 🏷 nome com métricas
            filename = (
                f"{save_name}_ep{epoch+1:03d}"
                f"_acc{final_val_acc:.2f}"
                f"_lr{current_lr:.2e}.pth"
            )

            best_model_path = os.path.join(SAVE_DIR, filename)

            torch.save({
                'epoch': epoch + 1,
                'val_acc': final_val_acc,
                'val_loss': final_val_loss,
                'lr': current_lr,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, best_model_path)

            print(f"💾 Novo melhor modelo salvo:")
            print(f"   📁 {best_model_path}")

        else:
            epochs_no_improve += 1

        # ── SCHEDULER ──
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(final_val_acc)  # 🔥 melhor usar ACC
            else:
                scheduler.step()

        # ── LOG ──
        print(f"\nÉpoca {epoch+1}: "
              f"Train Loss: {final_train_loss:.4f} | Train Acc: {final_train_acc:.2f}% | "
              f"Val Loss: {final_val_loss:.4f} | Val Acc: {final_val_acc:.2f}% | "
              f"LR: {current_lr:.2e}")

        # ── EARLY STOP ──
        if epochs_no_improve >= early_stop_patience:
            print(f"\n🛑 Early stopping ativado na época {epoch+1}")
            break

    print(f"\n✅ Melhor Val Acc: {best_val_acc:.2f}%")
    print(f"📁 Modelo salvo em: {best_model_path}")

    return best_model_path

In [ ]:


def get_indices_by_snr(target_snrs):
    """
    Separates and returns indices corresponding to specified SNR values.

    Args:
        target_snrs (list or np.ndarray): A list or array of SNR values
                                          (e.g., [-20, -10, 0, 10]) to filter by.

    Returns:
        np.ndarray: A sorted numpy array of indices from the original dataset (X, Y, Z, y)
                    where the SNR matches one of the target_snrs.
    """
    # Convert target_snrs to a set for efficient lookup
    target_snrs_set = set(target_snrs)

    # Get the SNR values from the Z array (assuming Z is globally available)
    # Z[:, 0] extracts the SNR column
    all_snr_values = Z[:, 0]

    # Find indices where the SNR value is in the target_snrs_set
    matching_indices = np.where(np.isin(all_snr_values, list(target_snrs_set)))[0]

    return np.sort(matching_indices)

# Example Usage:
# Let's say you want indices for SNR values of -20dB, 0dB, and 10dB
# desired_snrs = [-20, 0, 10]
# selected_snr_indices = get_indices_by_snr(desired_snrs)
#
# print(f"Found {len(selected_snr_indices)} samples with SNR in {desired_snrs}")
#
# # You can then use these indices to access the corresponding data
# # X_filtered_by_snr = X[selected_snr_indices]
# # y_filtered_by_snr = y[selected_snr_indices]
# # Z_filtered_by_snr = Z[selected_snr_indices]

# The previous content of this cell was likely a print statement.
# I'll add a simple print to confirm the function is defined.


In [ ]:
def get_samples_by_snr(target_snrs):
    """
    Returns data samples (X, Y, Z, y) corresponding to specified SNR values.

    Args:
        target_snrs (list or np.ndarray): A list or array of SNR values
                                          (e.g., [-20, -10, 0, 10]) to filter by.

    Returns:
        tuple: A tuple containing X_filtered, Y_filtered, Z_filtered, y_filtered
               for the selected SNR values.
    """
    # Use the existing function to get indices for the target SNRs
    selected_indices = get_indices_by_snr(target_snrs)

    # Use these indices to subset the global data arrays
    X_filtered = X[selected_indices]
    Y_filtered = Y[selected_indices]
    Z_filtered = Z[selected_indices]
    y_filtered = y[selected_indices]

    print(f"\nSubset created with {len(selected_indices)} samples for SNR values: {target_snrs}.")
    print(f"Shape of X_filtered: {X_filtered.shape}")
    print(f"Shape of Y_filtered: {Y_filtered.shape}")
    print(f"Shape of Z_filtered: {Z_filtered.shape}")
    print(f"Shape of y_filtered: {y_filtered.shape}")

    return X_filtered, Y_filtered, Z_filtered, y_filtered, selected_indices

print("Function 'get_samples_by_snr' defined.")

Function 'get_samples_by_snr' defined.


In [ ]:
def subset_snr (target_snrs, frac):
    X_snr, Y_snr, Z_snr, y_snn, selected_indices = get_samples_by_snr(target_snrs)

    ind_snr_frac, ind_lixo, Y_snr_frac, Ysnr_lixo = train_test_split(
        selected_indices,
        Y_snr,
        test_size=1-frac,
        random_state=42,
        stratify=Y_snr
    )

    ind_snr_frac = np.sort(ind_snr_frac)
    X_filtered = X[ind_snr_frac]
    Y_filtered = Y[ind_snr_frac]
    Z_filtered = Z[ind_snr_frac]
    y_filtered = y[ind_snr_frac]

    print(f"\nSubset created with {len(selected_indices)} samples for SNR values: {target_snrs}.")
    print(f"Shape of X_filtered: {X_filtered.shape}")
    print(f"Shape of Y_filtered: {Y_filtered.shape}")
    print(f"Shape of Z_filtered: {Z_filtered.shape}")
    print(f"Shape of y_filtered: {y_filtered.shape}")

    return X_filtered, Y_filtered, Z_filtered, y_filtered, ind_snr_frac





In [ ]:
def split (indices, lables):
  train_val_indices, test_indices, train_val_lables, test_lables = train_test_split(
        indices,
        lables,
        test_size=0.2,
        random_state=42,
        stratify=lables
    )

  train_indices, val_indices, train_lables, val_lables = train_test_split(
        train_val_indices,
        train_val_lables,
        test_size=0.25,
        random_state=42,
        stratify=train_val_lables
    )

  train_indices_sorted = np.sort(train_indices)
  val_indices_sorted = np.sort(val_indices)
  test_indices_sorted = np.sort(test_indices)

  return train_indices_sorted, val_indices_sorted, test_indices_sorted

#SUBSET HDF5

In [ ]:
import h5py
import numpy as np
import json
import os

# ── Configurações ─────────────────────────────────────────────────────────────

INPUT_FILE   = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'
CLASSES_FILE = path + '/classes-fixed.json'

DESIRED_SNRS = [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
#DESIRED_SNRS = [30]
# DESIRED_SNRS = [30]

BATCH_SIZE   = 2048
NUM_GROUPS   = 6

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}

for _dir in ['/kaggle/working', '/tmp', '/root']:
    if os.path.exists(_dir) and os.access(_dir, os.W_OK):
        snr_tag     = '_'.join(str(s) for s in DESIRED_SNRS)
        OUTPUT_FILE = os.path.join(_dir, f'subset_snr_{snr_tag}.hdf5')
        break

print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")
print(f"SNRs   : {DESIRED_SNRS}\n")

if os.path.exists(OUTPUT_FILE):
    try:
        os.remove(OUTPUT_FILE)
        print(f"Arquivo anterior removido: {OUTPUT_FILE}\n")
    except OSError as e:
        print(f"Erro ao remover {OUTPUT_FILE}: {e}")

# ── Lookup de grupos ──────────────────────────────────────────────────────────

modulation_classes = json.load(open(CLASSES_FILE, 'r'))
lookup = np.array([GROUP_MAP[modulation_classes[i]]
                   for i in range(len(modulation_classes))], dtype=np.int64)



# ── Passo 1: Identifica índices com os SNRs desejados ─────────────────────────

print("Passo 1/2 — Identificando índices dos SNRs selecionados...")

original_indices = []   # índices do dataset ORIGINAL — usados apenas para leitura
snr_set          = set(DESIRED_SNRS)

with h5py.File(INPUT_FILE, 'r') as src:
    N         = src['X'].shape[0]
    Z         = src['Z']
    n_batches = int(np.ceil(N / BATCH_SIZE))

    for i in range(n_batches):
        start, end = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, N)
        z_batch    = Z[start:end, 0]
        mask       = np.isin(z_batch, list(snr_set))
        global_idx = np.where(mask)[0] + start
        original_indices.extend(global_idx.tolist())

        if (i + 1) % 20 == 0 or i == n_batches - 1:
            print(f"  Batch {i+1:>4}/{n_batches}  | selecionados até agora: {len(original_indices)}")

original_indices = np.array(original_indices, dtype=np.int64)
M = len(original_indices)
print(f"\nTotal selecionado: {M} amostras de {N} ({100*M/N:.1f}%)\n")

# ── Passo 2: Cria novo arquivo com índices próprios a partir de 0 ─────────────

print("Passo 2/2 — Criando novo dataset com índices a partir de 0...")

with h5py.File(INPUT_FILE, 'r') as src:
    x_shape = src['X'].shape[1:]
    x_dtype = src['X'].dtype
    y_shape = src['Y'].shape[1:]
    y_dtype = src['Y'].dtype
    z_shape = src['Z'].shape[1:]
    z_dtype = src['Z'].dtype

    chunk = (min(BATCH_SIZE, M),)

    with h5py.File(OUTPUT_FILE, 'w') as out:

        # Grava metadado dos SNRs para rastreabilidade (não afeta indexação)
        out.attrs['snrs']            = DESIRED_SNRS
        out.attrs['source_file']     = INPUT_FILE
        out.attrs['total_samples']   = M




        ds_X  = out.create_dataset('X',         shape=(M,) + x_shape,
                                   dtype=x_dtype,    chunks=chunk + x_shape)

        ds_Y  = out.create_dataset('Y',         shape=(M,) + y_shape,
                                   dtype=y_dtype,    chunks=chunk + y_shape)

        ds_Z  = out.create_dataset('Z',         shape=(M,) + z_shape,
                                   dtype=z_dtype,    chunks=chunk + z_shape)

        ds_YG = out.create_dataset('Y_grouped', shape=(M, NUM_GROUPS),
                                   dtype=np.float32, chunks=chunk + (NUM_GROUPS,))

        n_batches = int(np.ceil(M / BATCH_SIZE))

        for i in range(n_batches):
            # new_start/new_end → índices do NOVO arquivo (começam em 0)
            new_start, new_end = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, M)

            # batch_idx → índices do arquivo ORIGINAL (apenas para leitura)
            batch_idx = original_indices[new_start:new_end].tolist()

            ds_X[new_start:new_end]  = src['X'][batch_idx]
            ds_Y[new_start:new_end]  = src['Y'][batch_idx]
            ds_Z[new_start:new_end]  = src['Z'][batch_idx]

            # Gera Y_grouped inline
            y_idx    = np.argmax(src['Y'][batch_idx], axis=1)
            y_group  = lookup[y_idx]
            YG_batch = np.zeros((new_end - new_start, NUM_GROUPS), dtype=np.float32)
            YG_batch[np.arange(new_end - new_start), y_group] = 1.0
            ds_YG[new_start:new_end] = YG_batch

            if (i + 1) % 10 == 0 or i == n_batches - 1:
                print(f"  Batch {i+1:>4}/{n_batches}  "
                      f"novo [{new_start:>8}:{new_end:>8}]  "
                      f"original [{batch_idx[0]:>8}:{batch_idx[-1]:>8}]")

# ── Deleta original_indices para não contaminar uso posterior ─────────────────

del original_indices
print("\noriginal_indices deletado do escopo — novo dataset usa apenas índices 0..M-1")

# ── Verificação ───────────────────────────────────────────────────────────────

print("\nVerificando arquivo gerado...")

with h5py.File(OUTPUT_FILE, 'r') as f:
    M_check = f['X'].shape[0]

    print(f"\n  Datasets : {list(f.keys())}")
    print(f"  Atributos: snrs={list(f.attrs['snrs'])}  "
          f"total_samples={f.attrs['total_samples']}")
    print(f"  X         : {f['X'].shape}  dtype={f['X'].dtype}")
    print(f"  Y         : {f['Y'].shape}  dtype={f['Y'].dtype}")
    print(f"  Z         : {f['Z'].shape}  dtype={f['Z'].dtype}")
    print(f"  Y_grouped : {f['Y_grouped'].shape}  dtype={f['Y_grouped'].dtype}")

    # Confirma que os índices do novo arquivo vão de 0 a M-1
    assert M_check == M, "Número de amostras diverge!"
    print(f"\n  Índices do novo dataset : 0 → {M_check - 1}  ✓")

    snrs_found = np.unique(f['Z'][:, 0])
    assert set(snrs_found.tolist()) == snr_set, "SNRs inesperados!"
    print(f"  SNRs presentes          : {snrs_found.tolist()}  ✓")

    yg = f['Y_grouped'][:]
    assert np.all(yg.sum(axis=1) == 1.0), "One-hot inválido em Y_grouped!"
    print(f"  Verificação one-hot     : OK  ✓")

print(f"\nArquivo salvo em: {OUTPUT_FILE}")

In [ ]:
OUTPUT_FILE

In [ ]:
h5py_path = OUTPUT_FILE


hdf5_file = h5py.File(h5py_path, 'r')
# Load the modulation classes. You can also copy and paste the content of classes-fixed.txt.
modulation_classes = json.load(open(modulation_classes_path, 'r'))

# Read the HDF5 groups
data = hdf5_file['X']
modulation_onehot = hdf5_file['Y_grouped']
# modulation_onehot = hdf5_file['Y']
snr = hdf5_file['Z']


# Load all data from the datasets in the HDF5 file
X = data
Y = modulation_onehot
Z = snr

# Convert one-hot encoded labels to numerical labels
y = np.argmax(Y, axis=1)

In [ ]:
num_samples = X.shape[0]
indices = np.arange(num_samples) #selected_indices_sorted
lables = np.column_stack((y, Z))

train_indices_sorted, val_indices_sorted, test_indices_sorted = split(indices, lables)

#.

In [ ]:
import os

# Verifica se o Drive já está acessível
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive já montado e acessível")
    print(f"   Conteúdo: {os.listdir('/content/drive/MyDrive')[:5]}")
else:
    # Só monta se não estiver acessível
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from torch.utils.data import Dataset
class H5PyDataset(Dataset):
    def __init__(self, h5_filepath, data_X_name, data_Y_name, data_Z_name, indices, formats):
        self.h5_filepath = h5_filepath
        self.data_X_name = data_X_name
        self.data_Y_name = data_Y_name
        self.indices = np.array(indices)
        self.formats = formats
        self.file = None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Abre o arquivo apenas uma vez por worker
        if self.file is None:
            self.file = h5py.File(self.h5_filepath, 'r')
            self.X_data = self.file[self.data_X_name]
            self.Y_data = self.file[self.data_Y_name]

        # Pega o índice real baseado na sua lista de sorteados
        real_idx = self.indices[idx]

        # Busca UMA amostra (1024, 2)
        x = self.X_data[real_idx, :]
        y = self.Y_data[real_idx]

        # Normalização Individual
        # mean = x.mean()
        # std = x.std()
        # x = (x - mean) / (std + 1e-8)

        # Trata o Label
        if self.formats == 0:
            # Se y for um vetor (One-Hot), pega o índice da classe
            if hasattr(y, "__len__"):
                y = np.argmax(y)

        # Conversão para Tensor
        x_tensor = torch.from_numpy(x).float() # [1024, 2]
        x_tensor = x_tensor.permute(1, 0)      # [2, 1024] (Canais primeiro)
        y_tensor = torch.tensor(y).long()      # Valor escalar

        return x_tensor, y_tensor

In [ ]:
num_samples = X.shape[0]
indices = np.arange(num_samples) #selected_indices_sorted
lables = np.column_stack((y, Z))

train_indices_sorted, val_indices_sorted, test_indices_sorted = split(indices, lables)

In [ ]:
from torch.utils.data import DataLoader, BatchSampler, RandomSampler, SequentialSampler, random_split

# 1. Dataset (Certifique-se que o __getitem__ aceita a lista de índices como discutimos)
# Dataset de Treino
train_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=train_indices_sorted,  # Seus índices de treino
    formats=0
)

# Dataset de Validação
val_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=val_indices_sorted,    # Seus índices de validação
    formats=0
)

test_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=test_indices_sorted,    # Seus índices de validação
    formats=0
)

batch_size = 64

# Remova o BatchSampler e o collate_fn.
# Use shuffle=True para o treino e shuffle=False para validação.
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,          # Isso substitui o RandomSampler
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,         # Isso substitui o SequentialSampler
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,         # Isso substitui o SequentialSampler
    drop_last=False
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATALOADERS COM SEED CONTROLADA
# ══════════════════════════════════════════════════════════════════════════════

import torch
from torch.utils.data import DataLoader

SEED       = 42
batch_size = 64

# Gerador dedicado para o DataLoader — controla o shuffle entre épocas
g = torch.Generator()
g.manual_seed(SEED)

def seed_worker(worker_id):
    """Garante seeds determinísticas nos workers do DataLoader."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    persistent_workers=False,
    generator=g,               # controla a ordem do shuffle
    worker_init_fn=seed_worker # controla seeds dos workers (se num_workers>0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    persistent_workers=False
)

In [ ]:
import torch.nn as nn
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=11, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=7, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=5, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.MaxPool1d(2)


        )

        # Técnica para detectar o tamanho do flatten automaticamente:
        self.flatten = nn.Flatten()
        with torch.no_grad():
            # Passamos um dado "dummy" de teste para ver o que sai das convs
            dummy_input = torch.zeros(1, 2, 1024)
            dummy_output = self.features(dummy_input)
            self.n_flatten = dummy_output.view(1, -1).size(1)

        print(f"Tamanho detectado para o Flatten: {self.n_flatten}")

        self.classifier = nn.Sequential(
            nn.Linear(self.n_flatten, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║         CÉLULA UNIFICADA — SELECIONAR → BUSCAR → CRIAR → TREINAR         ║
# ║                                                                          ║
# ║  Pré-requisitos (células anteriores já executadas):                      ║
# ║    • kagglehub.dataset_download() → variável  path                       ║
# ║    • modulation_classes_path definido                                    ║
# ║    • Classes H5PyDataset e CNN definidas                                 ║
# ║    • Funções split() / split_indices() e train_and_validate() definidas  ║
# ║    • Drive montado:                                                      ║
# ║        from google.colab import drive                                    ║
# ║        drive.mount('/content/drive', force_remount=True)                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, shutil, random
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

MODELO_ALVO = "ASK"
# Opções: "GROUP" | "ASK" | "PSK" | "APSK" | "QAM" | "AM" | "FM"

DESIRED_SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
   # todos os SNRs disponíveis
# Para treinar em SNR alto apenas: list(range(0, 32, 2))

EPOCHS       = 120
BATCH_SIZE   = 64
LR           = 5e-4
SEED         = 42

DRIVE_BASE   = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR    = "/content"          # onde o HDF5 fica durante a sessão
HDF5_BUILD_BATCH = 2048            # batch de I/O para montar o HDF5

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS (igual ao resto do notebook)
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES  = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
ALL_MODELS   = ["GROUP", "ASK", "PSK", "APSK", "QAM", "AM", "FM"]

INPUT_FILE   = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE = modulation_classes_path

assert MODELO_ALVO in ALL_MODELS, f"MODELO_ALVO inválido: {MODELO_ALVO}"

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _seed_worker(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

# ══════════════════════════════════════════════════════════════════════════════
# DERIVAÇÕES — group_indices, num_classes, global_to_local, nomes de arquivo
# ══════════════════════════════════════════════════════════════════════════════

mod_classes = json.load(open(CLASSES_FILE))

if MODELO_ALVO == "GROUP":
    # Classifica os 24 sinais em 6 grupos  →  label = GROUP_MAP[mod]
    group_indices   = np.arange(len(mod_classes), dtype=np.int64)
    num_classes     = 6
    global_to_local = {int(g): GROUP_MAP[mod_classes[g]] for g in group_indices}
    h5_label_key    = "Y_grouped"    # one-hot de 6 classes
    _label_note     = "6 grupos (ASK/PSK/APSK/QAM/AM/FM)"
else:
    # Classifica dentro do grupo escolhido
    group_indices = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_NAMES[GROUP_MAP[m]] == MODELO_ALVO]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}
    h5_label_key    = "Y"
    _label_note     = f"{num_classes} classes intra-grupo {MODELO_ALVO}"

snr_tag   = "_".join(str(s) for s in DESIRED_SNRS)
h5_name   = (f"GROUP_subset_snr_{snr_tag}.hdf5" if MODELO_ALVO == "GROUP"
             else f"{MODELO_ALVO}_subset_snr_{snr_tag}.hdf5")
h5_local  = os.path.join(LOCAL_DIR,  h5_name)
h5_drive  = os.path.join(DRIVE_BASE, MODELO_ALVO, "subset.hdf5")
ckpt_drive= os.path.join(DRIVE_BASE, MODELO_ALVO, "checkpoint.pth")
meta_drive= os.path.join(DRIVE_BASE, MODELO_ALVO, "meta.json")
idx_drive = lambda name: os.path.join(DRIVE_BASE, MODELO_ALVO, f"{name}.npy")

sep = "═" * 65
print(sep)
print(f"  Modelo  : {MODELO_ALVO}  ({_label_note})")
print(f"  SNRs    : {DESIRED_SNRS}")
print(f"  Device  : {device}")
print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1 — VERIFICAR DRIVE
# ══════════════════════════════════════════════════════════════════════════════

assert os.path.exists("/content/drive/MyDrive"), (
    "\n\n❌ Google Drive não montado!\n"
    "   Execute antes:\n"
    "       from google.colab import drive\n"
    "       drive.mount('/content/drive', force_remount=True)\n"
)

has_h5   = os.path.exists(h5_drive)
has_ckpt = os.path.exists(ckpt_drive)
has_idx  = all(os.path.exists(idx_drive(n))
               for n in ("train_indices", "val_indices", "test_indices"))

print(f"\n📂 Drive  →  {os.path.join(DRIVE_BASE, MODELO_ALVO)}")
print(f"   HDF5        : {'✅ encontrado' if has_h5   else '❌ não existe — será criado'}")
print(f"   Checkpoint  : {'✅ encontrado' if has_ckpt else '❌ não existe — treino do zero'}")
print(f"   Índices     : {'✅ encontrado' if has_idx  else '❌ não existe — será gerado'}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — OBTER HDF5 LOCAL (do Drive ou criar do zero)
# ══════════════════════════════════════════════════════════════════════════════

def _build_hdf5(input_file, output_file, target_snrs,
                group_indices, global_to_local, num_classes,
                modelo_id, batch_size=2048):
    """Filtra INPUT_FILE por SNR (e grupo se intra-grupo) e grava output_file."""
    snr_set = set(target_snrs)
    if os.path.exists(output_file):
        os.remove(output_file)

    print(f"\n  🔨 Construindo HDF5 para '{modelo_id}'...")

    original_indices = []
    with h5py.File(input_file, 'r') as src:
        N     = src['X'].shape[0]
        Z_ds  = src['Z']
        Y_ds  = src['Y']
        n_b   = int(np.ceil(N / batch_size))

        for i in range(n_b):
            s, e     = i * batch_size, min((i + 1) * batch_size, N)
            z_batch  = Z_ds[s:e, 0]
            y_batch  = np.argmax(Y_ds[s:e], axis=1)

            if modelo_id == "GROUP":
                mask = np.isin(z_batch, list(snr_set))
            else:
                mask = (np.isin(z_batch, list(snr_set)) &
                        np.isin(y_batch, group_indices))

            original_indices.extend((np.where(mask)[0] + s).tolist())
            if (i + 1) % 50 == 0 or i == n_b - 1:
                print(f"    Batch {i+1:>4}/{n_b}  |  amostras: {len(original_indices)}")

    original_indices = np.array(original_indices, dtype=np.int64)
    M = len(original_indices)
    if M == 0:
        raise ValueError("Nenhuma amostra encontrada com os filtros escolhidos.")

    NUM_GROUPS = 6
    lookup = np.array([GROUP_MAP[mod_classes[i]]
                       for i in range(len(mod_classes))], dtype=np.int64)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]

        with h5py.File(output_file, 'w') as out:
            # Metadados para reconstrução futura
            out.attrs['snrs']          = target_snrs
            out.attrs['modelo_id']     = modelo_id
            out.attrs['num_classes']   = num_classes
            out.attrs['group_indices'] = group_indices.tolist()
            out.attrs['total_samples'] = M
            out.attrs['seed']          = SEED

            ds_X  = out.create_dataset('X', shape=(M,)+x_shape, dtype=src['X'].dtype)
            ds_Z  = out.create_dataset('Z', shape=(M,)+z_shape, dtype=src['Z'].dtype)

            if modelo_id == "GROUP":
                # Y_grouped: one-hot de 6 grupos
                ds_Y = out.create_dataset('Y_grouped',
                                          shape=(M, NUM_GROUPS), dtype=np.float32)
            else:
                # Y: one-hot local (num_classes do grupo)
                ds_Y = out.create_dataset('Y',
                                          shape=(M, num_classes), dtype=np.int32)

            n_b = int(np.ceil(M / batch_size))
            for i in range(n_b):
                s, e = i * batch_size, min((i + 1) * batch_size, M)
                idx  = original_indices[s:e]

                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]

                y_global = np.argmax(src['Y'][idx], axis=1)

                if modelo_id == "GROUP":
                    y_group = lookup[y_global]
                    yg = np.zeros((e - s, NUM_GROUPS), dtype=np.float32)
                    yg[np.arange(e - s), y_group] = 1.0
                    ds_Y[s:e] = yg
                else:
                    y_local = np.array([global_to_local[int(g)] for g in y_global])
                    yo = np.zeros((e - s, num_classes), dtype=np.int32)
                    yo[np.arange(e - s), y_local] = 1
                    ds_Y[s:e] = yo

                if (i + 1) % 20 == 0 or i == n_b - 1:
                    print(f"    Escrevendo batch {i+1:>4}/{n_b}")

    print(f"  ✅ HDF5 criado: {output_file}  ({M} amostras)")
    return output_file


if has_h5:
    # ── Copia Drive → local (se ainda não copiado) ────────────────────────
    if not os.path.exists(h5_local):
        print(f"\n📦 Copiando HDF5 do Drive → {h5_local}  (pode demorar)…")
        shutil.copy(h5_drive, h5_local)
        print("  ✅ Cópia concluída.")
    else:
        print(f"\n📦 HDF5 já disponível localmente: {h5_local}")
else:
    # ── Cria do zero ─────────────────────────────────────────────────────
    _build_hdf5(INPUT_FILE, h5_local, DESIRED_SNRS,
                group_indices, global_to_local, num_classes, MODELO_ALVO,
                batch_size=HDF5_BUILD_BATCH)

    # Guarda no Drive para próximas sessões
    print(f"\n  💾 Salvando HDF5 no Drive…")
    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    shutil.copy(h5_local, h5_drive)
    print(f"  ✅ {h5_drive}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — ÍNDICES TRAIN / VAL / TEST  (carrega ou gera — sem data leakage)
# ══════════════════════════════════════════════════════════════════════════════

def _split_deterministic(h5_path, label_key, seed=42):
    """Gera split 60/20/20 estratificado apenas pelo label (sem SNR no stratify)."""
    with h5py.File(h5_path, 'r') as f:
        Y_all = f[label_key][:]
    y_all   = np.argmax(Y_all, axis=1)
    indices = np.arange(len(y_all))

    tv_idx, te_idx, tv_lbl, _ = train_test_split(
        indices, y_all, test_size=0.20, random_state=seed, stratify=y_all
    )
    tr_idx, va_idx, _, _ = train_test_split(
        tv_idx, tv_lbl, test_size=0.25, random_state=seed, stratify=tv_lbl
    )
    return np.sort(tr_idx), np.sort(va_idx), np.sort(te_idx)


if has_idx:
    print("\n📐 Carregando índices do Drive…")
    train_indices = np.load(idx_drive("train_indices"))
    val_indices   = np.load(idx_drive("val_indices"))
    test_indices  = np.load(idx_drive("test_indices"))
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")
else:
    print("\n📐 Gerando índices (split determinístico)…")
    train_indices, val_indices, test_indices = _split_deterministic(
        h5_local, h5_label_key, seed=SEED
    )
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")

    # Persiste no Drive
    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    np.save(idx_drive("train_indices"), train_indices)
    np.save(idx_drive("val_indices"),   val_indices)
    np.save(idx_drive("test_indices"),  test_indices)
    print("   ✅ Índices salvos no Drive.")

# Verificação anti-leakage
tr_s, va_s, te_s = set(train_indices), set(val_indices), set(test_indices)
assert not (tr_s & va_s), "❌ Sobreposição treino/val!"
assert not (tr_s & te_s), "❌ Sobreposição treino/test!"
assert not (va_s & te_s), "❌ Sobreposição val/test!"
print("   ✅ Sem sobreposição entre splits (anti-leakage OK)")

# Variáveis globais compatíveis com o restante do notebook
train_indices_sorted = train_indices
val_indices_sorted   = val_indices
test_indices_sorted  = test_indices
h5py_path            = h5_local

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

g = torch.Generator(); g.manual_seed(SEED)

train_dataset = H5PyDataset(
    h5_filepath=h5_local,
    data_X_name='X',
    data_Y_name=h5_label_key,
    data_Z_name='Z',
    indices=train_indices,
    formats=0
)

val_dataset = H5PyDataset(
    h5_filepath=h5_local,
    data_X_name='X',
    data_Y_name=h5_label_key,
    data_Z_name='Z',
    indices=val_indices,
    formats=0
)

test_dataset = H5PyDataset(
    h5_filepath=h5_local,
    data_X_name='X',
    data_Y_name=h5_label_key,
    data_Z_name='Z',
    indices=test_indices,
    formats=0
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=False, num_workers=0,
                          generator=g, worker_init_fn=_seed_worker)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, num_workers=0)

print(f"\n📊 DataLoaders prontos:")
print(f"   Treino     : {len(train_dataset):>8} amostras")
print(f"   Validação  : {len(val_dataset):>8} amostras")
print(f"   Teste      : {len(test_dataset):>8} amostras")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5 — INSTANCIAR MODELO  (carrega checkpoint se existir)
# ══════════════════════════════════════════════════════════════════════════════

model     = CNN(num_classes=num_classes).to(device)
epoch_ini = 0                    # época de onde o treino começa
best_acc  = 0.0                  # melhor val_acc conhecida até agora

if has_ckpt:
    print(f"\n🔄 Carregando checkpoint do Drive…")
    ckpt = torch.load(ckpt_drive, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    epoch_ini = ckpt.get("epoch", 0)
    best_acc  = ckpt.get("val_acc", 0.0)
    print(f"   Retomando da época {epoch_ini}  |  melhor val_acc = {best_acc:.2f}%")
else:
    print(f"\n⚙️  Nenhum checkpoint encontrado — treinando do zero.")

print(f"   CNN instanciada com {num_classes} classes | Flatten: {model.n_flatten}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 6 — OTIMIZADOR E SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-8
)

if has_ckpt:
    # Tenta restaurar estado do otimizador (opcional — ignora se não existir)
    try:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        print("   ✅ Estado do otimizador restaurado.")
    except Exception:
        print("   ⚠️  Estado do otimizador não restaurado — usando valores padrão.")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 7 — TREINAR
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print(f"  🚀 Iniciando treino — modelo '{MODELO_ALVO}'  |  épocas: {EPOCHS}  |  LR: {LR}")
print(sep)

best_model_path = train_and_validate(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler  = scheduler,
    epochs     = EPOCHS,
    save_name  = f"cnn_{MODELO_ALVO}",
)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 8 — SALVAR NO DRIVE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print("  💾 Salvando sessão no Drive…")
print(sep)

save_dir = os.path.join(DRIVE_BASE, MODELO_ALVO)
os.makedirs(save_dir, exist_ok=True)

# Checkpoint
shutil.copy(best_model_path, ckpt_drive)
print(f"  ✅ Checkpoint  → {ckpt_drive}")

# Índices (já salvos na etapa 3 se eram novos; re-salva para garantir)
np.save(idx_drive("train_indices"), train_indices_sorted)
np.save(idx_drive("val_indices"),   val_indices_sorted)
np.save(idx_drive("test_indices"),  test_indices_sorted)
print(f"  ✅ Índices     → {save_dir}/[train|val|test]_indices.npy")

# HDF5 (já copiado na etapa 2; só avisa)
print(f"  ✅ HDF5        → {h5_drive}  (já persistido)")

# meta.json
with h5py.File(best_model_path if best_model_path.endswith(".hdf5") else h5_local, 'r') as _f:
    h5_attrs = {k: (list(v) if hasattr(v, '__iter__') else int(v))
                for k, v in _f.attrs.items()}

ckpt_final = torch.load(best_model_path, map_location="cpu")
meta = {
    "modelo_id":    MODELO_ALVO,
    "seed":         SEED,
    "desired_snrs": DESIRED_SNRS,
    "num_classes":  num_classes,
    "group_indices": group_indices.tolist(),
    "train_size":   int(len(train_indices_sorted)),
    "val_size":     int(len(val_indices_sorted)),
    "test_size":    int(len(test_indices_sorted)),
    "best_epoch":   int(ckpt_final.get("epoch", -1)),
    "best_val_acc": float(ckpt_final.get("val_acc", -1)),
    "h5_attrs":     h5_attrs,
}
with open(meta_drive, "w") as fp:
    json.dump(meta, fp, indent=2, ensure_ascii=False)
print(f"  ✅ meta.json   → {meta_drive}")

print(f"\n{'═'*65}")
print(f"  ✅ Sessão '{MODELO_ALVO}' completa e persistida no Drive.")
print(f"  📁 {save_dir}")
print(f"{'═'*65}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 7 — TREINAR
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print(f"  🚀 Iniciando treino — modelo '{MODELO_ALVO}'  |  épocas: {EPOCHS}  |  LR: {LR}")
print(sep)

best_model_path = train_and_validate(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler  = scheduler,
    epochs     = EPOCHS,
    save_name  = f"cnn_{MODELO_ALVO}",
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 8 — SALVAR NO DRIVE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print("  💾 Salvando sessão no Drive…")
print(sep)

save_dir = os.path.join(DRIVE_BASE, MODELO_ALVO)
os.makedirs(save_dir, exist_ok=True)

# Checkpoint
shutil.copy(best_model_path, ckpt_drive)
print(f"  ✅ Checkpoint  → {ckpt_drive}")

# Índices (já salvos na etapa 3 se eram novos; re-salva para garantir)
np.save(idx_drive("train_indices"), train_indices_sorted)
np.save(idx_drive("val_indices"),   val_indices_sorted)
np.save(idx_drive("test_indices"),  test_indices_sorted)
print(f"  ✅ Índices     → {save_dir}/[train|val|test]_indices.npy")

# HDF5 (já copiado na etapa 2; só avisa)
print(f"  ✅ HDF5        → {h5_drive}  (já persistido)")

# meta.json
with h5py.File(best_model_path if best_model_path.endswith(".hdf5") else h5_local, 'r') as _f:
    h5_attrs = {k: (list(v) if hasattr(v, '__iter__') else int(v))
                for k, v in _f.attrs.items()}

ckpt_final = torch.load(best_model_path, map_location="cpu")
meta = {
    "modelo_id":    MODELO_ALVO,
    "seed":         SEED,
    "desired_snrs": DESIRED_SNRS,
    "num_classes":  num_classes,
    "group_indices": group_indices.tolist(),
    "train_size":   int(len(train_indices_sorted)),
    "val_size":     int(len(val_indices_sorted)),
    "test_size":    int(len(test_indices_sorted)),
    "best_epoch":   int(ckpt_final.get("epoch", -1)),
    "best_val_acc": float(ckpt_final.get("val_acc", -1)),
    "h5_attrs":     h5_attrs,
}
with open(meta_drive, "w") as fp:
    json.dump(meta, fp, indent=2, ensure_ascii=False)
print(f"  ✅ meta.json   → {meta_drive}")

print(f"\n{'═'*65}")
print(f"  ✅ Sessão '{MODELO_ALVO}' completa e persistida no Drive.")
print(f"  📁 {save_dir}")
print(f"{'═'*65}")

In [ ]:
print(H5PyDataset.__init__.__code__.co_varnames)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   SALVAR CHECKPOINT LOCAL (/content) → GOOGLE DRIVE         ║
# ╚══════════════════════════════════════════════════════════════╝

import os
import shutil

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────

MODELO_ALVO = "GROUP"

# arquivo atual no Colab
ckpt_local = "/content/cnn_GROUP_ep007_acc90.86_lr1.00e-03.pth"

# destino esperado pelo seu pipeline
DRIVE_BASE = "/content/drive/MyDrive/radioml_sessions"
ckpt_drive = os.path.join(DRIVE_BASE, MODELO_ALVO, "checkpoint.pth")

# ──────────────────────────────────────────────────────────────
# GARANTIR QUE O DRIVE ESTÁ MONTADO
# ──────────────────────────────────────────────────────────────

assert os.path.exists("/content/drive/MyDrive"), \
    "❌ Drive não montado! Rode: drive.mount('/content/drive')"

# ──────────────────────────────────────────────────────────────
# GARANTIR QUE O ARQUIVO EXISTE
# ──────────────────────────────────────────────────────────────

assert os.path.exists(ckpt_local), \
    f"❌ Arquivo não encontrado: {ckpt_local}"

# ──────────────────────────────────────────────────────────────
# CRIAR DIRETÓRIO NO DRIVE (se necessário)
# ──────────────────────────────────────────────────────────────

os.makedirs(os.path.dirname(ckpt_drive), exist_ok=True)

# ──────────────────────────────────────────────────────────────
# COPIAR E RENOMEAR
# ──────────────────────────────────────────────────────────────

shutil.copy(ckpt_local, ckpt_drive)

print("✅ Checkpoint salvo no Drive!")
print(f"📂 Origem : {ckpt_local}")
print(f"📂 Destino: {ckpt_drive}")

# ──────────────────────────────────────────────────────────────
# VERIFICAÇÃO FINAL
# ──────────────────────────────────────────────────────────────

print("\n🔍 Verificação:")
print("Existe no destino?", os.path.exists(ckpt_drive))

In [ ]:
import os

path = "/content/drive/MyDrive/radioml_sessions/GROUP/checkpoint.pth"

print("Existe:", os.path.exists(path))
print("Tamanho (bytes):", os.path.getsize(path))

In [ ]:
ckpt = torch.load(ckpt_drive)

In [ ]:
ckpt = torch.load("/content/cnn_GROUP_ep001_acc87.64_lr1.00e-05.pth")
print("OK LOCAL")

In [ ]:
import os

ckpt_drive = "/content/drive/MyDrive/radioml_sessions/GROUP/checkpoint.pth"

if os.path.exists(ckpt_drive):
    os.remove(ckpt_drive)
    print("🗑️ Checkpoint antigo removido")

In [ ]:
import shutil

src = "/content/cnn_GROUP_ep001_acc88.61_lr1.00e-03.pth"
dst = "/content/drive/MyDrive/radioml_sessions/GROUP/checkpoint.pth"

shutil.copyfile(src, dst)
print("✅ Copiado novamente")

In [ ]:
import torch

torch.load(dst, map_location="cpu")
print("✅ Checkpoint válido no Drive")

#BUSCA DE HIPERPARAMETROS


In [ ]:
MODELO_ALVO = "APSK"

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FlexCNN — CNN Conv1D com arquitetura configurável
# Cole esta célula ANTES da célula de busca de hiperparâmetros
# ══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn

class FlexCNN(nn.Module):
    """
    CNN Conv1D com número de camadas e filtros variáveis.

    Args:
        num_classes   : número de classes de saída
        arch          : lista de dicts {out_channels, kernel_size, pool}
        classifier    : lista de inteiros — neurônios das camadas densas ocultas
        dropout       : taxa de Dropout nas camadas densas
        in_channels   : canais de entrada (padrão 2 — I/Q do RadioML)
        input_length  : comprimento temporal da série (padrão 1024)
    """

    def __init__(
        self,
        num_classes  : int,
        arch         : list,
        classifier   : list  = None,
        dropout      : float = 0.5,
        in_channels  : int   = 2,
        input_length : int   = 1024,
    ):
        super().__init__()
        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels

        for block in arch:
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            pad    = ks // 2          # padding "same" aproximado

            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=pad),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))
            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        # Detecta o tamanho do flatten com um forward pass dummy
        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, input_length)
            n_flat = self.features(dummy).view(1, -1).size(1)

        head = []
        prev = n_flat
        for units in classifier:
            head += [
                nn.Linear(prev, units),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            prev = units
        head.append(nn.Linear(prev, num_classes))
        self.classifier = nn.Sequential(*head)

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))


# Teste rápido
_dummy = torch.zeros(4, 2, 1024)
_model = FlexCNN(
    num_classes = 6,
    arch        = [
        {"out_channels": 64,  "kernel_size": 7, "pool": True},
        {"out_channels": 128, "kernel_size": 5, "pool": True},
    ],
)
assert _model(_dummy).shape == (4, 6), "FlexCNN: shape de saída incorreto"
del _dummy, _model

print("✅ FlexCNN definida e testada com sucesso")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DIAGNÓSTICO — verifica labels antes de iniciar a busca
# ══════════════════════════════════════════════════════════════════════════════

import os, json
import numpy as np
import torch
import h5py
from torch.utils.data import DataLoader

print(f"Verificando labels para modelo '{MODELO_ALVO}'...")
print(f"  h5_local      : {h5_local}")
print(f"  h5_label_key  : {h5_label_key}")
print(f"  num_classes   : {num_classes}")

# ── Verifica o conteúdo do HDF5 ──────────────────────────────────────────────
with h5py.File(h5_local, 'r') as f:
    datasets = list(f.keys())
    attrs    = dict(f.attrs)
    print(f"\n  Datasets no HDF5 : {datasets}")
    print(f"  Atributos        : {attrs}")

    y_raw    = np.argmax(f[h5_label_key][:100], axis=1)
    y_unique = np.unique(np.argmax(f[h5_label_key][:], axis=1))
    print(f"\n  Primeiros 10 labels (argmax de {h5_label_key}): {y_raw[:10]}")
    print(f"  Labels únicos no HDF5 : {y_unique}")
    print(f"  y.min()={y_unique.min()}  y.max()={y_unique.max()}")

# ── Verifica o que o H5PyDataset retorna ─────────────────────────────────────
print(f"\n  Verificando H5PyDataset com os primeiros {BATCH_SIZE} exemplos...")

ds_test = H5PyDataset(
    h5_local, 'X', h5_label_key, 'Z',
    train_indices_sorted[:BATCH_SIZE], formats=0
)
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

xb, yb = next(iter(loader_test))
print(f"  xb.shape      : {xb.shape}")
print(f"  yb.shape      : {yb.shape}")
print(f"  yb.dtype      : {yb.dtype}")
print(f"  yb únicos     : {yb.unique().tolist()}")
print(f"  yb.min()      : {yb.min().item()}")
print(f"  yb.max()      : {yb.max().item()}")

# ── Verifica compatibilidade com num_classes ──────────────────────────────────
if yb.max().item() >= num_classes:
    print(f"\n  ❌ PROBLEMA ENCONTRADO!")
    print(f"     yb.max()={yb.max().item()} >= num_classes={num_classes}")
    print(f"     O H5PyDataset está retornando labels globais (0-23)")
    print(f"     em vez de labels locais (0-{num_classes-1})")
    print(f"\n  CAUSA: H5PyDataset usa argmax de '{h5_label_key}' que")
    print(f"     contém one-hot com 24 posições (índices globais),")
    print(f"     não remapeado para o grupo {MODELO_ALVO}.")
else:
    print(f"\n  ✅ Labels OK — dentro do intervalo [0, {num_classes-1}]")

# ── Forward pass de teste ─────────────────────────────────────────────────────
print(f"\n  Testando forward pass com FlexCNN ({num_classes} classes)...")
_model = FlexCNN(
    num_classes = num_classes,
    arch        = ARCHITECTURES[0]["arch"],
    classifier  = CLASSIFIER_HEAD,
    dropout     = DROPOUT,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model.to(device)
xb_d = xb.to(device)
yb_d = yb.to(device)

with torch.no_grad():
    try:
        out  = _model(xb_d)
        loss = torch.nn.CrossEntropyLoss()(out, yb_d)
        print(f"  ✅ Forward pass OK — loss={loss.item():.4f}")
        print(f"     out.shape = {out.shape}")
    except Exception as e:
        print(f"  ❌ Erro no forward pass: {e}")

del _model
torch.cuda.empty_cache()

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║      BUSCA DE ARQUITETURA — CNN Conv1D com K-Fold Cross-Validation (k=5)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO DA BUSCA — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

MODELO_ALVO = "ASK"
# Opções: "GROUP" | "ASK" | "PSK" | "APSK" | "QAM" | "AM" | "FM"

DESIRED_SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]

K_FOLDS             = 5
EPOCHS_PER_FOLD     = 120
EARLY_STOP_PATIENCE = 8
SEED                = 42
DROPOUT             = 0.5
LEARNING_RATE       = 1e-3
BATCH_SIZE          = 64
CLASSIFIER_HEAD     = [512]
SCHEDULER           = "plateau"
SCHEDULER_PATIENCE  = 2

DRIVE_BASE  = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR   = "/content"
HDF5_BUILD_BATCH = 2048

ARCHITECTURES = [

    # ── 2 camadas ─────────────────────────────────────────────────────────────
    {
        "label": "2L_32-64",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
        ],
    },
    {
        "label": "2L_64-128",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
        ],
    },

    # ── 3 camadas ─────────────────────────────────────────────────────────────
    {
        "label": "3L_32-64-128",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
            {"out_channels": 128, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_64-128-256",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
            {"out_channels": 256, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_128-256-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 7, "pool": True},
            {"out_channels": 256, "kernel_size": 5, "pool": True},
            {"out_channels": 512, "kernel_size": 3, "pool": True},
        ],
    },

    # ── 4 camadas ─────────────────────────────────────────────────────────────
    {
        "label": "4L_32-64-128-256",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": True},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_64-128-256-512",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 128, "kernel_size": 7,  "pool": True},
            {"out_channels": 256, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_128-256-512-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 11, "pool": True},
            {"out_channels": 256, "kernel_size": 7,  "pool": True},
            {"out_channels": 512, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },

    # ── 5 camadas ─────────────────────────────────────────────────────────────
    {
        "label": "5L_32-64-128-256-512",
        "arch": [
            {"out_channels": 32,   "kernel_size": 11, "pool": True},
            {"out_channels": 64,   "kernel_size": 7,  "pool": True},
            {"out_channels": 128,  "kernel_size": 5,  "pool": True},
            {"out_channels": 256,  "kernel_size": 3,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "5L_64-128-256-512-1024",
        "arch": [
            {"out_channels": 64,   "kernel_size": 11, "pool": True},
            {"out_channels": 128,  "kernel_size": 7,  "pool": True},
            {"out_channels": 256,  "kernel_size": 5,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            {"out_channels": 1024, "kernel_size": 3,  "pool": True},
        ],
    },

    # ── 6 camadas com pooling misto ───────────────────────────────────────────
    {
        "label": "6L_64-64-128-128-256-512_mixpool",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "6L_32-64-64-128-256-512_mixpool",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 64,  "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
]

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS (idêntico ao resto do notebook)
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
ALL_MODELS  = ["GROUP", "ASK", "PSK", "APSK", "QAM", "AM", "FM"]

INPUT_FILE   = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE = modulation_classes_path

assert MODELO_ALVO in ALL_MODELS, \
    f"MODELO_ALVO inválido: '{MODELO_ALVO}'. Opções: {ALL_MODELS}"

sep = "═" * 65

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1 — CONFIGURAR GRUPO / CLASSES
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print(f"  🔍  BUSCA DE ARQUITETURA — modelo '{MODELO_ALVO}'")
print(sep)

mod_classes = json.load(open(CLASSES_FILE))

if MODELO_ALVO == "GROUP":
    group_indices    = np.arange(len(mod_classes), dtype=np.int64)
    num_classes      = 6
    h5_label_key     = "Y_grouped"
    global_to_local  = None
else:
    target_id       = {v: k for k, v in GROUP_NAMES.items()}[MODELO_ALVO]
    group_indices   = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_MAP[m] == target_id]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    h5_label_key    = "Y"
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}

print(f"  Modelo       : {MODELO_ALVO}")
print(f"  num_classes  : {num_classes}")
print(f"  SNRs         : {DESIRED_SNRS}")
print(f"  Arquiteturas : {len(ARCHITECTURES)}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — MONTAR / CARREGAR HDF5 FILTRADO (igual à célula principal)
# ══════════════════════════════════════════════════════════════════════════════

snr_tag   = '_'.join(str(s) for s in DESIRED_SNRS)
h5_fname  = f"{MODELO_ALVO}_subset_snr_{snr_tag}.hdf5"
h5_local  = os.path.join(LOCAL_DIR, h5_fname)
h5_drive  = os.path.join(DRIVE_BASE, MODELO_ALVO, h5_fname)

print(f"\n[1/5] HDF5 filtrado: {h5_local}")

if not os.path.exists(h5_local):
    if os.path.exists(h5_drive):
        print(f"  📥 Copiando do Drive...")
        os.makedirs(os.path.dirname(h5_drive), exist_ok=True)
        import shutil
        shutil.copy(h5_drive, h5_local)
        print(f"  ✅ Copiado.")
    else:
        print(f"  🔨 Construindo HDF5 para '{MODELO_ALVO}'...")
        snr_set = set(DESIRED_SNRS)

        original_indices = []
        with h5py.File(INPUT_FILE, 'r') as src:
            N    = src['X'].shape[0]
            Z_ds = src['Z']
            Y_ds = src['Y']
            n_b  = int(np.ceil(N / HDF5_BUILD_BATCH))

            for i in range(n_b):
                s, e     = i * HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, N)
                z_batch  = Z_ds[s:e, 0]
                y_batch  = np.argmax(Y_ds[s:e], axis=1)

                if MODELO_ALVO == "GROUP":
                    mask = np.isin(z_batch, list(snr_set))
                else:
                    mask = (np.isin(z_batch, list(snr_set)) &
                            np.isin(y_batch, group_indices))

                original_indices.extend((np.where(mask)[0] + s).tolist())

                if (i+1) % 20 == 0 or i == n_b-1:
                    print(f"  Batch {i+1}/{n_b} | selecionados: "
                          f"{len(original_indices)}")

        original_indices = np.array(original_indices, dtype=np.int64)
        M = len(original_indices)
        print(f"  Total: {M} amostras")

        with h5py.File(INPUT_FILE, 'r') as src:
            x_shape  = src['X'].shape[1:]
            z_shape  = src['Z'].shape[1:]
            y_nc     = num_classes if MODELO_ALVO != "GROUP" else 6
            y_nc_raw = len(mod_classes)

            with h5py.File(h5_local, 'w') as out:
                out.attrs['modelo_alvo']   = MODELO_ALVO
                out.attrs['snrs']          = DESIRED_SNRS
                out.attrs['num_classes']   = num_classes
                out.attrs['total_samples'] = M
                out.attrs['group_indices'] = group_indices.tolist()

                ds_X = out.create_dataset('X', shape=(M,)+x_shape,
                                          dtype=src['X'].dtype)
                ds_Y = out.create_dataset('Y', shape=(M, y_nc_raw),
                                          dtype=src['Y'].dtype)
                ds_Z = out.create_dataset('Z', shape=(M,)+z_shape,
                                          dtype=src['Z'].dtype)

                if MODELO_ALVO == "GROUP":
                    y_nc_grouped = 6
                    ds_Yg = out.create_dataset(
                        'Y_grouped', shape=(M, y_nc_grouped), dtype=np.int32)

                n_b2 = int(np.ceil(M / HDF5_BUILD_BATCH))
                for i in range(n_b2):
                    s, e  = i*HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, M)
                    idx   = original_indices[s:e]
                    ds_X[s:e] = src['X'][idx]
                    ds_Y[s:e] = src['Y'][idx]
                    ds_Z[s:e] = src['Z'][idx]

                    if MODELO_ALVO == "GROUP":
                        y_raw  = np.argmax(src['Y'][idx], axis=1)
                        y_grp  = np.array(
                            [GROUP_MAP[mod_classes[y]] for y in y_raw])
                        y_oh   = np.zeros((len(y_grp), 6), dtype=np.int32)
                        y_oh[np.arange(len(y_grp)), y_grp] = 1
                        ds_Yg[s:e] = y_oh

                    if (i+1) % 10 == 0 or i == n_b2-1:
                        print(f"  Gravando batch {i+1}/{n_b2}")

        print(f"  ✅ HDF5 construído: {h5_local}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — SPLIT DETERMINÍSTICO (igual à célula principal)
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n[2/5] Carregando/gerando split de índices...")

save_dir  = os.path.join(DRIVE_BASE, MODELO_ALVO)
os.makedirs(save_dir, exist_ok=True)

idx_drive = lambda name: os.path.join(save_dir, f"{name}.npy")

# Tenta carregar do Drive
if (os.path.exists(idx_drive("train_indices")) and
        os.path.exists(idx_drive("val_indices")) and
        os.path.exists(idx_drive("test_indices"))):

    train_indices_sorted = np.load(idx_drive("train_indices"))
    val_indices_sorted   = np.load(idx_drive("val_indices"))
    test_indices_sorted  = np.load(idx_drive("test_indices"))
    print(f"  ✅ Índices carregados do Drive.")

else:
    with h5py.File(h5_local, 'r') as f:
        M     = f['X'].shape[0]
        Y_lbl = np.argmax(f[h5_label_key][:], axis=1)

    from sklearn.model_selection import train_test_split

    indices = np.arange(M)
    tv_idx, test_idx, tv_y, _ = train_test_split(
        indices, Y_lbl,
        test_size=0.2, random_state=SEED, stratify=Y_lbl
    )
    train_idx, val_idx, _, _ = train_test_split(
        tv_idx, tv_y,
        test_size=0.25, random_state=SEED, stratify=tv_y
    )

    train_indices_sorted = np.sort(train_idx)
    val_indices_sorted   = np.sort(val_idx)
    test_indices_sorted  = np.sort(test_idx)

    np.save(idx_drive("train_indices"), train_indices_sorted)
    np.save(idx_drive("val_indices"),   val_indices_sorted)
    np.save(idx_drive("test_indices"),  test_indices_sorted)
    print(f"  ✅ Índices gerados e salvos no Drive.")

print(f"     treino={len(train_indices_sorted):,}  "
      f"val={len(val_indices_sorted):,}  "
      f"teste={len(test_indices_sorted):,}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — FOLDS DETERMINÍSTICOS (apenas train_indices — sem vazamento)
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n[3/5] Carregando/gerando folds...")

folds_path = os.path.join(save_dir, f"kfold_search_folds_k{K_FOLDS}.json")

if os.path.exists(folds_path):
    with open(folds_path) as f:
        fd = json.load(f)
    assert fd["modelo_alvo"]  == MODELO_ALVO, "modelo_alvo diverge do arquivo de folds!"
    assert fd["k_folds"]      == K_FOLDS,     "k_folds diverge!"
    assert fd["seed"]         == SEED,         "seed diverge!"
    assert fd["n_train"]      == len(train_indices_sorted), "tamanho de treino mudou!"
    fold_splits = [(np.array(fd["folds"][i]["train_idx"]),
                    np.array(fd["folds"][i]["val_idx"]))
                   for i in range(K_FOLDS)]
    print(f"  ✅ Folds carregados: {folds_path}")
else:
    # Rótulos dos índices de treino para estratificação
    with h5py.File(h5_local, 'r') as f:
        y_all_train = np.argmax(
            f[h5_label_key][train_indices_sorted], axis=1
        )

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True,
                          random_state=SEED)
    fold_splits = list(skf.split(np.arange(len(train_indices_sorted)),
                                 y_all_train))

    folds_data = {
        "modelo_alvo": MODELO_ALVO,
        "k_folds"    : K_FOLDS,
        "seed"       : SEED,
        "n_train"    : len(train_indices_sorted),
        "folds": [
            {"fold": i, "n_fold_train": len(tr),
             "n_fold_val": len(vl),
             "train_idx": tr.tolist(), "val_idx": vl.tolist()}
            for i, (tr, vl) in enumerate(fold_splits)
        ]
    }
    with open(folds_path, "w") as f:
        json.dump(folds_data, f)
    print(f"  ✅ {K_FOLDS} folds salvos: {folds_path}")
    for i, (tr, vl) in enumerate(fold_splits):
        print(f"     Fold {i+1}: treino={len(tr):,}  val={len(vl):,}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5 — BUSCA DE ARQUITETURA COM K-FOLD CV
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n[4/5] Verificando progresso anterior...")

results_path = os.path.join(save_dir, "kfold_search_fold_results.json")
summary_path = os.path.join(save_dir, "kfold_search_summary.json")

# Carrega resultados anteriores
if os.path.exists(results_path):
    with open(results_path) as f:
        all_fold_results = json.load(f)
    done_set = {(r["label"], r["fold"]) for r in all_fold_results}
    print(f"  ✅ {len(all_fold_results)} resultados anteriores carregados "
          f"({len(done_set)} combinações concluídas)")
else:
    all_fold_results = []
    done_set         = set()
    print(f"  Nenhum resultado anterior — iniciando do zero")

total_jobs = len(ARCHITECTURES) * K_FOLDS
print(f"  Total jobs : {total_jobs}  │  "
      f"Concluídos : {len(done_set)}  │  "
      f"Restantes : {total_jobs - len(done_set)}")


def save_fold_result(result):
    """Salva imediatamente após cada fold."""
    data = []
    if os.path.exists(results_path):
        with open(results_path) as f:
            data = json.load(f)
    data.append(result)
    with open(results_path, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def update_summary():
    """Recalcula e salva média/std/var por arquitetura."""
    from collections import defaultdict
    stats    = defaultdict(list)
    meta     = {}
    for r in all_fold_results:
        stats[r["label"]].append(r["best_val_acc"])
        if r["label"] not in meta:
            meta[r["label"]] = {
                k: r[k] for k in ["arch", "n_layers", "filters",
                                   "classifier", "dropout", "lr"]
                if k in r
            }

    summary = []
    for label, accs in stats.items():
        e = {
            "modelo_alvo"  : MODELO_ALVO,
            "num_classes"  : num_classes,
            "label"        : label,
            "folds_done"   : len(accs),
            "complete"     : len(accs) == K_FOLDS,
            "accs_per_fold": [round(a, 4) for a in accs],
            "mean_val_acc" : round(float(np.mean(accs)), 4),
            "std_val_acc"  : round(float(np.std(accs)),  4),
            "var_val_acc"  : round(float(np.var(accs)),  4),
            "min_val_acc"  : round(float(np.min(accs)),  4),
            "max_val_acc"  : round(float(np.max(accs)),  4),
        }
        e.update(meta.get(label, {}))
        summary.append(e)

    summary.sort(key=lambda x: (x["complete"], x["mean_val_acc"]),
                 reverse=True)
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    return summary


print(f"\n[5/5] Executando busca...\n")
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_mean = max(
    (r["mean_val_acc"] for r in update_summary()
     if r.get("complete")),
    default=-1.0
)

for arch_idx, arch_entry in enumerate(ARCHITECTURES, 1):
    label = arch_entry["label"]
    arch  = arch_entry["arch"]

    folds_done = [f for f in range(K_FOLDS) if (label, f) in done_set]
    if len(folds_done) == K_FOLDS:
        accs = [r["best_val_acc"] for r in all_fold_results
                if r["label"] == label]
        print(f"  ⏭  [{arch_idx}/{len(ARCHITECTURES)}] "
              f"{label}  (média={np.mean(accs):.2f}%)")
        continue

    print(f"\n{sep}")
    print(f"  [{arch_idx}/{len(ARCHITECTURES)}]  "
          f"Modelo: {MODELO_ALVO}  │  Arq: {label}")
    print(f"  Filtros : " + " → ".join(
          str(b["out_channels"]) for b in arch))
    restantes = [f+1 for f in range(K_FOLDS) if (label, f) not in done_set]
    print(f"  Folds restantes: {restantes}")
    print(sep)

    t0_arch = time.time()

    for fold_i, (tr_pos, vl_pos) in enumerate(fold_splits):

        if (label, fold_i) in done_set:
            acc = next(r["best_val_acc"] for r in all_fold_results
                       if r["label"] == label and r["fold"] == fold_i)
            print(f"\n  Fold {fold_i+1}/{K_FOLDS} — ⏭  já concluído "
                  f"(val_acc={acc:.2f}%)")
            continue

        print(f"\n  ── Fold {fold_i+1}/{K_FOLDS} "
              f"(treino={len(tr_pos):,}  val={len(vl_pos):,}) ──")

        # Seed determinístico e único por (arquitetura, fold)
        fold_seed = SEED + arch_idx * 100 + fold_i
        random.seed(fold_seed)
        np.random.seed(fold_seed)
        torch.manual_seed(fold_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(fold_seed)

        g = torch.Generator()
        g.manual_seed(fold_seed)

        # Índices absolutos no HDF5
        tr_abs = train_indices_sorted[tr_pos]
        vl_abs = train_indices_sorted[vl_pos]

        tr_loader = DataLoader(
            H5PyDataset(h5_local, 'X', h5_label_key, 'Z',
                        tr_abs, formats=0),
            batch_size=BATCH_SIZE, shuffle=True,
            num_workers=0, generator=g
        )
        vl_loader = DataLoader(
            H5PyDataset(h5_local, 'X', h5_label_key, 'Z',
                        vl_abs, formats=0),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )

        model = FlexCNN(
            num_classes=num_classes, arch=arch,
            classifier=CLASSIFIER_HEAD, dropout=DROPOUT
        )
        n_params = sum(p.numel() for p in model.parameters()
                      if p.requires_grad)
        print(f"  Parâmetros: {n_params:,}")

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5,
            patience=SCHEDULER_PATIENCE
        )

        best_val_acc      = 0.0
        best_state        = None
        epochs_no_improve = 0
        history           = []
        model.to(device)
        t0_fold = time.time()

        for epoch in range(EPOCHS_PER_FOLD):

            model.train()
            tr_loss = tr_correct = tr_total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                out  = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                tr_loss    += loss.item()
                tr_correct += (out.argmax(1) == yb).sum().item()
                tr_total   += yb.size(0)

            model.eval()
            vl_loss = vl_correct = vl_total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    out     = model(xb)
                    loss    = criterion(out, yb)
                    vl_loss    += loss.item()
                    vl_correct += (out.argmax(1) == yb).sum().item()
                    vl_total   += yb.size(0)

            tr_acc  = 100.0 * tr_correct / tr_total
            vl_acc  = 100.0 * vl_correct / vl_total
            tr_loss /= len(tr_loader)
            vl_loss /= len(vl_loader)
            cur_lr   = optimizer.param_groups[0]["lr"]

            history.append({
                "epoch"     : epoch + 1,
                "train_loss": round(tr_loss, 4),
                "train_acc" : round(tr_acc,  2),
                "val_loss"  : round(vl_loss, 4),
                "val_acc"   : round(vl_acc,  2),
                "lr"        : cur_lr,
            })

            print(f"      Ép {epoch+1:>3}/{EPOCHS_PER_FOLD} │ "
                  f"tr={tr_loss:.4f}/{tr_acc:.2f}% │ "
                  f"vl={vl_loss:.4f}/{vl_acc:.2f}% │ lr={cur_lr:.2e}")

            scheduler.step(vl_acc)

            if vl_acc > best_val_acc:
                best_val_acc      = vl_acc
                best_state        = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f"      🛑 Early stopping na época {epoch+1}")
                    break

        elapsed_fold = time.time() - t0_fold
        print(f"\n  ✅ Fold {fold_i+1}/{K_FOLDS}  "
              f"val_acc={best_val_acc:.2f}%  ({elapsed_fold:.0f}s)")

        fold_result = {
            # ── Identificação associada ao modelo ─────────────────────────────
            "modelo_alvo" : MODELO_ALVO,
            "num_classes" : num_classes,
            # ── Arquitetura ───────────────────────────────────────────────────
            "label"       : label,
            "arch_idx"    : arch_idx,
            "fold"        : fold_i,
            # ── Resultado ─────────────────────────────────────────────────────
            "best_val_acc": round(best_val_acc, 4),
            "elapsed_s"   : round(elapsed_fold, 1),
            "n_params"    : n_params,
            "fold_seed"   : fold_seed,
            # ── Hiperparâmetros ───────────────────────────────────────────────
            "arch"        : arch,
            "n_layers"    : len(arch),
            "filters"     : [b["out_channels"] for b in arch],
            "classifier"  : CLASSIFIER_HEAD,
            "dropout"     : DROPOUT,
            "lr"          : LEARNING_RATE,
            # ── Histórico ─────────────────────────────────────────────────────
            "history"     : history,
        }

        save_fold_result(fold_result)
        all_fold_results.append(fold_result)
        done_set.add((label, fold_i))
        update_summary()

        model.cpu()
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Estatísticas ao completar todos os folds
    accs_arch = [r["best_val_acc"] for r in all_fold_results
                 if r["label"] == label]
    if len(accs_arch) == K_FOLDS:
        print(f"\n  📊 {MODELO_ALVO} │ {label}")
        print(f"     Folds     : {[round(a,2) for a in accs_arch]}")
        print(f"     Média     : {np.mean(accs_arch):.2f}%")
        print(f"     Std       : {np.std(accs_arch):.2f}%")
        print(f"     Variância : {np.var(accs_arch):.4f}")
        print(f"     Tempo     : {time.time()-t0_arch:.0f}s")
        if np.mean(accs_arch) > best_mean:
            best_mean = np.mean(accs_arch)
            print(f"  🏆 Novo melhor: {label}  (média={best_mean:.2f}%)")

# ── Sumário final ─────────────────────────────────────────────────────────────
summary = update_summary()

print(f"\n{sep}")
print(f"  🏆  RANKING FINAL — {MODELO_ALVO}  ({num_classes} classes)")
print(f"{'─'*65}")
print(f"  {'Arquitetura':<42} {'Folds':>6} {'Média':>8}"
      f" {'Std':>7} {'Var':>8}")
print(f"{'─'*65}")
for s in summary[:5]:
    flag   = "✅" if s["complete"] else "⏳"
    status = f"{s['folds_done']}/{K_FOLDS}"
    print(f"  {flag} {s['label'][:40]:<40} {status:>6}"
          f" {s['mean_val_acc']:>7.2f}%"
          f" {s['std_val_acc']:>6.2f}%"
          f" {s['var_val_acc']:>8.4f}")
print(f"{'─'*65}")
print(f"  Resultados : {results_path}")
print(f"  Sumário    : {summary_path}")
print(sep)

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║      BUSCA DE ARQUITETURA — CNN Conv1D com K-Fold Cross-Validation (k=5)     ║
# ║                                                                              ║
# ║  Foco: variar quantidade de camadas Conv1D e número de filtros               ║
# ║                                                                              ║
# ║  Pré-requisitos (células anteriores já executadas):                          ║
# ║    • train_dataset, val_dataset, test_loader definidos                       ║
# ║    • H5PyDataset disponível (ou qualquer Dataset compatível)                 ║
# ║    • num_classes definido                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset
from sklearn.model_selection import KFold

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO DA BUSCA — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

SEARCH_CONFIG = {

    # ── Cross-Validation ──────────────────────────────────────────────────────
    "k_folds":              5,       # número de folds
    "epochs_per_fold":      30,      # épocas máximas por fold
    "early_stop_patience":  8,       # paciência do early stopping

    # ── Hiperparâmetros fixos (foco está nas arquiteturas) ────────────────────
    "dropout":              0.5,
    "learning_rate":        1e-3,
    "batch_size":           64,
    "classifier_head":      [512],   # única camada densa oculta
    "scheduler":            "plateau",
    "scheduler_patience":   4,

    # ── Arquiteturas candidatas (quantidade de camadas × filtros) ─────────────
    #    Cada entrada é uma lista de dicts: {out_channels, kernel_size, pool}
    #    "pool": True → MaxPool1d(2); False → sem pooling
    "architectures": [

        # ── 2 camadas — muito rasa ────────────────────────────────────────────
        {
            "label": "2L_32-64",
            "arch": [
                {"out_channels": 32,  "kernel_size": 7, "pool": True},
                {"out_channels": 64,  "kernel_size": 5, "pool": True},
            ],
        },
        {
            "label": "2L_64-128",
            "arch": [
                {"out_channels": 64,  "kernel_size": 7, "pool": True},
                {"out_channels": 128, "kernel_size": 5, "pool": True},
            ],
        },

        # ── 3 camadas — rasa ──────────────────────────────────────────────────
        {
            "label": "3L_32-64-128",
            "arch": [
                {"out_channels": 32,  "kernel_size": 7, "pool": True},
                {"out_channels": 64,  "kernel_size": 5, "pool": True},
                {"out_channels": 128, "kernel_size": 3, "pool": True},
            ],
        },
        {
            "label": "3L_64-128-256",
            "arch": [
                {"out_channels": 64,  "kernel_size": 7, "pool": True},
                {"out_channels": 128, "kernel_size": 5, "pool": True},
                {"out_channels": 256, "kernel_size": 3, "pool": True},
            ],
        },
        {
            "label": "3L_128-256-512",
            "arch": [
                {"out_channels": 128, "kernel_size": 7, "pool": True},
                {"out_channels": 256, "kernel_size": 5, "pool": True},
                {"out_channels": 512, "kernel_size": 3, "pool": True},
            ],
        },

        # ── 4 camadas — média ─────────────────────────────────────────────────
        {
            "label": "4L_32-64-128-256",
            "arch": [
                {"out_channels": 32,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": True},
                {"out_channels": 128, "kernel_size": 5,  "pool": True},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "4L_64-128-256-512",
            "arch": [
                {"out_channels": 64,  "kernel_size": 11, "pool": True},
                {"out_channels": 128, "kernel_size": 7,  "pool": True},
                {"out_channels": 256, "kernel_size": 5,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "4L_128-256-512-512",
            "arch": [
                {"out_channels": 128, "kernel_size": 11, "pool": True},
                {"out_channels": 256, "kernel_size": 7,  "pool": True},
                {"out_channels": 512, "kernel_size": 5,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },

        # ── 5 camadas — profunda ──────────────────────────────────────────────
        {
            "label": "5L_32-64-128-256-512",
            "arch": [
                {"out_channels": 32,   "kernel_size": 11, "pool": True},
                {"out_channels": 64,   "kernel_size": 7,  "pool": True},
                {"out_channels": 128,  "kernel_size": 5,  "pool": True},
                {"out_channels": 256,  "kernel_size": 3,  "pool": True},
                {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "5L_64-128-256-512-1024",
            "arch": [
                {"out_channels": 64,   "kernel_size": 11, "pool": True},
                {"out_channels": 128,  "kernel_size": 7,  "pool": True},
                {"out_channels": 256,  "kernel_size": 5,  "pool": True},
                {"out_channels": 512,  "kernel_size": 3,  "pool": True},
                {"out_channels": 1024, "kernel_size": 3,  "pool": True},
            ],
        },

        # ── 6 camadas — profunda com blocos sem pooling ───────────────────────
        {
            "label": "6L_64-64-128-128-256-512_mixpool",
            "arch": [
                {"out_channels": 64,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": False},
                {"out_channels": 128, "kernel_size": 5,  "pool": True},
                {"out_channels": 128, "kernel_size": 3,  "pool": False},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "6L_32-64-64-128-256-512_mixpool",
            "arch": [
                {"out_channels": 32,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": False},
                {"out_channels": 64,  "kernel_size": 5,  "pool": True},
                {"out_channels": 128, "kernel_size": 3,  "pool": False},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
    ],

    # ── Salvar resultados ─────────────────────────────────────────────────────
    "results_dir":  "/content",
    "results_file": "kfold_search_results.json",
}

# ══════════════════════════════════════════════════════════════════════════════
# CNN FLEXÍVEL
# ══════════════════════════════════════════════════════════════════════════════

class FlexCNN(nn.Module):
    """
    CNN Conv1D com número de camadas e filtros variáveis.

    Args:
        num_classes   : número de classes de saída
        arch          : lista de dicts {out_channels, kernel_size, pool}
        classifier    : lista de inteiros — neurônios das camadas densas ocultas
        dropout       : taxa de Dropout nas camadas densas
        in_channels   : canais de entrada (padrão 2 — I/Q do RadioML)
        input_length  : comprimento temporal da série (padrão 1024)
    """

    def __init__(
        self,
        num_classes:  int,
        arch:         list,
        classifier:   list  = None,
        dropout:      float = 0.5,
        in_channels:  int   = 2,
        input_length: int   = 1024,
    ):
        super().__init__()
        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels

        for block in arch:
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            pad    = ks // 2          # padding "same" aproximado

            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=pad),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))
            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        # Detecta o tamanho do flatten com um forward pass dummy
        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, input_length)
            n_flat = self.features(dummy).view(1, -1).size(1)

        head = []
        prev = n_flat
        for units in classifier:
            head += [nn.Linear(prev, units), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = units
        head.append(nn.Linear(prev, num_classes))
        self.classifier = nn.Sequential(*head)

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))


# ══════════════════════════════════════════════════════════════════════════════
# TREINO DE UM FOLD
# ══════════════════════════════════════════════════════════════════════════════

def train_fold(model, tr_loader, vl_loader, lr, epochs,
               early_stop_patience, scheduler_cfg, scheduler_patience, device):
    """
    Treina 'model' em um único fold.
    Retorna (best_val_acc, history).
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    if scheduler_cfg == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=scheduler_patience)
    elif scheduler_cfg == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-6)
    else:
        scheduler = None

    best_val_acc      = 0.0
    best_state        = None
    epochs_no_improve = 0
    history           = []

    model.to(device)

    for epoch in range(epochs):

        # ── Treino ──────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_correct = tr_total = 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            pred        = out.argmax(1)
            tr_correct += (pred == yb).sum().item()
            tr_total   += yb.size(0)

        # ── Validação ────────────────────────────────────────────────────────
        model.eval()
        vl_loss = vl_correct = vl_total = 0
        with torch.no_grad():
            for xb, yb in vl_loader:
                xb, yb = xb.to(device), yb.to(device)
                out     = model(xb)
                loss    = criterion(out, yb)
                vl_loss    += loss.item()
                vl_correct += (out.argmax(1) == yb).sum().item()
                vl_total   += yb.size(0)

        tr_acc  = 100. * tr_correct / tr_total
        vl_acc  = 100. * vl_correct / vl_total
        tr_loss /= len(tr_loader)
        vl_loss /= len(vl_loader)
        cur_lr  = optimizer.param_groups[0]['lr']

        history.append({
            "epoch":      epoch + 1,
            "train_loss": round(tr_loss, 4),
            "train_acc":  round(tr_acc,  2),
            "val_loss":   round(vl_loss, 4),
            "val_acc":    round(vl_acc,  2),
            "lr":         cur_lr,
        })

        print(f"      Ép {epoch+1:>3}/{epochs} │ "
              f"tr={tr_loss:.4f}/{tr_acc:.2f}% │ "
              f"vl={vl_loss:.4f}/{vl_acc:.2f}% │ lr={cur_lr:.2e}")

        if vl_acc > best_val_acc:
            best_val_acc      = vl_acc
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(vl_acc)
            else:
                scheduler.step()

        if epochs_no_improve >= early_stop_patience:
            print(f"      🛑 Early stopping na época {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_acc, history


# ══════════════════════════════════════════════════════════════════════════════
# BUSCA POR ARQUITETURA COM K-FOLD CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def architecture_search_kfold(num_classes, config=SEARCH_CONFIG):
    """
    Para cada arquitetura candidata, executa k-fold CV e registra:
      • val_acc de cada fold
      • média e desvio-padrão das val_acc

    Salva o melhor modelo (maior média de val_acc entre os folds).

    Args:
        num_classes : número de classes
        config      : dicionário SEARCH_CONFIG

    Returns:
        all_results : lista de resultados ordenada por mean_val_acc (desc.)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    k      = config["k_folds"]

    print(f"\n{'═'*70}")
    print(f"  🔍  BUSCA DE ARQUITETURA — CNN Conv1D com {k}-Fold CV")
    print(f"  Dispositivo  : {device}")
    print(f"  Classes      : {num_classes}")
    print(f"  Arquiteturas : {len(config['architectures'])}")
    print(f"  Épocas/fold  : {config['epochs_per_fold']}")
    print(f"{'═'*70}\n")

    os.makedirs(config["results_dir"], exist_ok=True)
    results_path = os.path.join(config["results_dir"], config["results_file"])

    # Combina train + val para o CV (o test_loader permanece intocado)
    full_dataset = ConcatDataset([train_dataset, val_dataset])
    kf           = KFold(n_splits=k, shuffle=True, random_state=42)
    indices      = list(range(len(full_dataset)))

    all_results  = []
    best_mean    = -1.0
    best_model_state = None
    best_cfg_saved   = None

    for arch_idx, arch_entry in enumerate(config["architectures"], 1):
        label = arch_entry["label"]
        arch  = arch_entry["arch"]

        print(f"\n{'═'*70}")
        print(f"  Arquitetura {arch_idx}/{len(config['architectures'])} : {label}")
        print(f"  Camadas: {len(arch)}  |  Filtros: "
              + " → ".join(str(b["out_channels"]) for b in arch))
        print(f"{'═'*70}")

        fold_accs    = []
        fold_history = []
        best_fold_model_state = None

        t0_arch = time.time()

        for fold, (tr_idx, vl_idx) in enumerate(kf.split(indices), 1):
            print(f"\n  ── Fold {fold}/{k} ─────────────────────────────────────")

            tr_loader = DataLoader(
                Subset(full_dataset, tr_idx),
                batch_size  = config["batch_size"],
                shuffle     = True,
                drop_last   = False,
                num_workers = 0,
            )
            vl_loader = DataLoader(
                Subset(full_dataset, vl_idx),
                batch_size  = config["batch_size"],
                shuffle     = False,
                num_workers = 0,
            )

            torch.manual_seed(42)
            model = FlexCNN(
                num_classes = num_classes,
                arch        = arch,
                classifier  = config["classifier_head"],
                dropout     = config["dropout"],
            )
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Parâmetros: {n_params:,}")

            val_acc, history = train_fold(
                model               = model,
                tr_loader           = tr_loader,
                vl_loader           = vl_loader,
                lr                  = config["learning_rate"],
                epochs              = config["epochs_per_fold"],
                early_stop_patience = config["early_stop_patience"],
                scheduler_cfg       = config["scheduler"],
                scheduler_patience  = config["scheduler_patience"],
                device              = device,
            )

            print(f"\n  ✅ Fold {fold} | val_acc = {val_acc:.2f}%")
            fold_accs.append(val_acc)
            fold_history.append(history)

            # Guarda estado do melhor fold desta arquitetura
            if val_acc == max(fold_accs):
                best_fold_model_state = copy.deepcopy(model.state_dict())

            model.cpu()
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ── Estatísticas da arquitetura ────────────────────────────────────────
        elapsed   = time.time() - t0_arch
        mean_acc  = float(np.mean(fold_accs))
        std_acc   = float(np.std(fold_accs))

        print(f"\n  📊 {label}")
        print(f"     Fold accs : {[round(a, 2) for a in fold_accs]}")
        print(f"     Média     : {mean_acc:.2f}%  ±  {std_acc:.2f}%")
        print(f"     Tempo     : {elapsed:.0f}s")

        result = {
            "arch_idx":      arch_idx,
            "label":         label,
            "arch":          arch,
            "n_layers":      len(arch),
            "filters":       [b["out_channels"] for b in arch],
            "classifier":    config["classifier_head"],
            "dropout":       config["dropout"],
            "lr":            config["learning_rate"],
            "batch_size":    config["batch_size"],
            "fold_val_accs": [round(a, 4) for a in fold_accs],
            "mean_val_acc":  round(mean_acc, 4),
            "std_val_acc":   round(std_acc,  4),
            "elapsed_s":     round(elapsed,  1),
            "fold_histories": fold_history,
        }
        all_results.append(result)

        # Salva parcialmente
        with open(results_path, "w") as f:
            json.dump(all_results, f, indent=2)

        # Melhor arquitetura global
        if mean_acc > best_mean:
            best_mean            = mean_acc
            best_model_state     = copy.deepcopy(best_fold_model_state)
            best_cfg_saved       = result.copy()
            best_cfg_saved.pop("fold_histories")

            ckpt_path = os.path.join(config["results_dir"], "best_model.pth")
            torch.save({
                "model_state_dict": best_model_state,
                "config":           best_cfg_saved,
                "num_classes":      num_classes,
            }, ckpt_path)
            print(f"  💾 Novo melhor modelo salvo → {ckpt_path}")
            print(f"  🏆 Melhor média até agora : {best_mean:.2f}%")

    # ── Resumo final ───────────────────────────────────────────────────────────
    all_results.sort(key=lambda r: r["mean_val_acc"], reverse=True)

    print(f"\n{'═'*70}")
    print(f"  🏆  BUSCA CONCLUÍDA — Top-5 arquiteturas (por média de val_acc)")
    print(f"{'═'*70}")
    for rank, r in enumerate(all_results[:5], 1):
        print(f"  {rank}. [{r['label']}]")
        print(f"     Camadas : {r['n_layers']}  |  Filtros : {r['filters']}")
        print(f"     Média   : {r['mean_val_acc']:.2f}%  ±  {r['std_val_acc']:.2f}%")
        print(f"     Folds   : {r['fold_val_accs']}")

    print(f"\n  Resultados completos : {results_path}")
    print(f"  Melhor modelo (pth)  : "
          f"{os.path.join(config['results_dir'], 'best_model.pth')}")

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# RECONSTRUÇÃO DO MELHOR MODELO
# ══════════════════════════════════════════════════════════════════════════════

def load_best_model(ckpt_path=None, results_dir=None):
    """
    Carrega o melhor modelo salvo durante a busca.

    Returns:
        model (FlexCNN) pronto para inferência / fine-tuning
    """
    if ckpt_path is None:
        results_dir = results_dir or SEARCH_CONFIG["results_dir"]
        ckpt_path   = os.path.join(results_dir, "best_model.pth")

    ckpt  = torch.load(ckpt_path, map_location="cpu")
    cfg   = ckpt["config"]
    model = FlexCNN(
        num_classes = ckpt["num_classes"],
        arch        = cfg["arch"],
        classifier  = cfg["classifier"],
        dropout     = cfg["dropout"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    print(f"✅ Melhor modelo carregado de: {ckpt_path}")
    print(f"   Arquitetura  : {cfg['label']}")
    print(f"   Camadas      : {cfg['n_layers']}  |  Filtros : {cfg['filters']}")
    print(f"   mean_val_acc : {cfg['mean_val_acc']:.2f}%  ±  {cfg['std_val_acc']:.2f}%")
    return model


# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_test(model, test_loader):
    """Avalia o modelo no conjunto de teste e retorna acurácia."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    acc = 100. * correct / total
    print(f"\n🎯 Test Accuracy : {acc:.2f}%  ({correct}/{total})")
    return acc


# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  EXECUÇÃO PRINCIPAL  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # num_classes deve estar definido de células anteriores.
    # Ajuste manualmente se executar de forma isolada:
    # num_classes = 6   # GROUP
    # num_classes = 3   # ASK

    all_results = architecture_search_kfold(num_classes=num_classes)

    # Carrega e avalia o melhor modelo no conjunto de teste
    best = load_best_model()
    evaluate_test(best, test_loader)

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║      BUSCA DE ARQUITETURA — CNN Conv1D com K-Fold Cross-Validation (k=5)    ║
# ║                                                                              ║
# ║  Foco: variar quantidade de camadas Conv1D e número de filtros               ║
# ║                                                                              ║
# ║  Pré-requisitos (células anteriores já executadas):                          ║
# ║    • train_dataset, val_dataset, test_loader definidos                       ║
# ║    • H5PyDataset disponível (ou qualquer Dataset compatível)                 ║
# ║    • num_classes definido                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset
from sklearn.model_selection import KFold

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO DA BUSCA — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

SEARCH_CONFIG = {

    # ── Cross-Validation ──────────────────────────────────────────────────────
    "k_folds":              5,       # número de folds
    "epochs_per_fold":      30,      # épocas máximas por fold
    "early_stop_patience":  8,       # paciência do early stopping

    # ── Hiperparâmetros fixos (foco está nas arquiteturas) ────────────────────
    "dropout":              0.5,
    "learning_rate":        1e-3,
    "batch_size":           64,
    "classifier_head":      [512],   # única camada densa oculta
    # LR cai pela metade a cada 3 épocas sem melhora na val_acc
    "scheduler_patience":   3,

    # ── Arquiteturas candidatas (quantidade de camadas × filtros) ─────────────
    #    Cada entrada é uma lista de dicts: {out_channels, kernel_size, pool}
    #    "pool": True → MaxPool1d(2); False → sem pooling
    "architectures": [

        # ── 2 camadas — muito rasa ────────────────────────────────────────────
        {
            "label": "2L_32-64",
            "arch": [
                {"out_channels": 32,  "kernel_size": 7, "pool": True},
                {"out_channels": 64,  "kernel_size": 5, "pool": True},
            ],
        },
        {
            "label": "2L_64-128",
            "arch": [
                {"out_channels": 64,  "kernel_size": 7, "pool": True},
                {"out_channels": 128, "kernel_size": 5, "pool": True},
            ],
        },

        # ── 3 camadas — rasa ──────────────────────────────────────────────────
        {
            "label": "3L_32-64-128",
            "arch": [
                {"out_channels": 32,  "kernel_size": 7, "pool": True},
                {"out_channels": 64,  "kernel_size": 5, "pool": True},
                {"out_channels": 128, "kernel_size": 3, "pool": True},
            ],
        },
        {
            "label": "3L_64-128-256",
            "arch": [
                {"out_channels": 64,  "kernel_size": 7, "pool": True},
                {"out_channels": 128, "kernel_size": 5, "pool": True},
                {"out_channels": 256, "kernel_size": 3, "pool": True},
            ],
        },
        {
            "label": "3L_128-256-512",
            "arch": [
                {"out_channels": 128, "kernel_size": 7, "pool": True},
                {"out_channels": 256, "kernel_size": 5, "pool": True},
                {"out_channels": 512, "kernel_size": 3, "pool": True},
            ],
        },

        # ── 4 camadas — média ─────────────────────────────────────────────────
        {
            "label": "4L_32-64-128-256",
            "arch": [
                {"out_channels": 32,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": True},
                {"out_channels": 128, "kernel_size": 5,  "pool": True},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "4L_64-128-256-512",
            "arch": [
                {"out_channels": 64,  "kernel_size": 11, "pool": True},
                {"out_channels": 128, "kernel_size": 7,  "pool": True},
                {"out_channels": 256, "kernel_size": 5,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "4L_128-256-512-512",
            "arch": [
                {"out_channels": 128, "kernel_size": 11, "pool": True},
                {"out_channels": 256, "kernel_size": 7,  "pool": True},
                {"out_channels": 512, "kernel_size": 5,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },

        # ── 5 camadas — profunda ──────────────────────────────────────────────
        {
            "label": "5L_32-64-128-256-512",
            "arch": [
                {"out_channels": 32,   "kernel_size": 11, "pool": True},
                {"out_channels": 64,   "kernel_size": 7,  "pool": True},
                {"out_channels": 128,  "kernel_size": 5,  "pool": True},
                {"out_channels": 256,  "kernel_size": 3,  "pool": True},
                {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "5L_64-128-256-512-1024",
            "arch": [
                {"out_channels": 64,   "kernel_size": 11, "pool": True},
                {"out_channels": 128,  "kernel_size": 7,  "pool": True},
                {"out_channels": 256,  "kernel_size": 5,  "pool": True},
                {"out_channels": 512,  "kernel_size": 3,  "pool": True},
                {"out_channels": 1024, "kernel_size": 3,  "pool": True},
            ],
        },

        # ── 6 camadas — profunda com blocos sem pooling ───────────────────────
        {
            "label": "6L_64-64-128-128-256-512_mixpool",
            "arch": [
                {"out_channels": 64,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": False},
                {"out_channels": 128, "kernel_size": 5,  "pool": True},
                {"out_channels": 128, "kernel_size": 3,  "pool": False},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
        {
            "label": "6L_32-64-64-128-256-512_mixpool",
            "arch": [
                {"out_channels": 32,  "kernel_size": 11, "pool": True},
                {"out_channels": 64,  "kernel_size": 7,  "pool": False},
                {"out_channels": 64,  "kernel_size": 5,  "pool": True},
                {"out_channels": 128, "kernel_size": 3,  "pool": False},
                {"out_channels": 256, "kernel_size": 3,  "pool": True},
                {"out_channels": 512, "kernel_size": 3,  "pool": True},
            ],
        },
    ],

    # ── Salvar resultados ─────────────────────────────────────────────────────
    "results_dir":  "/content",
    "results_file": "kfold_search_results.json",
}

# ══════════════════════════════════════════════════════════════════════════════
# CNN FLEXÍVEL
# ══════════════════════════════════════════════════════════════════════════════

class FlexCNN(nn.Module):
    """
    CNN Conv1D com número de camadas e filtros variáveis.

    Args:
        num_classes   : número de classes de saída
        arch          : lista de dicts {out_channels, kernel_size, pool}
        classifier    : lista de inteiros — neurônios das camadas densas ocultas
        dropout       : taxa de Dropout nas camadas densas
        in_channels   : canais de entrada (padrão 2 — I/Q do RadioML)
        input_length  : comprimento temporal da série (padrão 1024)
    """

    def __init__(
        self,
        num_classes:  int,
        arch:         list,
        classifier:   list  = None,
        dropout:      float = 0.5,
        in_channels:  int   = 2,
        input_length: int   = 1024,
    ):
        super().__init__()
        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels

        for block in arch:
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            pad    = ks // 2          # padding "same" aproximado

            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=pad),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))
            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        # Detecta o tamanho do flatten com um forward pass dummy
        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, input_length)
            n_flat = self.features(dummy).view(1, -1).size(1)

        head = []
        prev = n_flat
        for units in classifier:
            head += [nn.Linear(prev, units), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = units
        head.append(nn.Linear(prev, num_classes))
        self.classifier = nn.Sequential(*head)

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))


# ══════════════════════════════════════════════════════════════════════════════
# TREINO DE UM FOLD
# ══════════════════════════════════════════════════════════════════════════════

def train_fold(model, tr_loader, vl_loader, lr, epochs,
               early_stop_patience, scheduler_patience, device):
    """
    Treina 'model' em um único fold.
    O LR e reduzido a metade a cada 'scheduler_patience' epocas consecutivas
    sem melhora na val_acc (ReduceLROnPlateau, mode='max', factor=0.5).
    Retorna (best_val_acc, history).
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # LR / 2 apos 'scheduler_patience' epocas sem melhora
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5,
        patience=scheduler_patience, min_lr=1e-6,
    )

    best_val_acc      = 0.0
    best_state        = None
    epochs_no_improve = 0
    history           = []

    model.to(device)

    for epoch in range(epochs):

        # ── Treino ──────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_correct = tr_total = 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            pred        = out.argmax(1)
            tr_correct += (pred == yb).sum().item()
            tr_total   += yb.size(0)

        # ── Validação ────────────────────────────────────────────────────────
        model.eval()
        vl_loss = vl_correct = vl_total = 0
        with torch.no_grad():
            for xb, yb in vl_loader:
                xb, yb = xb.to(device), yb.to(device)
                out     = model(xb)
                loss    = criterion(out, yb)
                vl_loss    += loss.item()
                vl_correct += (out.argmax(1) == yb).sum().item()
                vl_total   += yb.size(0)

        tr_acc  = 100. * tr_correct / tr_total
        vl_acc  = 100. * vl_correct / vl_total
        tr_loss /= len(tr_loader)
        vl_loss /= len(vl_loader)
        cur_lr  = optimizer.param_groups[0]['lr']

        history.append({
            "epoch":      epoch + 1,
            "train_loss": round(tr_loss, 4),
            "train_acc":  round(tr_acc,  2),
            "val_loss":   round(vl_loss, 4),
            "val_acc":    round(vl_acc,  2),
            "lr":         cur_lr,
        })

        print(f"      Ép {epoch+1:>3}/{epochs} │ "
              f"tr={tr_loss:.4f}/{tr_acc:.2f}% │ "
              f"vl={vl_loss:.4f}/{vl_acc:.2f}% │ lr={cur_lr:.2e}")

        if vl_acc > best_val_acc:
            best_val_acc      = vl_acc
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        scheduler.step(vl_acc)

        if epochs_no_improve >= early_stop_patience:
            print(f"      🛑 Early stopping na época {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_acc, history


# ══════════════════════════════════════════════════════════════════════════════
# BUSCA POR ARQUITETURA COM K-FOLD CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def architecture_search_kfold(num_classes, config=SEARCH_CONFIG):
    """
    Para cada arquitetura candidata, executa k-fold CV e registra:
      • val_acc de cada fold
      • média e desvio-padrão das val_acc

    Salva o melhor modelo (maior média de val_acc entre os folds).

    Args:
        num_classes : número de classes
        config      : dicionário SEARCH_CONFIG

    Returns:
        all_results : lista de resultados ordenada por mean_val_acc (desc.)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    k      = config["k_folds"]

    print(f"\n{'═'*70}")
    print(f"  🔍  BUSCA DE ARQUITETURA — CNN Conv1D com {k}-Fold CV")
    print(f"  Dispositivo  : {device}")
    print(f"  Classes      : {num_classes}")
    print(f"  Arquiteturas : {len(config['architectures'])}")
    print(f"  Épocas/fold  : {config['epochs_per_fold']}")
    print(f"{'═'*70}\n")

    os.makedirs(config["results_dir"], exist_ok=True)
    results_path = os.path.join(config["results_dir"], config["results_file"])

    # Combina train + val para o CV (o test_loader permanece intocado)
    full_dataset = ConcatDataset([train_dataset, val_dataset])
    kf           = KFold(n_splits=k, shuffle=True, random_state=42)
    indices      = list(range(len(full_dataset)))

    all_results  = []
    best_mean    = -1.0
    best_model_state = None
    best_cfg_saved   = None

    for arch_idx, arch_entry in enumerate(config["architectures"], 1):
        label = arch_entry["label"]
        arch  = arch_entry["arch"]

        print(f"\n{'═'*70}")
        print(f"  Arquitetura {arch_idx}/{len(config['architectures'])} : {label}")
        print(f"  Camadas: {len(arch)}  |  Filtros: "
              + " → ".join(str(b["out_channels"]) for b in arch))
        print(f"{'═'*70}")

        fold_accs    = []
        fold_history = []
        best_fold_model_state = None

        t0_arch = time.time()

        for fold, (tr_idx, vl_idx) in enumerate(kf.split(indices), 1):
            print(f"\n  ── Fold {fold}/{k} ─────────────────────────────────────")

            tr_loader = DataLoader(
                Subset(full_dataset, tr_idx),
                batch_size  = config["batch_size"],
                shuffle     = True,
                drop_last   = False,
                num_workers = 0,
            )
            vl_loader = DataLoader(
                Subset(full_dataset, vl_idx),
                batch_size  = config["batch_size"],
                shuffle     = False,
                num_workers = 0,
            )

            torch.manual_seed(42)
            model = FlexCNN(
                num_classes = num_classes,
                arch        = arch,
                classifier  = config["classifier_head"],
                dropout     = config["dropout"],
            )
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Parâmetros: {n_params:,}")

            val_acc, history = train_fold(
                model               = model,
                tr_loader           = tr_loader,
                vl_loader           = vl_loader,
                lr                  = config["learning_rate"],
                epochs              = config["epochs_per_fold"],
                early_stop_patience = config["early_stop_patience"],
                scheduler_patience  = config["scheduler_patience"],
                device              = device,
            )

            print(f"\n  ✅ Fold {fold} | val_acc = {val_acc:.2f}%")
            fold_accs.append(val_acc)
            fold_history.append(history)

            # Guarda estado do melhor fold desta arquitetura
            if val_acc == max(fold_accs):
                best_fold_model_state = copy.deepcopy(model.state_dict())

            model.cpu()
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ── Estatísticas da arquitetura ────────────────────────────────────────
        elapsed   = time.time() - t0_arch
        mean_acc  = float(np.mean(fold_accs))
        std_acc   = float(np.std(fold_accs))

        print(f"\n  📊 {label}")
        print(f"     Fold accs : {[round(a, 2) for a in fold_accs]}")
        print(f"     Média     : {mean_acc:.2f}%  ±  {std_acc:.2f}%")
        print(f"     Tempo     : {elapsed:.0f}s")

        result = {
            "arch_idx":      arch_idx,
            "label":         label,
            "arch":          arch,
            "n_layers":      len(arch),
            "filters":       [b["out_channels"] for b in arch],
            "classifier":    config["classifier_head"],
            "dropout":       config["dropout"],
            "lr":            config["learning_rate"],
            "batch_size":    config["batch_size"],
            "fold_val_accs": [round(a, 4) for a in fold_accs],
            "mean_val_acc":  round(mean_acc, 4),
            "std_val_acc":   round(std_acc,  4),
            "elapsed_s":     round(elapsed,  1),
            "fold_histories": fold_history,
        }
        all_results.append(result)

        # Salva parcialmente
        with open(results_path, "w") as f:
            json.dump(all_results, f, indent=2)

        # Melhor arquitetura global
        if mean_acc > best_mean:
            best_mean            = mean_acc
            best_model_state     = copy.deepcopy(best_fold_model_state)
            best_cfg_saved       = result.copy()
            best_cfg_saved.pop("fold_histories")

            ckpt_path = os.path.join(config["results_dir"], "best_model.pth")
            torch.save({
                "model_state_dict": best_model_state,
                "config":           best_cfg_saved,
                "num_classes":      num_classes,
            }, ckpt_path)
            print(f"  💾 Novo melhor modelo salvo → {ckpt_path}")
            print(f"  🏆 Melhor média até agora : {best_mean:.2f}%")

    # ── Resumo final ───────────────────────────────────────────────────────────
    all_results.sort(key=lambda r: r["mean_val_acc"], reverse=True)

    print(f"\n{'═'*70}")
    print(f"  🏆  BUSCA CONCLUÍDA — Top-5 arquiteturas (por média de val_acc)")
    print(f"{'═'*70}")
    for rank, r in enumerate(all_results[:5], 1):
        print(f"  {rank}. [{r['label']}]")
        print(f"     Camadas : {r['n_layers']}  |  Filtros : {r['filters']}")
        print(f"     Média   : {r['mean_val_acc']:.2f}%  ±  {r['std_val_acc']:.2f}%")
        print(f"     Folds   : {r['fold_val_accs']}")

    print(f"\n  Resultados completos : {results_path}")
    print(f"  Melhor modelo (pth)  : "
          f"{os.path.join(config['results_dir'], 'best_model.pth')}")

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# RECONSTRUÇÃO DO MELHOR MODELO
# ══════════════════════════════════════════════════════════════════════════════

def load_best_model(ckpt_path=None, results_dir=None):
    """
    Carrega o melhor modelo salvo durante a busca.

    Returns:
        model (FlexCNN) pronto para inferência / fine-tuning
    """
    if ckpt_path is None:
        results_dir = results_dir or SEARCH_CONFIG["results_dir"]
        ckpt_path   = os.path.join(results_dir, "best_model.pth")

    ckpt  = torch.load(ckpt_path, map_location="cpu")
    cfg   = ckpt["config"]
    model = FlexCNN(
        num_classes = ckpt["num_classes"],
        arch        = cfg["arch"],
        classifier  = cfg["classifier"],
        dropout     = cfg["dropout"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    print(f"✅ Melhor modelo carregado de: {ckpt_path}")
    print(f"   Arquitetura  : {cfg['label']}")
    print(f"   Camadas      : {cfg['n_layers']}  |  Filtros : {cfg['filters']}")
    print(f"   mean_val_acc : {cfg['mean_val_acc']:.2f}%  ±  {cfg['std_val_acc']:.2f}%")
    return model


# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_test(model, test_loader):
    """Avalia o modelo no conjunto de teste e retorna acurácia."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    acc = 100. * correct / total
    print(f"\n🎯 Test Accuracy : {acc:.2f}%  ({correct}/{total})")
    return acc


# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  EXECUÇÃO PRINCIPAL  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # num_classes deve estar definido de células anteriores.
    # Ajuste manualmente se executar de forma isolada:
    # num_classes = 6   # GROUP
    # num_classes = 3   # ASK

    all_results = architecture_search_kfold(num_classes=num_classes)

    # Carrega e avalia o melhor modelo no conjunto de teste
    best = load_best_model()
    evaluate_test(best, test_loader)

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║         BLOCO DE BUSCA DE HIPERPARÂMETROS — CNN Conv1D                      ║
# ║                                                                              ║
# ║  Pré-requisitos (células anteriores já executadas):                          ║
# ║    • train_loader, val_loader, test_loader definidos                         ║
# ║    • H5PyDataset, split(), train_and_validate() definidos                    ║
# ║    • h5py_path, train/val/test_indices_sorted disponíveis                    ║
# ║    • num_classes definido (6 para GROUP, ou N para intra-grupo)              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, itertools, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from dataclasses import dataclass, field, asdict
from typing import List, Optional

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO DA BUSCA — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

SEARCH_CONFIG = {

    # ── Épocas ────────────────────────────────────────────────────────────────
    "epochs_per_trial": 30,          # épocas máximas por combinação
    "early_stop_patience": 8,        # para early stopping dentro do trial

    # ── Número de camadas Conv1D ───────────────────────────────────────────────
    # Cada elemento define uma arquitetura completa:
    #   lista de dicts com  {out_channels, kernel_size, pool}
    # "pool": True → MaxPool1d(2) após a conv; False → sem pooling
    "architectures": [

        # ── 3 camadas (rasa) ─────────────────────────────────────────────────
        [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
            {"out_channels": 256, "kernel_size": 3, "pool": True},
        ],

        # ── 4 camadas (média) ────────────────────────────────────────────────
        [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 128, "kernel_size": 7,  "pool": True},
            {"out_channels": 256, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],

        # ── 5 camadas (profunda — igual ao modelo original) ──────────────────
        [
            {"out_channels": 64,   "kernel_size": 11, "pool": True},
            {"out_channels": 128,  "kernel_size": 7,  "pool": True},
            {"out_channels": 256,  "kernel_size": 5,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            {"out_channels": 1024, "kernel_size": 3,  "pool": True},
        ],

        # ── 5 camadas (canais menores, menos parâmetros) ─────────────────────
        [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": True},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],

        # ── 6 camadas (profunda com passo fino) ──────────────────────────────
        [
            {"out_channels": 64,   "kernel_size": 11, "pool": True},
            {"out_channels": 64,   "kernel_size": 7,  "pool": False},  # sem pool
            {"out_channels": 128,  "kernel_size": 5,  "pool": True},
            {"out_channels": 128,  "kernel_size": 3,  "pool": False},  # sem pool
            {"out_channels": 256,  "kernel_size": 3,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
        ],
    ],

    # ── Cabeça classificadora (Dense após Flatten) ────────────────────────────
    "classifier_heads": [
        [512],           # 1 camada oculta
        [512, 256],      # 2 camadas ocultas
        [1024, 512],     # 2 camadas ocultas maiores
    ],

    # ── Dropout ───────────────────────────────────────────────────────────────
    "dropout_rates": [0.3, 0.5],

    # ── Learning rate inicial ─────────────────────────────────────────────────
    "learning_rates": [1e-3, 5e-4],

    # ── Batch size ────────────────────────────────────────────────────────────
    "batch_sizes": [64],             # adicione 128 se quiser explorar

    # ── Scheduler ────────────────────────────────────────────────────────────
    # "plateau" → ReduceLROnPlateau;  "cosine" → CosineAnnealingLR;  None
    "scheduler": "plateau",
    "scheduler_patience": 4,         # só usado se scheduler == "plateau"

    # ── Salvar resultados ─────────────────────────────────────────────────────
    "results_dir": "/content",       # onde salvar JSONs e o melhor modelo
    "results_file": "search_results.json",
}

# ══════════════════════════════════════════════════════════════════════════════
# CNN FLEXÍVEL — varia camadas e hiperparâmetros dinamicamente
# ══════════════════════════════════════════════════════════════════════════════

class FlexCNN(nn.Module):
    """
    CNN Conv1D com número de camadas e hiperparâmetros variáveis.

    Args:
        num_classes   : número de classes de saída
        arch          : lista de dicts com {out_channels, kernel_size, pool}
        classifier    : lista de inteiros com nº de neurônios das camadas Dense
        dropout       : taxa de Dropout nas camadas densas
        in_channels   : canais de entrada (padrão 2 — I e Q do RadioML)
        input_length  : comprimento temporal da série (padrão 1024)
    """

    def __init__(
        self,
        num_classes:   int,
        arch:          list,
        classifier:    list  = None,
        dropout:       float = 0.5,
        in_channels:   int   = 2,
        input_length:  int   = 1024,
    ):
        super().__init__()

        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels

        for i, block in enumerate(arch):
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            pad    = ks // 2          # padding "same" aproximado

            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=pad),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))

            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        # Detecta o tamanho do flatten com um forward pass dummy
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, input_length)
            flat  = self.features(dummy).view(1, -1)
            n_flat = flat.size(1)

        # Cabeça classificadora
        head = []
        prev = n_flat
        for units in classifier:
            head += [
                nn.Linear(prev, units),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            prev = units
        head.append(nn.Linear(prev, num_classes))

        self.classifier = nn.Sequential(*head)

        print(f"  FlexCNN criada: {len(arch)} blocos conv | "
              f"flatten={n_flat} | head={classifier} | dropout={dropout}")

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        return self.classifier(x)


# ══════════════════════════════════════════════════════════════════════════════
# LOOP DE TREINO POR TRIAL (versão leve — sem salvar checkpoint por época)
# ══════════════════════════════════════════════════════════════════════════════

def run_trial(model, train_loader, val_loader, lr, epochs,
              early_stop_patience, scheduler_cfg, device):
    """
    Treina 'model' e retorna (best_val_acc, history).
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    if scheduler_cfg == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5,
            patience=SEARCH_CONFIG["scheduler_patience"]
        )
    elif scheduler_cfg == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-6
        )
    else:
        scheduler = None

    best_val_acc     = 0.0
    best_state       = None
    epochs_no_improve = 0
    history          = []

    model.to(device)

    for epoch in range(epochs):
        # ── Treino ──────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_correct = tr_total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            tr_loss    += loss.item()
            pred        = out.argmax(1)
            tr_correct += (pred == yb).sum().item()
            tr_total   += yb.size(0)

        # ── Validação ────────────────────────────────────────────────────────
        model.eval()
        vl_loss = vl_correct = vl_total = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out     = model(xb)
                loss    = criterion(out, yb)
                vl_loss    += loss.item()
                pred        = out.argmax(1)
                vl_correct += (pred == yb).sum().item()
                vl_total   += yb.size(0)

        tr_acc  = 100. * tr_correct / tr_total
        vl_acc  = 100. * vl_correct / vl_total
        tr_loss /= len(train_loader)
        vl_loss /= len(val_loader)
        cur_lr  = optimizer.param_groups[0]['lr']

        history.append({
            "epoch": epoch + 1,
            "train_loss": round(tr_loss, 4),
            "train_acc":  round(tr_acc,  2),
            "val_loss":   round(vl_loss, 4),
            "val_acc":    round(vl_acc,  2),
            "lr":         cur_lr,
        })

        print(f"    Ép {epoch+1:>3}/{epochs} │ "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.2f}% │ "
              f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.2f}% │ lr={cur_lr:.2e}")

        # Checkpoint do melhor estado
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # Scheduler
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(vl_acc)
            else:
                scheduler.step()

        # Early stopping
        if epochs_no_improve >= early_stop_patience:
            print(f"    🛑 Early stopping na época {epoch+1}")
            break

    # Restaura melhor estado
    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_acc, history


# ══════════════════════════════════════════════════════════════════════════════
# BUSCA EXAUSTIVA (grid) SOBRE COMBINAÇÕES
# ══════════════════════════════════════════════════════════════════════════════

def build_loaders(batch_size):
    """Reconstrói DataLoaders com o batch_size desejado."""
    g = torch.Generator(); g.manual_seed(42)

    def _sw(wid):
        s = torch.initial_seed() % (2**32)
        np.random.seed(s); __import__('random').seed(s)

    tr = DataLoader(train_dataset, batch_size=batch_size,
                    shuffle=True, drop_last=False,
                    num_workers=0, generator=g, worker_init_fn=_sw)
    vl = DataLoader(val_dataset,   batch_size=batch_size,
                    shuffle=False, drop_last=False, num_workers=0)
    return tr, vl


def hyperparameter_search(num_classes, config=SEARCH_CONFIG):
    """
    Itera sobre todas as combinações e retorna a lista de resultados ordenada
    por val_acc decrescente.

    Ao final salva:
      • results_dir/results_file  (JSON com todos os trials)
      • results_dir/best_model.pth (checkpoint do melhor modelo)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'═'*70}")
    print(f"  🔍  BUSCA DE HIPERPARÂMETROS — CNN Conv1D")
    print(f"  Dispositivo : {device}")
    print(f"  Classes     : {num_classes}")
    print(f"{'═'*70}\n")

    os.makedirs(config["results_dir"], exist_ok=True)
    results_path = os.path.join(config["results_dir"], config["results_file"])

    # Produto cartesiano dos hiperparâmetros
    combos = list(itertools.product(
        config["architectures"],
        config["classifier_heads"],
        config["dropout_rates"],
        config["learning_rates"],
        config["batch_sizes"],
    ))

    total = len(combos)
    print(f"  Total de combinações : {total}\n")

    all_results  = []
    best_val_acc = -1.0
    best_model   = None

    for trial_idx, (arch, head, dropout, lr, bs) in enumerate(combos, 1):

        # Identificador legível
        arch_tag = f"{len(arch)}conv_" + "-".join(
            str(b["out_channels"]) for b in arch
        )
        head_tag = "-".join(str(h) for h in head)
        trial_id = (f"trial{trial_idx:03d}|arch={arch_tag}|"
                    f"head={head_tag}|drop={dropout}|lr={lr:.0e}|bs={bs}")

        print(f"\n{'─'*70}")
        print(f"  Trial {trial_idx}/{total}")
        print(f"  Arch     : {arch_tag}")
        print(f"  Head     : {head_tag}")
        print(f"  Dropout  : {dropout}   LR: {lr}   Batch: {bs}")
        print(f"{'─'*70}")

        # DataLoaders com o batch_size deste trial
        tr_loader, vl_loader = build_loaders(bs)

        # Modelo
        torch.manual_seed(42)
        model = FlexCNN(
            num_classes  = num_classes,
            arch         = arch,
            classifier   = head,
            dropout      = dropout,
        )

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parâmetros treináveis: {n_params:,}")

        t0 = time.time()
        val_acc, history = run_trial(
            model             = model,
            train_loader      = tr_loader,
            val_loader        = vl_loader,
            lr                = lr,
            epochs            = config["epochs_per_trial"],
            early_stop_patience = config["early_stop_patience"],
            scheduler_cfg     = config["scheduler"],
            device            = device,
        )
        elapsed = time.time() - t0

        result = {
            "trial":       trial_idx,
            "trial_id":    trial_id,
            "arch":        arch,
            "head":        head,
            "dropout":     dropout,
            "lr":          lr,
            "batch_size":  bs,
            "n_params":    n_params,
            "best_val_acc": round(val_acc, 4),
            "elapsed_s":   round(elapsed, 1),
            "history":     history,
        }
        all_results.append(result)

        print(f"\n  ✅  Trial {trial_idx} concluído │ "
              f"best_val_acc={val_acc:.2f}% │ tempo={elapsed:.0f}s")

        # Salva o melhor modelo até agora
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model   = copy.deepcopy(model.state_dict())
            best_cfg     = result.copy(); best_cfg.pop("history")

            ckpt_path = os.path.join(config["results_dir"], "best_model.pth")
            torch.save({
                "model_state_dict": best_model,
                "config":           best_cfg,
                "num_classes":      num_classes,
            }, ckpt_path)
            print(f"  💾  Novo melhor modelo salvo → {ckpt_path}")
            print(f"  🏆  Best val_acc até agora: {best_val_acc:.2f}%")

        # Persiste resultados parciais a cada trial
        with open(results_path, "w") as f:
            json.dump(all_results, f, indent=2)

        # Libera VRAM
        model.cpu()
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Resumo final ──────────────────────────────────────────────────────────
    all_results.sort(key=lambda r: r["best_val_acc"], reverse=True)

    print(f"\n{'═'*70}")
    print(f"  🏆  BUSCA CONCLUÍDA — Top-5 modelos")
    print(f"{'═'*70}")
    for rank, r in enumerate(all_results[:5], 1):
        arch_tag = f"{len(r['arch'])}conv_" + "-".join(
            str(b["out_channels"]) for b in r["arch"]
        )
        print(f"  {rank}. val_acc={r['best_val_acc']:.2f}% │ "
              f"arch={arch_tag} │ head={r['head']} │ "
              f"drop={r['dropout']} │ lr={r['lr']:.0e} │ bs={r['batch_size']}")

    print(f"\n  Resultados completos : {results_path}")
    print(f"  Melhor modelo (pth)  : "
          f"{os.path.join(config['results_dir'], 'best_model.pth')}")

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# RECONSTRUÇÃO DO MELHOR MODELO A PARTIR DO CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════

def load_best_model(ckpt_path=None, results_dir=None):
    """
    Carrega o melhor modelo salvo durante a busca.

    Args:
        ckpt_path   : caminho direto para o .pth; se None, usa results_dir
        results_dir : pasta onde 'best_model.pth' foi salvo

    Returns:
        model (FlexCNN) pronto para inferência / fine-tuning
    """
    if ckpt_path is None:
        results_dir = results_dir or SEARCH_CONFIG["results_dir"]
        ckpt_path   = os.path.join(results_dir, "best_model.pth")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg  = ckpt["config"]

    model = FlexCNN(
        num_classes  = ckpt["num_classes"],
        arch         = cfg["arch"],
        classifier   = cfg["head"],
        dropout      = cfg["dropout"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    print(f"✅ Melhor modelo carregado de: {ckpt_path}")
    print(f"   val_acc reportada : {cfg['best_val_acc']:.2f}%")
    print(f"   arch              : {len(cfg['arch'])} blocos conv")
    print(f"   head              : {cfg['head']}")
    print(f"   dropout           : {cfg['dropout']}  lr_trial: {cfg['lr']:.0e}")

    return model


# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_test(model, test_loader):
    """Avalia o modelo no conjunto de teste e retorna acurácia."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred    = model(xb).argmax(1)
            correct += (pred == yb).sum().item()
            total   += yb.size(0)

    acc = 100. * correct / total
    print(f"\n🎯 Test Accuracy : {acc:.2f}%  ({correct}/{total})")
    return acc


# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  EXECUÇÃO PRINCIPAL  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # num_classes deve estar definido de células anteriores.
    # Ajuste manualmente se executar de forma isolada:
    # num_classes = 6  # GROUP
    # num_classes = 3  # ASK

    all_results = hyperparameter_search(num_classes=num_classes)

    # Carrega e avalia o melhor modelo no teste
    best = load_best_model()
    evaluate_test(best, test_loader)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO COMBINADA — CM + Acurácia/SNR + Acurácia Combinada (2 estágios)
#
# Fluxo:
#   1. Modelo geral → prediz o GRUPO (ASK, PSK, QAM, ...)
#   2. Modelo do grupo predito → prediz a MODULAÇÃO ESPECÍFICA
#
# Requer que os seguintes arquivos estejam em MODEL_BASE_DIR:
#   • cnn_AM_APSK_ASK_FM_PSK_QAM_*.pth   ← modelo geral
#   • cnn_ASK_*.pth, cnn_PSK_*.pth, ...  ← um por grupo
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import h5py
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

# ── Parâmetros globais ────────────────────────────────────────────────────────

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_SNRS     = list(range(-20, 32, 2))
BATCH_SIZE   = 256
SEED         = 42

MODEL_BASE_DIR = "/content/drive/MyDrive/modelos_radioml"  # ← ajuste se necessário

GROUP_MAP = {
    "OOK":       "ASK", "4ASK":      "ASK", "8ASK":      "ASK",
    "BPSK":      "PSK", "QPSK":      "PSK", "8PSK":      "PSK",
    "16PSK":     "PSK", "32PSK":     "PSK", "GMSK":      "PSK", "OQPSK":     "PSK",
    "16APSK":   "APSK", "32APSK":   "APSK", "64APSK":   "APSK", "128APSK":  "APSK",
    "16QAM":     "QAM", "32QAM":     "QAM", "64QAM":     "QAM",
    "128QAM":    "QAM", "256QAM":    "QAM",
    "AM-SSB-WC": "AM",  "AM-SSB-SC": "AM",
    "AM-DSB-WC": "AM",  "AM-DSB-SC": "AM",
    "FM":        "FM",
}
GROUP_NAMES   = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
GROUP_REVERSE = {v: k for k, v in GROUP_NAMES.items()}

modulation_classes = json.load(open(modulation_classes_path, 'r'))

# Classes por grupo — ordem determinística (índice global crescente)
CLASSES_PER_GROUP = {}
for gname in GROUP_NAMES.values():
    CLASSES_PER_GROUP[gname] = [
        modulation_classes[i]
        for i in sorted([
            idx for idx, mod in enumerate(modulation_classes)
            if GROUP_MAP[mod] == gname
        ])
    ]

# Mapa modulação → índice local dentro do grupo
MOD_TO_LOCAL = {}
for gname, mods in CLASSES_PER_GROUP.items():
    for local_idx, mod in enumerate(mods):
        MOD_TO_LOCAL[(gname, mod)] = local_idx

# HDF5 paths
def _workdir():
    for d in ['/kaggle/working', '/content', '/tmp', '/root']:
        if os.path.exists(d) and os.access(d, os.W_OK):
            return d

snr_tag_group = '_'.join(str(s) for s in [0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30])
snr_tag_geral = '_'.join(str(s) for s in ALL_SNRS)

H5_GROUP = {
    g: os.path.join(_workdir(), f'{g}_subset_snr_{snr_tag_group}.hdf5')
    for g in GROUP_NAMES.values()
}
H5_GERAL = os.path.join(_workdir(), f'subset_snr_{snr_tag_geral}.hdf5')

# ══════════════════════════════════════════════════════════════════════════════
# UTILITÁRIOS
# ══════════════════════════════════════════════════════════════════════════════

def find_checkpoint(directory, contains):
    if not os.path.isdir(directory):
        return None
    cands = [f for f in os.listdir(directory)
             if f.endswith('.pth') and contains.lower() in f.lower()]
    if not cands:
        return None
    def _acc(f):
        m = re.search(r'acc([\d.]+)', f)
        return float(m.group(1)) if m else 0.0
    cands.sort(key=_acc, reverse=True)
    return os.path.join(directory, cands[0])


def load_model(path, num_classes):
    m = CNN(num_classes=num_classes)
    ck = torch.load(path, map_location=DEVICE)
    sd = ck.get('model_state_dict', ck)
    m.load_state_dict(sd)
    m.to(DEVICE).eval()
    return m


@torch.no_grad()
def predict_batch(model, x_np):
    """x_np: (N, 1024, 2) numpy → retorna (N,) predições."""
    x = torch.from_numpy(x_np).float().permute(0, 2, 1).to(DEVICE)
    out = model(x)
    return out.argmax(dim=1).cpu().numpy()


def plot_cm(cm, class_names, title, ax, cmap='Blues'):
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im   = ax.imshow(cm_n, vmin=0, vmax=1, cmap=cmap, interpolation='nearest')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predito',     fontsize=7)
    ax.set_ylabel('Verdadeiro',  fontsize=7)
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks); ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(ticks); ax.set_yticklabels(class_names, fontsize=6)
    for i in range(cm_n.shape[0]):
        for j in range(cm_n.shape[1]):
            v = cm_n[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=4,
                    color='white' if v > .5 else 'black')
    return im


# ══════════════════════════════════════════════════════════════════════════════
# 1. CARREGAR MODELOS
# ══════════════════════════════════════════════════════════════════════════════

print("Carregando modelos...")

# Modelo geral
ckpt_geral = find_checkpoint(MODEL_BASE_DIR, 'AM_APSK_ASK_FM_PSK_QAM') \
          or find_checkpoint('/content',      'AM_APSK_ASK_FM_PSK_QAM')
assert ckpt_geral, "❌ Checkpoint do modelo geral não encontrado!"
model_geral = load_model(ckpt_geral, num_classes=6)
print(f"  ✓ Geral       : {os.path.basename(ckpt_geral)}")

# Modelos de grupo
models_grupo = {}
for gname in GROUP_NAMES.values():
    nc    = len(CLASSES_PER_GROUP[gname])
    ckpt  = find_checkpoint(MODEL_BASE_DIR, f'cnn_{gname}') \
         or find_checkpoint('/content',      f'cnn_{gname}')
    if ckpt is None:
        print(f"  ⚠️  {gname}: checkpoint não encontrado — grupo será ignorado")
        continue
    models_grupo[gname] = load_model(ckpt, num_classes=nc)
    print(f"  ✓ {gname:<6}       : {os.path.basename(ckpt)}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. AVALIAÇÃO INDIVIDUAL DE CADA MODELO DE GRUPO
# ══════════════════════════════════════════════════════════════════════════════

print("\nAvaliando modelos de grupo...")

results_grupo = {}   # gname → {cm, snr_accs, test_acc, class_names}

for gname, model in models_grupo.items():
    h5 = H5_GROUP[gname]
    if not os.path.exists(h5):
        print(f"  ⚠️  HDF5 não encontrado: {h5}")
        continue

    with h5py.File(h5, 'r') as f:
        X_all = f['X'][:]
        Y_all = np.argmax(f['Y'][:], axis=1)
        Z_all = f['Z'][:, 0]

    class_names = CLASSES_PER_GROUP[gname]

    # Conjunto de teste (mesma seed do treino)
    indices = np.arange(len(Y_all))
    _, _, test_idx = split(indices, Y_all)

    X_test = X_all[test_idx]
    y_test = Y_all[test_idx]

    # Predição em batches
    y_pred = []
    for s in range(0, len(X_test), BATCH_SIZE):
        y_pred.append(predict_batch(model, X_test[s:s+BATCH_SIZE]))
    y_pred = np.concatenate(y_pred)

    test_acc = 100.0 * (y_pred == y_test).mean()
    cm       = confusion_matrix(y_test, y_pred)

    # Acurácia por SNR
    snr_accs = {}
    for snr in ALL_SNRS:
        mask = Z_all == snr
        if mask.sum() == 0:
            snr_accs[snr] = None; continue
        xb = X_all[mask]
        yb = Y_all[mask]
        yp = []
        for s in range(0, len(xb), BATCH_SIZE):
            yp.append(predict_batch(model, xb[s:s+BATCH_SIZE]))
        yp = np.concatenate(yp)
        snr_accs[snr] = 100.0 * (yp == yb).mean()

    results_grupo[gname] = dict(cm=cm, snr_accs=snr_accs,
                                 test_acc=test_acc, class_names=class_names)
    print(f"  ✓ {gname:<6}  Acc={test_acc:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# 3. AVALIAÇÃO DO MODELO GERAL
# ══════════════════════════════════════════════════════════════════════════════

print("\nAvaliando modelo geral...")

result_geral = None
if os.path.exists(H5_GERAL):
    with h5py.File(H5_GERAL, 'r') as f:
        X_g = f['X'][:]
        Y_g = np.argmax(f['Y_grouped'][:], axis=1)   # labels 0-5 (grupos)
        Z_g = f['Z'][:, 0]

    indices_g = np.arange(len(Y_g))
    _, _, test_idx_g = split(indices_g, Y_g)

    X_test_g = X_g[test_idx_g]
    y_test_g = Y_g[test_idx_g]

    y_pred_g = []
    for s in range(0, len(X_test_g), BATCH_SIZE):
        y_pred_g.append(predict_batch(model_geral, X_test_g[s:s+BATCH_SIZE]))
    y_pred_g = np.concatenate(y_pred_g)

    test_acc_g = 100.0 * (y_pred_g == y_test_g).mean()
    cm_g       = confusion_matrix(y_test_g, y_pred_g)

    snr_accs_g = {}
    for snr in ALL_SNRS:
        mask = Z_g == snr
        if mask.sum() == 0:
            snr_accs_g[snr] = None; continue
        xb = X_g[mask]; yb = Y_g[mask]
        yp = []
        for s in range(0, len(xb), BATCH_SIZE):
            yp.append(predict_batch(model_geral, xb[s:s+BATCH_SIZE]))
        snr_accs_g[snr] = 100.0 * (np.concatenate(yp) == yb).mean()

    result_geral = dict(cm=cm_g, snr_accs=snr_accs_g,
                        test_acc=test_acc_g,
                        class_names=list(GROUP_NAMES.values()))
    print(f"  ✓ Geral  Acc={test_acc_g:.2f}%")
else:
    print(f"  ⚠️  HDF5 geral não encontrado: {H5_GERAL}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. AVALIAÇÃO COMBINADA (2 estágios)
#    Estágio 1: modelo geral → prediz grupo
#    Estágio 2: modelo do grupo predito → prediz modulação
#    Resultado: acurácia sobre as 24 modulações reais
# ══════════════════════════════════════════════════════════════════════════════

print("\nCalculando predição combinada (2 estágios)...")

# Usa o HDF5 original com todas as 24 classes e todos os SNRs
h5_original = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'

# Índice global → nome da modulação
# Índice global → grupo
# Índice global → índice local no grupo
global_to_mod   = {i: modulation_classes[i] for i in range(len(modulation_classes))}
global_to_group = {i: GROUP_MAP[modulation_classes[i]] for i in range(len(modulation_classes))}
global_to_local = {
    i: MOD_TO_LOCAL[(GROUP_MAP[modulation_classes[i]], modulation_classes[i])]
    for i in range(len(modulation_classes))
}

# Para a CM combinada: eixo = 24 modulações na ordem do dataset original
all_mod_names = modulation_classes   # lista de 24 strings

y_true_comb = []
y_pred_comb = []
snr_true_comb = []

snr_accs_comb = {}

with h5py.File(h5_original, 'r') as f:
    N        = f['X'].shape[0]
    Z_orig   = f['Z'][:, 0]
    Y_orig   = np.argmax(f['Y'][:], axis=1)   # 0-23

    # Avalia SNR a SNR para calcular acurácia por SNR
    for snr in ALL_SNRS:
        mask = Z_orig == snr
        if mask.sum() == 0:
            snr_accs_comb[snr] = None
            continue

        idx   = np.where(mask)[0]
        y_true_snr = Y_orig[idx]   # índices globais 0-23

        y_pred_snr = np.full(len(idx), -1, dtype=np.int64)

        # Estágio 1: modelo geral
        for s in range(0, len(idx), BATCH_SIZE):
            e    = min(s + BATCH_SIZE, len(idx))
            xb   = f['X'][idx[s:e]]
            grp_pred = predict_batch(model_geral, xb)   # 0-5

            # Estágio 2: modelo do grupo predito
            # Agrupa por grupo predito para chamar o modelo correto
            for grp_id in np.unique(grp_pred):
                gname = GROUP_NAMES[grp_id]
                if gname not in models_grupo:
                    continue   # grupo sem modelo → deixa -1

                sub_mask  = grp_pred == grp_id
                x_sub     = xb[sub_mask]
                local_pred = predict_batch(models_grupo[gname], x_sub)

                # Converte local → global
                local_to_global = {
                    local: glob
                    for glob, (gn, local) in [
                        (gi, (GROUP_MAP[modulation_classes[gi]],
                              MOD_TO_LOCAL[(GROUP_MAP[modulation_classes[gi]],
                                            modulation_classes[gi])]))
                        for gi in range(len(modulation_classes))
                        if GROUP_MAP[modulation_classes[gi]] == gname
                    ]
                }
                global_pred_sub = np.array([
                    local_to_global.get(lp, -1) for lp in local_pred
                ])

                batch_positions = np.where(sub_mask)[0] + s
                y_pred_snr[batch_positions] = global_pred_sub

        valid = y_pred_snr != -1
        if valid.sum() > 0:
            snr_accs_comb[snr] = 100.0 * (y_pred_snr[valid] == y_true_snr[valid]).mean()
        else:
            snr_accs_comb[snr] = None

        y_true_comb.append(y_true_snr)
        y_pred_comb.append(y_pred_snr)
        snr_true_comb.append(np.full(len(idx), snr))

y_true_comb = np.concatenate(y_true_comb)
y_pred_comb = np.concatenate(y_pred_comb)

# Remove amostras não classificadas (grupo sem modelo)
valid_mask  = y_pred_comb != -1
y_true_comb = y_true_comb[valid_mask]
y_pred_comb = y_pred_comb[valid_mask]

test_acc_comb = 100.0 * (y_true_comb == y_pred_comb).mean()
cm_comb       = confusion_matrix(y_true_comb, y_pred_comb,
                                  labels=list(range(len(all_mod_names))))

print(f"  ✓ Combinado  Acc={test_acc_comb:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# 5. FIGURA 1 — MATRIZES DE CONFUSÃO
# ══════════════════════════════════════════════════════════════════════════════

n_grupo  = len(results_grupo)
n_total  = n_grupo + (1 if result_geral else 0) + 1   # grupos + geral + combinado

ncols = min(3, n_total)
nrows = int(np.ceil(n_total / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 7 * nrows))
axes_flat = np.array(axes).flatten() if n_total > 1 else [axes]

ax_idx = 0

# ── Grupos ────────────────────────────────────────────────────────────────────
for gname, res in results_grupo.items():
    im = plot_cm(res['cm'], res['class_names'],
                 f'{gname} — Acc={res["test_acc"]:.2f}%',
                 axes_flat[ax_idx])
    fig.colorbar(im, ax=axes_flat[ax_idx], fraction=0.046, pad=0.04)
    ax_idx += 1

# ── Geral ─────────────────────────────────────────────────────────────────────
if result_geral:
    im = plot_cm(result_geral['cm'], result_geral['class_names'],
                 f'Geral (6 grupos) — Acc={result_geral["test_acc"]:.2f}%',
                 axes_flat[ax_idx])
    fig.colorbar(im, ax=axes_flat[ax_idx], fraction=0.046, pad=0.04)
    ax_idx += 1

# ── Combinado ─────────────────────────────────────────────────────────────────
im = plot_cm(cm_comb, all_mod_names,
             f'Combinado (2 estágios) — Acc={test_acc_comb:.2f}%',
             axes_flat[ax_idx], cmap='Purples')
fig.colorbar(im, ax=axes_flat[ax_idx], fraction=0.046, pad=0.04)
ax_idx += 1

# Desativa eixos extras
for i in range(ax_idx, len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle('Matrizes de Confusão — Todos os Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: /content/confusion_matrices_all.png")

# ══════════════════════════════════════════════════════════════════════════════
# 6. FIGURA 2 — ACURÁCIA × SNR (todos + combinado)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Painel esquerdo: modelos de grupo + combinado ────────────────────────────
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, n_grupo + 1))

for ci, (gname, res) in enumerate(results_grupo.items()):
    snrs_ok = sorted([s for s, a in res['snr_accs'].items() if a is not None])
    vals    = [res['snr_accs'][s] for s in snrs_ok]
    ax.plot(snrs_ok, vals, marker='o', markersize=3,
            linewidth=1.5, label=gname, color=colors[ci])

# Combinado
snrs_ok_c = sorted([s for s, a in snr_accs_comb.items() if a is not None])
vals_c    = [snr_accs_comb[s] for s in snrs_ok_c]
ax.plot(snrs_ok_c, vals_c, marker='D', markersize=5,
        linewidth=2.5, linestyle='--', label='Combinado (2 estágios)',
        color='black')

ax.set_title('Modelos de Grupo + Combinado', fontsize=11, fontweight='bold')
ax.set_xlabel('SNR (dB)', fontsize=9)
ax.set_ylabel('Acurácia (%)', fontsize=9)
ax.set_ylim(0, 105)
ax.set_xticks(ALL_SNRS)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

# ── Painel direito: modelo geral + combinado ─────────────────────────────────
ax = axes[1]

if result_geral:
    snrs_ok_g = sorted([s for s, a in result_geral['snr_accs'].items() if a is not None])
    vals_g    = [result_geral['snr_accs'][s] for s in snrs_ok_g]
    ax.plot(snrs_ok_g, vals_g, marker='s', markersize=4,
            linewidth=2, label='Geral (6 grupos)', color='royalblue')

ax.plot(snrs_ok_c, vals_c, marker='D', markersize=5,
        linewidth=2.5, linestyle='--', label='Combinado (2 estágios)',
        color='black')

ax.set_title('Modelo Geral vs. Combinado', fontsize=11, fontweight='bold')
ax.set_xlabel('SNR (dB)', fontsize=9)
ax.set_ylabel('Acurácia (%)', fontsize=9)
ax.set_ylim(0, 105)
ax.set_xticks(ALL_SNRS)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle('Acurácia × SNR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/accuracy_vs_snr_all.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: /content/accuracy_vs_snr_all.png")

# ══════════════════════════════════════════════════════════════════════════════
# 7. FIGURA 3 — ACURÁCIA × SNR INDIVIDUAL (subplot por modelo)
# ══════════════════════════════════════════════════════════════════════════════

all_curves = {}
for gname, res in results_grupo.items():
    all_curves[f'Grupo {gname}'] = (res['snr_accs'], res['test_acc'], 'steelblue')
if result_geral:
    all_curves['Geral (6 grupos)'] = (result_geral['snr_accs'],
                                       result_geral['test_acc'], 'darkorange')
all_curves['Combinado (2 estágios)'] = (snr_accs_comb, test_acc_comb, 'purple')

nc2   = min(3, len(all_curves))
nr2   = int(np.ceil(len(all_curves) / nc2))
fig2, axs2 = plt.subplots(nr2, nc2, figsize=(7 * nc2, 4 * nr2))
axs2_flat  = np.array(axs2).flatten() if len(all_curves) > 1 else [axs2]

for ai, (label, (accs, tot_acc, color)) in enumerate(all_curves.items()):
    ax   = axs2_flat[ai]
    snrs = sorted([s for s, a in accs.items() if a is not None])
    vals = [accs[s] for s in snrs]
    ax.plot(snrs, vals, marker='o', markersize=4, linewidth=1.8, color=color)
    ax.fill_between(snrs, vals, alpha=0.12, color=color)
    ax.axhline(tot_acc, color='red', linestyle='--', linewidth=1.2,
               label=f'Acc total: {tot_acc:.1f}%')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('SNR (dB)', fontsize=8)
    ax.set_ylabel('Acurácia (%)', fontsize=8)
    ax.set_ylim(0, 105)
    ax.set_xticks(ALL_SNRS[::2])
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for i in range(len(all_curves), len(axs2_flat)):
    axs2_flat[i].set_visible(False)

fig2.suptitle('Acurácia × SNR — Individual por Modelo',
              fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/accuracy_vs_snr_individual.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: /content/accuracy_vs_snr_individual.png")

# ══════════════════════════════════════════════════════════════════════════════
# 8. TABELA RESUMO
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print(f"{'Modelo':<35} {'Classes':>8} {'Acc Teste':>12}")
print(f"{'─'*65}")
for gname, res in results_grupo.items():
    print(f"  Grupo {gname:<29} {len(res['class_names']):>8} {res['test_acc']:>11.2f}%")
if result_geral:
    print(f"  Geral (6 grupos){'':>19} {'6':>8} {result_geral['test_acc']:>11.2f}%")
print(f"  {'─'*62}")
print(f"  Combinado (2 estágios){'':>13} {'24':>8} {test_acc_comb:>11.2f}%")
print(f"{'='*65}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CAMINHOS DOS MODELOS — edite esta seção antes de executar
# ══════════════════════════════════════════════════════════════════════════════

CKPT_GERAL = "/content/cnn_GROUP_ep001_acc88.61_lr1.00e-03.pth"

CKPT_GRUPOS = {
    "ASK"  : "/content/cnn_ASK_ep001_acc97.99_lr1.00e-06.pth",   # 3 classes : OOK, 4ASK, 8ASK
    "PSK"  : "/content/cnn_PSK_ep007_acc95.93_lr5.00e-06.pth",   # 7 classes : BPSK, QPSK, 8PSK, 16PSK, 32PSK, GMSK, OQPSK
    "APSK" : "/content/cnn_APSK_ep007_acc92.47_lr1.00e-06.pth",  # 4 classes : 16APSK, 32APSK, 64APSK, 128APSK
    "QAM"  : "/content/cnn_QAM_ep014_acc84.15_lr5.00e-06.pth",   # 5 classes : 16QAM, 32QAM, 64QAM, 128QAM, 256QAM
    "AM"   : "/content/cnn_AM_ep006_acc80.08_lr5.00e-06.pth",    # 4 classes : AM-SSB-WC, AM-SSB-SC, AM-DSB-WC, AM-DSB-SC
    "FM"   : None,   # 1 classe apenas (FM) — sem modelo de grupo necessário
}

# ══════════════════════════════════════════════════════════════════════════════
# CARREGAMENTO AUTOMÁTICO — não edite abaixo desta linha
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modulation_classes = json.load(open(modulation_classes_path, 'r'))

GROUP_MAP = {
    "OOK":       "ASK", "4ASK":      "ASK", "8ASK":      "ASK",
    "BPSK":      "PSK", "QPSK":      "PSK", "8PSK":      "PSK",
    "16PSK":     "PSK", "32PSK":     "PSK", "GMSK":      "PSK", "OQPSK":     "PSK",
    "16APSK":   "APSK", "32APSK":   "APSK", "64APSK":   "APSK", "128APSK":  "APSK",
    "16QAM":     "QAM", "32QAM":     "QAM", "64QAM":     "QAM",
    "128QAM":    "QAM", "256QAM":    "QAM",
    "AM-SSB-WC": "AM",  "AM-SSB-SC": "AM",
    "AM-DSB-WC": "AM",  "AM-DSB-SC": "AM",
    "FM":        "FM",
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}

# FM tem apenas 1 classe — não precisa de modelo de grupo
# Quando o modelo geral prediz FM, a resposta já é determinística
SINGLE_CLASS_GROUPS = {"FM"}

CLASSES_PER_GROUP = {}
for gname in GROUP_NAMES.values():
    CLASSES_PER_GROUP[gname] = [
        modulation_classes[i]
        for i in sorted([
            idx for idx, mod in enumerate(modulation_classes)
            if GROUP_MAP[mod] == gname
        ])
    ]

MOD_TO_LOCAL = {}
for gname, mods in CLASSES_PER_GROUP.items():
    for local_idx, mod in enumerate(mods):
        MOD_TO_LOCAL[(gname, mod)] = local_idx

# ── Função de carregamento ────────────────────────────────────────────────────

def load_model_from_path(path, num_classes):
    m  = CNN(num_classes=num_classes)
    ck = torch.load(path, map_location=DEVICE)
    sd = ck.get('model_state_dict', ck)
    m.load_state_dict(sd)
    m.to(DEVICE).eval()
    return m

# ── Validação dos caminhos ────────────────────────────────────────────────────

print("Verificando caminhos configurados...\n")

errors = []

if CKPT_GERAL is None:
    errors.append("CKPT_GERAL não definido")
elif not os.path.exists(CKPT_GERAL):
    errors.append(f"CKPT_GERAL não encontrado: {CKPT_GERAL}")

for gname, ckpt in CKPT_GRUPOS.items():
    if gname in SINGLE_CLASS_GROUPS:
        continue   # FM não precisa de modelo — pula validação
    if ckpt is not None and not os.path.exists(ckpt):
        errors.append(f"CKPT_GRUPOS['{gname}'] não encontrado: {ckpt}")

if errors:
    print("❌ Erros encontrados:")
    for e in errors:
        print(f"   • {e}")
    raise FileNotFoundError(
        "Corrija os caminhos na seção de configuração e execute novamente."
    )

print("✅ Todos os caminhos validados.\n")

# ── Carregamento do modelo geral ──────────────────────────────────────────────

print("Carregando modelo geral...")
model_geral = load_model_from_path(CKPT_GERAL, num_classes=6)
print(f"  ✅ Geral (6 grupos) : {os.path.basename(CKPT_GERAL)}\n")

# ── Carregamento dos modelos de grupo ─────────────────────────────────────────

models_grupo = {}

print("Carregando modelos de grupo...")
for gname in GROUP_NAMES.values():
    nc   = len(CLASSES_PER_GROUP[gname])
    ckpt = CKPT_GRUPOS.get(gname)

    # Grupos com 1 classe não precisam de modelo
    if gname in SINGLE_CLASS_GROUPS:
        print(f"  ➖ {gname:<6} ({nc} classe ) : classificação direta — sem modelo necessário")
        continue

    if ckpt is None:
        print(f"  ⚠️  {gname:<6} ({nc} classes) : não configurado — grupo será ignorado")
        continue

    models_grupo[gname] = load_model_from_path(ckpt, num_classes=nc)
    print(f"  ✅ {gname:<6} ({nc} classes) : {os.path.basename(ckpt)}")

# ── Resumo ────────────────────────────────────────────────────────────────────

print(f"\n{'='*65}")
print(f"  {'Modelo':<35} {'Classes':>8}  Status")
print(f"{'─'*65}")
print(f"  {'Geral (6 grupos)':<35} {'6':>8}  ✅  carregado")
for gname in GROUP_NAMES.values():
    nc = len(CLASSES_PER_GROUP[gname])
    if gname in SINGLE_CLASS_GROUPS:
        status = "➖  classe única — determinístico"
    elif gname in models_grupo:
        status = "✅  carregado"
    else:
        status = "⚠️   não carregado"
    print(f"  {f'Grupo {gname}':<35} {nc:>8}  {status}")
print(f"{'='*65}")
print(f"\n▶ Prossiga para a célula de avaliação.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO COMPLETA
# ══════════════════════════════════════════════════════════════════════════════

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_SNRS    = list(range(-20, 32, 2))
BATCH_SIZE  = 256

# ── Paths HDF5 ────────────────────────────────────────────────────────────────

def _workdir():
    for d in ['/kaggle/working', '/content', '/tmp', '/root']:
        if os.path.exists(d) and os.access(d, os.W_OK):
            return d

snr_tag_group = '_'.join(str(s) for s in [0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30])
snr_tag_geral = '_'.join(str(s) for s in ALL_SNRS)

H5_GROUP = {
    g: os.path.join(_workdir(), f'{g}_subset_snr_{snr_tag_group}.hdf5')
    for g in GROUP_NAMES.values()
}
H5_GERAL = os.path.join(_workdir(), f'subset_snr_{snr_tag_geral}.hdf5')

# ══════════════════════════════════════════════════════════════════════════════
# UTILITÁRIOS
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def predict_batch(model, x_np):
    """x_np: (N,1024,2) numpy → (N,) predições."""
    x = torch.from_numpy(x_np).float().permute(0, 2, 1).to(DEVICE)
    return model(x).argmax(dim=1).cpu().numpy()


def predict_all(model, X, batch_size=BATCH_SIZE):
    """Prediz em batches sobre um array numpy completo."""
    preds = []
    for s in range(0, len(X), batch_size):
        preds.append(predict_batch(model, X[s:s+batch_size]))
    return np.concatenate(preds)


def plot_cm(cm, class_names, title, ax, cmap='Blues'):
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im   = ax.imshow(cm_n, vmin=0, vmax=1, cmap=cmap, interpolation='nearest')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predito',    fontsize=7)
    ax.set_ylabel('Verdadeiro', fontsize=7)
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(ticks)
    ax.set_yticklabels(class_names, fontsize=6)
    for i in range(cm_n.shape[0]):
        for j in range(cm_n.shape[1]):
            v = cm_n[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=4, color='white' if v > .5 else 'black')
    return im


def plot_snr_curve(snr_accs, label, ax, color, marker='o',
                   linestyle='-', lw=1.8, ms=4):
    snrs = sorted([s for s, a in snr_accs.items() if a is not None])
    vals = [snr_accs[s] for s in snrs]
    ax.plot(snrs, vals, marker=marker, markersize=ms,
            linewidth=lw, linestyle=linestyle,
            label=label, color=color)
    return snrs, vals


# ══════════════════════════════════════════════════════════════════════════════
# 1. AVALIAÇÃO DOS MODELOS DE GRUPO
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("1. Avaliando modelos de grupo")
print("=" * 60)

results_grupo = {}

for gname, model in models_grupo.items():
    h5 = H5_GROUP[gname]
    if not os.path.exists(h5):
        print(f"  ⚠️  HDF5 não encontrado para {gname}: {h5}")
        continue

    with h5py.File(h5, 'r') as f:
        X_all = f['X'][:]
        Y_all = np.argmax(f['Y'][:], axis=1)
        Z_all = f['Z'][:, 0]

    class_names = CLASSES_PER_GROUP[gname]

    # Conjunto de teste
    _, _, test_idx = split(np.arange(len(Y_all)), Y_all)
    X_test = X_all[test_idx]
    y_test = Y_all[test_idx]

    y_pred   = predict_all(model, X_test)
    test_acc = 100.0 * (y_pred == y_test).mean()
    cm       = confusion_matrix(y_test, y_pred)

    # Acurácia por SNR
    snr_accs = {}
    for snr in ALL_SNRS:
        mask = Z_all == snr
        if mask.sum() == 0:
            snr_accs[snr] = None
            continue
        yp = predict_all(model, X_all[mask])
        snr_accs[snr] = 100.0 * (yp == Y_all[mask]).mean()

    results_grupo[gname] = dict(
        cm=cm, snr_accs=snr_accs,
        test_acc=test_acc, class_names=class_names
    )
    print(f"  ✅ {gname:<6}  Acc = {test_acc:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# 2. AVALIAÇÃO DO MODELO GERAL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("2. Avaliando modelo geral")
print("=" * 60)

result_geral = None

if os.path.exists(H5_GERAL):
    with h5py.File(H5_GERAL, 'r') as f:
        X_g = f['X'][:]
        Y_g = np.argmax(f['Y_grouped'][:], axis=1)
        Z_g = f['Z'][:, 0]

    _, _, test_idx_g = split(np.arange(len(Y_g)), Y_g)
    X_test_g = X_g[test_idx_g]
    y_test_g = Y_g[test_idx_g]

    y_pred_g   = predict_all(model_geral, X_test_g)
    test_acc_g = 100.0 * (y_pred_g == y_test_g).mean()
    cm_g       = confusion_matrix(y_test_g, y_pred_g)

    snr_accs_g = {}
    for snr in ALL_SNRS:
        mask = Z_g == snr
        if mask.sum() == 0:
            snr_accs_g[snr] = None
            continue
        yp = predict_all(model_geral, X_g[mask])
        snr_accs_g[snr] = 100.0 * (yp == Y_g[mask]).mean()

    result_geral = dict(
        cm=cm_g, snr_accs=snr_accs_g,
        test_acc=test_acc_g,
        class_names=list(GROUP_NAMES.values())
    )
    print(f"  ✅ Geral (6 grupos)  Acc = {test_acc_g:.2f}%")
else:
    print(f"  ⚠️  HDF5 geral não encontrado: {H5_GERAL}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. PREDIÇÃO COMBINADA (2 estágios)
#    Estágio 1 → modelo geral prediz grupo
#    Estágio 2 → modelo do grupo prediz modulação
#    FM (1 classe) → resposta determinística, sem modelo de grupo
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("3. Predição combinada (2 estágios)")
print("=" * 60)

# Mapas de conversão local → global por grupo
local_to_global = {}
for gname, mods in CLASSES_PER_GROUP.items():
    local_to_global[gname] = {
        local: modulation_classes.index(mod)
        for local, mod in enumerate(mods)
    }

all_mod_names = modulation_classes   # 24 classes na ordem global

y_true_comb   = []
y_pred_comb   = []
snr_accs_comb = {}

h5_original = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'

with h5py.File(h5_original, 'r') as f:
    Z_orig = f['Z'][:, 0]
    Y_orig = np.argmax(f['Y'][:], axis=1)   # 0-23

    for snr in ALL_SNRS:
        mask = Z_orig == snr
        if mask.sum() == 0:
            snr_accs_comb[snr] = None
            continue

        idx        = np.where(mask)[0]
        y_true_snr = Y_orig[idx]
        y_pred_snr = np.full(len(idx), -1, dtype=np.int64)

        # Processa em batches
        for s in range(0, len(idx), BATCH_SIZE):
            e   = min(s + BATCH_SIZE, len(idx))
            xb  = f['X'][idx[s:e]]                          # (B,1024,2)

            # ── Estágio 1: prediz grupo ───────────────────────────────────────
            grp_pred = predict_batch(model_geral, xb)        # 0-5

            # ── Estágio 2: prediz modulação dentro do grupo ───────────────────
            for grp_id in np.unique(grp_pred):
                gname    = GROUP_NAMES[grp_id]
                sub_mask = grp_pred == grp_id
                x_sub    = xb[sub_mask]

                if gname in SINGLE_CLASS_GROUPS:
                    # FM: 1 classe → índice global direto, sem modelo
                    global_idx = local_to_global[gname][0]
                    global_pred_sub = np.full(sub_mask.sum(), global_idx,
                                             dtype=np.int64)

                elif gname in models_grupo:
                    local_pred      = predict_batch(models_grupo[gname], x_sub)
                    global_pred_sub = np.array([
                        local_to_global[gname].get(lp, -1)
                        for lp in local_pred
                    ], dtype=np.int64)

                else:
                    # Grupo sem modelo → marca como -1
                    global_pred_sub = np.full(sub_mask.sum(), -1,
                                             dtype=np.int64)

                batch_positions           = np.where(sub_mask)[0] + s
                y_pred_snr[batch_positions] = global_pred_sub

        valid = y_pred_snr != -1
        if valid.sum() > 0:
            snr_accs_comb[snr] = 100.0 * (
                y_pred_snr[valid] == y_true_snr[valid]
            ).mean()
        else:
            snr_accs_comb[snr] = None

        y_true_comb.append(y_true_snr)
        y_pred_comb.append(y_pred_snr)

        acc_str = f"{snr_accs_comb[snr]:.2f}%" if snr_accs_comb[snr] else "N/A"
        print(f"  SNR {snr:>4} dB  →  Acc = {acc_str}")

y_true_comb = np.concatenate(y_true_comb)
y_pred_comb = np.concatenate(y_pred_comb)

valid_mask    = y_pred_comb != -1
y_true_comb   = y_true_comb[valid_mask]
y_pred_comb   = y_pred_comb[valid_mask]
test_acc_comb = 100.0 * (y_true_comb == y_pred_comb).mean()
cm_comb       = confusion_matrix(y_true_comb, y_pred_comb,
                                  labels=list(range(len(all_mod_names))))

print(f"\n  ✅ Combinado  Acc = {test_acc_comb:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# 4. FIGURA 1 — MATRIZES DE CONFUSÃO
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("4. Plotando matrizes de confusão")
print("=" * 60)

# Ordem: grupos → geral → combinado
plots_cm = []
for gname, res in results_grupo.items():
    plots_cm.append((
        res['cm'], res['class_names'],
        f'{gname} — Acc={res["test_acc"]:.2f}%',
        'Blues'
    ))
if result_geral:
    plots_cm.append((
        result_geral['cm'], result_geral['class_names'],
        f'Geral (6 grupos) — Acc={result_geral["test_acc"]:.2f}%',
        'Greens'
    ))
plots_cm.append((
    cm_comb, all_mod_names,
    f'Combinado (2 estágios) — Acc={test_acc_comb:.2f}%',
    'Purples'
))

ncols    = min(3, len(plots_cm))
nrows    = int(np.ceil(len(plots_cm) / ncols))
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(7.5 * ncols, 7 * nrows))
axes_flat = np.array(axes).flatten() if len(plots_cm) > 1 else [axes]

for ai, (cm_, cnames, title, cmap) in enumerate(plots_cm):
    im = plot_cm(cm_, cnames, title, axes_flat[ai], cmap=cmap)
    fig.colorbar(im, ax=axes_flat[ai], fraction=0.046, pad=0.04)

for i in range(len(plots_cm), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle('Matrizes de Confusão — Todos os Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print("  ✅ Salvo: /content/confusion_matrices_all.png")

# ══════════════════════════════════════════════════════════════════════════════
# 5. FIGURA 2 — ACURÁCIA × SNR (painel duplo)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("5. Plotando acurácia × SNR")
print("=" * 60)

colors_grupo = plt.cm.tab10(np.linspace(0, 0.7, len(results_grupo)))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Painel esquerdo: grupos + combinado ──────────────────────────────────────
ax = axes[0]
for ci, (gname, res) in enumerate(results_grupo.items()):
    plot_snr_curve(res['snr_accs'], gname, ax,
                   color=colors_grupo[ci])

plot_snr_curve(snr_accs_comb, 'Combinado (2 estágios)', ax,
               color='black', marker='D', linestyle='--', lw=2.5, ms=5)

ax.set_title('Modelos de Grupo + Combinado', fontsize=11, fontweight='bold')
ax.set_xlabel('SNR (dB)', fontsize=9)
ax.set_ylabel('Acurácia (%)', fontsize=9)
ax.set_ylim(0, 105)
ax.set_xticks(ALL_SNRS)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

# ── Painel direito: geral + combinado ───────────────────────────────────────
ax = axes[1]
if result_geral:
    plot_snr_curve(result_geral['snr_accs'], 'Geral (6 grupos)',
                   ax, color='royalblue', marker='s', lw=2)

plot_snr_curve(snr_accs_comb, 'Combinado (2 estágios)',
               ax, color='black', marker='D', linestyle='--', lw=2.5, ms=5)

ax.set_title('Modelo Geral vs. Combinado', fontsize=11, fontweight='bold')
ax.set_xlabel('SNR (dB)', fontsize=9)
ax.set_ylabel('Acurácia (%)', fontsize=9)
ax.set_ylim(0, 105)
ax.set_xticks(ALL_SNRS)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle('Acurácia × SNR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/accuracy_vs_snr_all.png', dpi=150, bbox_inches='tight')
plt.show()
print("  ✅ Salvo: /content/accuracy_vs_snr_all.png")

# ══════════════════════════════════════════════════════════════════════════════
# 6. FIGURA 3 — ACURÁCIA × SNR INDIVIDUAL
# ══════════════════════════════════════════════════════════════════════════════

all_curves = {}
for gname, res in results_grupo.items():
    all_curves[f'Grupo {gname}'] = (res['snr_accs'], res['test_acc'], 'steelblue')
if result_geral:
    all_curves['Geral (6 grupos)'] = (
        result_geral['snr_accs'], result_geral['test_acc'], 'darkorange'
    )
all_curves['Combinado (2 estágios)'] = (snr_accs_comb, test_acc_comb, 'purple')

nc2   = min(3, len(all_curves))
nr2   = int(np.ceil(len(all_curves) / nc2))
fig2, axs2 = plt.subplots(nr2, nc2, figsize=(7 * nc2, 4 * nr2))
axs2_flat  = np.array(axs2).flatten() if len(all_curves) > 1 else [axs2]

for ai, (label, (accs, tot_acc, color)) in enumerate(all_curves.items()):
    ax   = axs2_flat[ai]
    snrs = sorted([s for s, a in accs.items() if a is not None])
    vals = [accs[s] for s in snrs]
    ax.plot(snrs, vals, marker='o', markersize=4,
            linewidth=1.8, color=color)
    ax.fill_between(snrs, vals, alpha=0.12, color=color)
    ax.axhline(tot_acc, color='red', linestyle='--', linewidth=1.2,
               label=f'Acc total: {tot_acc:.1f}%')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('SNR (dB)', fontsize=8)
    ax.set_ylabel('Acurácia (%)', fontsize=8)
    ax.set_ylim(0, 105)
    ax.set_xticks(ALL_SNRS[::2])
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for i in range(len(all_curves), len(axs2_flat)):
    axs2_flat[i].set_visible(False)

fig2.suptitle('Acurácia × SNR — Individual por Modelo',
              fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/accuracy_vs_snr_individual.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("  ✅ Salvo: /content/accuracy_vs_snr_individual.png")

# ══════════════════════════════════════════════════════════════════════════════
# 7. TABELA RESUMO
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print(f"  {'Modelo':<35} {'Classes':>8}  {'Acc Teste':>10}")
print(f"{'─'*65}")
for gname, res in results_grupo.items():
    nc_ = len(res['class_names'])
    print(f"  {'Grupo ' + gname:<35} {nc_:>8}  {res['test_acc']:>9.2f}%")
if result_geral:
    print(f"  {'Geral (6 grupos)':<35} {'6':>8}  {result_geral['test_acc']:>9.2f}%")
print(f"  {'─'*62}")
print(f"  {'Combinado (2 estágios)':<35} {'24':>8}  {test_acc_comb:>9.2f}%")
print(f"{'='*65}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO — usa dataset original (como módulo 2, seção 3)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import confusion_matrix

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_SNRS   = list(range(-20, 32, 2))
BATCH_SIZE = 256

h5_original = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'

# ── Utilitários (inalterados) ─────────────────────────────────────────────────

@torch.no_grad()
def predict_batch(model, x_np):
    x = torch.from_numpy(x_np).float().permute(0, 2, 1).to(DEVICE)
    return model(x).argmax(dim=1).cpu().numpy()

def predict_all(model, X):
    preds = []
    for s in range(0, len(X), BATCH_SIZE):
        preds.append(predict_batch(model, X[s:s+BATCH_SIZE]))
    return np.concatenate(preds)

def plot_cm_ax(cm, class_names, title, ax, cmap='Blues'):
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im   = ax.imshow(cm_n, vmin=0, vmax=1, cmap=cmap, interpolation='nearest')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Predito',    fontsize=8)
    ax.set_ylabel('Verdadeiro', fontsize=8)
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks); ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(class_names, fontsize=7)
    for i in range(cm_n.shape[0]):
        for j in range(cm_n.shape[1]):
            v = cm_n[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if v > .5 else 'black')
    return im

def plot_snr_ax(snr_accs, test_acc, title, ax, color):
    snrs = sorted([s for s, a in snr_accs.items() if a is not None])
    vals = [snr_accs[s] for s in snrs]
    ax.plot(snrs, vals, marker='o', markersize=4, linewidth=1.8, color=color)
    ax.fill_between(snrs, vals, alpha=0.12, color=color)
    ax.axhline(test_acc, color='red', linestyle='--',
               linewidth=1.2, label=f'Acc total: {test_acc:.1f}%')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('SNR (dB)', fontsize=8)
    ax.set_ylabel('Acurácia (%)', fontsize=8)
    ax.set_ylim(0, 105)
    ax.set_xticks(ALL_SNRS[::2])
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# ══════════════════════════════════════════════════════════════════════════════
# COLETA DE RESULTADOS — lê dataset original, filtra por grupo via Y
# ══════════════════════════════════════════════════════════════════════════════

plot_entries = []
cores_grupo  = plt.cm.tab10(np.linspace(0, 0.7, len(models_grupo)))

# Mapa: nome_mod → índice global
mod_to_global = {m: i for i, m in enumerate(modulation_classes)}

print("Avaliando modelos de grupo (via dataset original)...")

with h5py.File(h5_original, 'r') as f:
    Y_orig_oh = f['Y'][:]          # (N, 24) one-hot
    Z_orig    = f['Z'][:, 0]       # SNR por amostra

    Y_orig = np.argmax(Y_orig_oh, axis=1)   # índice global 0-23

    for ci, (gname, model) in enumerate(models_grupo.items()):
        class_names  = CLASSES_PER_GROUP[gname]
        # Índices globais das classes deste grupo
        global_idxs  = [mod_to_global[m] for m in class_names]
        # Máscara: amostras que pertencem a este grupo
        group_mask   = np.isin(Y_orig, global_idxs)

        if group_mask.sum() == 0:
            print(f"  ⚠️  Nenhuma amostra para {gname}")
            continue

        # Rótulos locais (0..n_classes_grupo-1)
        global_to_local = {g: l for l, g in enumerate(global_idxs)}
        Y_local = np.array([global_to_local[y] for y in Y_orig[group_mask]])
        Z_group = Z_orig[group_mask]

        # Split → pega apenas índices de teste
        _, _, test_idx = split(np.arange(group_mask.sum()), Y_local)

        # Carrega X apenas para este grupo
        X_group  = f['X'][np.where(group_mask)[0]]   # leitura seletiva
        X_test   = X_group[test_idx]
        y_test   = Y_local[test_idx]

        y_pred   = predict_all(model, X_test)
        test_acc = 100.0 * (y_pred == y_test).mean()
        cm       = confusion_matrix(y_test, y_pred)

        # Acurácia por SNR (todas as amostras do grupo)
        snr_accs = {}
        for snr in ALL_SNRS:
            mask = Z_group == snr
            if mask.sum() == 0:
                snr_accs[snr] = None
                continue
            yp = predict_all(model, X_group[mask])
            snr_accs[snr] = 100.0 * (yp == Y_local[mask]).mean()

        plot_entries.append((
            f'Grupo {gname}  (Acc={test_acc:.2f}%)',
            cm, class_names, snr_accs, test_acc,
            'Blues', cores_grupo[ci]
        ))
        print(f"  ✅ {gname:<6}  Acc = {test_acc:.2f}%")

# ── Modelo geral ──────────────────────────────────────────────────────────────

print("\nAvaliando modelo geral (via dataset original)...")

with h5py.File(h5_original, 'r') as f:
    # Y_grouped: índice de grupo (0-5) por amostra
    Y_grp  = np.argmax(f['Y_grouped'][:], axis=1) if 'Y_grouped' in f \
             else np.array([                              # deriva se não existir
                 next(gi for gi, mods in CLASSES_PER_GROUP.items()
                      if modulation_classes[y] in CLASSES_PER_GROUP[gi])
                 for y in Y_orig
             ])
    Z_orig = f['Z'][:, 0]

    _, _, test_idx_g = split(np.arange(len(Y_grp)), Y_grp)
    X_test_g = f['X'][test_idx_g]
    y_test_g = Y_grp[test_idx_g]

    y_pred_g   = predict_all(model_geral, X_test_g)
    test_acc_g = 100.0 * (y_pred_g == y_test_g).mean()
    cm_g       = confusion_matrix(y_test_g, y_pred_g)

    snr_accs_g = {}
    for snr in ALL_SNRS:
        mask = Z_orig == snr
        if mask.sum() == 0:
            snr_accs_g[snr] = None
            continue
        yp = predict_all(model_geral, f['X'][mask])
        snr_accs_g[snr] = 100.0 * (yp == Y_grp[mask]).mean()

plot_entries.append((
    f'Geral — 6 Grupos  (Acc={test_acc_g:.2f}%)',
    cm_g, list(GROUP_NAMES.values()),
    snr_accs_g, test_acc_g,
    'Greens', 'darkorange'
))
print(f"  ✅ Geral  Acc = {test_acc_g:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURA
# ══════════════════════════════════════════════════════════════════════════════

if not plot_entries:
    print("\n❌ Nenhum modelo avaliado. Verifique os modelos carregados.")
else:
    n_models = len(plot_entries)
    fig, axes = plt.subplots(
        n_models, 2,
        figsize=(16, 6.5 * n_models),
        gridspec_kw={'width_ratios': [1, 1.4]}
    )
    if n_models == 1:
        axes = np.array([axes])

    for row, (title, cm, cnames, snr_accs, test_acc, cmap, color) in enumerate(plot_entries):
        im = plot_cm_ax(cm, cnames, f'Matriz de Confusão\n{title}', axes[row,0], cmap=cmap)
        fig.colorbar(im, ax=axes[row,0], fraction=0.046, pad=0.04)
        plot_snr_ax(snr_accs, test_acc, f'Acurácia × SNR\n{title}', axes[row,1], color=color)

    fig.suptitle('Avaliação por Modelo — Matriz de Confusão + Acurácia × SNR',
                 fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    out = path + '/avaliacao_por_modelo.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Salvo: {out}")

    print(f"\n{'='*55}")
    print(f"  {'Modelo':<30} {'Classes':>8}  {'Acc Teste':>10}")
    print(f"{'─'*55}")
    for (title, _, cnames, _, test_acc, _, _) in plot_entries:
        label = title.split('(')[0].strip()
        print(f"  {label:<30} {len(cnames):>8}  {test_acc:>9.2f}%")
    print(f"{'='*55}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AVALIAÇÃO — streaming por batch, RAM mínima
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import confusion_matrix

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_SNRS   = list(range(-20, 32, 2))
BATCH_SIZE = 256

h5_original = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'

# ── Utilitários ───────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_batch(model, x_np):
    x = torch.from_numpy(x_np).float().permute(0, 2, 1).to(DEVICE)
    return model(x).argmax(dim=1).cpu().numpy()

def plot_cm_ax(cm, class_names, title, ax, cmap='Blues'):
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im   = ax.imshow(cm_n, vmin=0, vmax=1, cmap=cmap, interpolation='nearest')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Predito', fontsize=8)
    ax.set_ylabel('Verdadeiro', fontsize=8)
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks); ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(class_names, fontsize=7)
    for i in range(cm_n.shape[0]):
        for j in range(cm_n.shape[1]):
            v = cm_n[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if v > .5 else 'black')
    return im

def plot_snr_ax(snr_accs, test_acc, title, ax, color):
    snrs = sorted([s for s, a in snr_accs.items() if a is not None])
    vals = [snr_accs[s] for s in snrs]
    ax.plot(snrs, vals, marker='o', markersize=4, linewidth=1.8, color=color)
    ax.fill_between(snrs, vals, alpha=0.12, color=color)
    ax.axhline(test_acc, color='red', linestyle='--',
               linewidth=1.2, label=f'Acc total: {test_acc:.1f}%')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('SNR (dB)', fontsize=8); ax.set_ylabel('Acurácia (%)', fontsize=8)
    ax.set_ylim(0, 105); ax.set_xticks(ALL_SNRS[::2])
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Lê metadados uma única vez (Y e Z são pequenos) ──────────────────────────

print("Carregando metadados (Y, Z)...")
with h5py.File(h5_original, 'r') as f:
    Y_orig = np.argmax(f['Y'][:], axis=1)   # índice global 0-23  (~few MB)
    Z_orig = f['Z'][:, 0]                   # SNR por amostra      (~few MB)
    N_total = len(Y_orig)
    # Y_grouped se existir
    Y_grp = np.argmax(f['Y_grouped'][:], axis=1) if 'Y_grouped' in f else None

mod_to_global   = {m: i for i, m in enumerate(modulation_classes)}
cores_grupo     = plt.cm.tab10(np.linspace(0, 0.7, len(models_grupo)))
plot_entries    = []

# ══════════════════════════════════════════════════════════════════════════════
# MODELOS DE GRUPO — um por vez, streaming do HDF5
# ══════════════════════════════════════════════════════════════════════════════

for ci, (gname, model) in enumerate(models_grupo.items()):
    print(f"\n{'─'*50}")
    print(f"Avaliando grupo: {gname}")

    class_names = CLASSES_PER_GROUP[gname]
    global_idxs = [mod_to_global[m] for m in class_names]
    global_to_local = {g: l for l, g in enumerate(global_idxs)}

    # Índices das amostras deste grupo (só índices, sem carregar X)
    group_idx = np.where(np.isin(Y_orig, global_idxs))[0]
    Y_local   = np.array([global_to_local[Y_orig[i]] for i in group_idx])
    Z_group   = Z_orig[group_idx]

    if len(group_idx) == 0:
        print(f"  ⚠️  Nenhuma amostra encontrada para {gname}")
        continue

    # Split → índices de teste dentro de group_idx
    _, _, test_rel = split(np.arange(len(group_idx)), Y_local)
    test_abs       = group_idx[test_rel]   # posições absolutas no HDF5
    y_test         = Y_local[test_rel]

    # ── Avaliação no conjunto de teste — batch streaming ─────────────────────
    y_pred_test = []
    with h5py.File(h5_original, 'r') as f:
        for s in range(0, len(test_abs), BATCH_SIZE):
            idx_b = test_abs[s:s + BATCH_SIZE]
            # h5py aceita índices fancy apenas se ordenados
            order  = np.argsort(idx_b)
            idx_s  = idx_b[order]
            xb     = f['X'][idx_s]          # lê só este batch do disco
            # restaura ordem original
            xb     = xb[np.argsort(order)]
            y_pred_test.append(predict_batch(model, xb))
            del xb

    y_pred_test = np.concatenate(y_pred_test)
    test_acc    = 100.0 * (y_pred_test == y_test).mean()
    cm          = confusion_matrix(y_test, y_pred_test)
    del y_pred_test
    print(f"  Acc teste = {test_acc:.2f}%")

    # ── Acurácia por SNR — batch streaming ───────────────────────────────────
    snr_accs = {}
    with h5py.File(h5_original, 'r') as f:
        for snr in ALL_SNRS:
            snr_mask  = Z_group == snr
            snr_abs   = group_idx[snr_mask]
            y_snr     = Y_local[snr_mask]

            if len(snr_abs) == 0:
                snr_accs[snr] = None
                continue

            preds = []
            for s in range(0, len(snr_abs), BATCH_SIZE):
                idx_b  = snr_abs[s:s + BATCH_SIZE]
                order  = np.argsort(idx_b)
                xb     = f['X'][idx_b[order]][np.argsort(order)]
                preds.append(predict_batch(model, xb))
                del xb

            preds = np.concatenate(preds)
            snr_accs[snr] = 100.0 * (preds == y_snr).mean()
            del preds

    plot_entries.append((
        f'Grupo {gname}  (Acc={test_acc:.2f}%)',
        cm, class_names, snr_accs, test_acc,
        'Blues', cores_grupo[ci]
    ))
    print(f"  ✅ {gname} concluído")

    # Libera modelo da GPU após uso
    model.cpu()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ══════════════════════════════════════════════════════════════════════════════
# MODELO GERAL — streaming
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'─'*50}")
print("Avaliando modelo geral...")

if Y_grp is None:
    # Deriva Y_grp a partir do mapa de grupos
    grp_map = {}
    for gi, (gname, mods) in enumerate(CLASSES_PER_GROUP.items()):
        for m in mods:
            grp_map[mod_to_global[m]] = gi
    Y_grp = np.array([grp_map.get(y, -1) for y in Y_orig])

_, _, test_idx_g = split(np.arange(N_total), Y_grp)
y_test_g         = Y_grp[test_idx_g]
test_abs_g       = np.array(test_idx_g)

model_geral.to(DEVICE)
y_pred_g = []
with h5py.File(h5_original, 'r') as f:
    for s in range(0, len(test_abs_g), BATCH_SIZE):
        idx_b  = test_abs_g[s:s + BATCH_SIZE]
        order  = np.argsort(idx_b)
        xb     = f['X'][idx_b[order]][np.argsort(order)]
        y_pred_g.append(predict_batch(model_geral, xb))
        del xb

y_pred_g   = np.concatenate(y_pred_g)
test_acc_g = 100.0 * (y_pred_g == y_test_g).mean()
cm_g       = confusion_matrix(y_test_g, y_pred_g)
del y_pred_g
print(f"  Acc teste = {test_acc_g:.2f}%")

snr_accs_g = {}
with h5py.File(h5_original, 'r') as f:
    for snr in ALL_SNRS:
        mask   = Z_orig == snr
        abs_idx = np.where(mask)[0]
        y_snr  = Y_grp[mask]
        if len(abs_idx) == 0:
            snr_accs_g[snr] = None
            continue
        preds = []
        for s in range(0, len(abs_idx), BATCH_SIZE):
            idx_b = abs_idx[s:s + BATCH_SIZE]
            order = np.argsort(idx_b)
            xb    = f['X'][idx_b[order]][np.argsort(order)]
            preds.append(predict_batch(model_geral, xb))
            del xb
        preds = np.concatenate(preds)
        snr_accs_g[snr] = 100.0 * (preds == y_snr).mean()
        del preds

model_geral.cpu()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

plot_entries.append((
    f'Geral — 6 Grupos  (Acc={test_acc_g:.2f}%)',
    cm_g, list(GROUP_NAMES.values()),
    snr_accs_g, test_acc_g,
    'Greens', 'darkorange'
))
print("  ✅ Geral concluído")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURA
# ══════════════════════════════════════════════════════════════════════════════

if not plot_entries:
    print("\n❌ Nenhum modelo avaliado.")
else:
    n_models = len(plot_entries)
    fig, axes = plt.subplots(
        n_models, 2,
        figsize=(16, 6.5 * n_models),
        gridspec_kw={'width_ratios': [1, 1.4]}
    )
    if n_models == 1:
        axes = np.array([axes])

    for row, (title, cm, cnames, snr_accs, test_acc, cmap, color) in enumerate(plot_entries):
        im = plot_cm_ax(cm, cnames, f'Matriz de Confusão\n{title}', axes[row, 0], cmap=cmap)
        fig.colorbar(im, ax=axes[row, 0], fraction=0.046, pad=0.04)
        plot_snr_ax(snr_accs, test_acc, f'Acurácia × SNR\n{title}', axes[row, 1], color=color)

    fig.suptitle('Avaliação por Modelo — Matriz de Confusão + Acurácia × SNR',
                 fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    out = path + '/avaliacao_por_modelo.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Salvo: {out}")

    print(f"\n{'='*55}")
    print(f"  {'Modelo':<30} {'Classes':>8}  {'Acc Teste':>10}")
    print(f"{'─'*55}")
    for (title, _, cnames, _, test_acc, _, _) in plot_entries:
        print(f"  {title.split('(')[0].strip():<30} {len(cnames):>8}  {test_acc:>9.2f}%")
    print(f"{'='*55}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODELO GERAL — baixíssimo consumo de RAM
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'─'*50}")
print("Avaliando modelo geral...")

# Y_grp já calculado anteriormente; Z_orig também
# ── Amostragem estratificada leve ────────────────────────────────────────────
# Em vez de usar split() no dataset inteiro, pega até MAX_PER_CLASS
# amostras de teste por grupo → RAM controlada

MAX_PER_CLASS = 2000   # ajuste para cima se houver RAM sobrando

rng = np.random.default_rng(42)
test_abs_g_list = []
y_test_g_list   = []

for grp_id in np.unique(Y_grp):
    idx_grp = np.where(Y_grp == grp_id)[0]
    n_pick  = min(MAX_PER_CLASS, len(idx_grp))
    chosen  = rng.choice(idx_grp, size=n_pick, replace=False)
    test_abs_g_list.append(chosen)
    y_test_g_list.append(np.full(n_pick, grp_id))

test_abs_g = np.concatenate(test_abs_g_list)
y_test_g   = np.concatenate(y_test_g_list)

# Ordena para leitura sequencial no HDF5 (muito mais rápido)
order      = np.argsort(test_abs_g)
test_abs_g = test_abs_g[order]
y_test_g   = y_test_g[order]

del test_abs_g_list, y_test_g_list, order

print(f"  Amostras de teste: {len(test_abs_g):,}  "
      f"({MAX_PER_CLASS} por grupo × {len(np.unique(Y_grp))} grupos)")

# ── Predição no teste — batch streaming ──────────────────────────────────────
model_geral.to(DEVICE)
correct = 0
total   = 0
y_pred_g_all = []   # só para a CM — integers, custo baixo

with h5py.File(h5_original, 'r') as f:
    for s in range(0, len(test_abs_g), BATCH_SIZE):
        idx_b  = test_abs_g[s:s + BATCH_SIZE]
        xb     = f['X'][idx_b]              # sequencial → sem reordenação
        yb     = y_test_g[s:s + BATCH_SIZE]
        preds  = predict_batch(model_geral, xb)
        correct += (preds == yb).sum()
        total   += len(yb)
        y_pred_g_all.append(preds)
        del xb, preds

test_acc_g = 100.0 * correct / total
y_pred_g   = np.concatenate(y_pred_g_all)
del y_pred_g_all
cm_g = confusion_matrix(y_test_g, y_pred_g,
                        labels=list(range(len(GROUP_NAMES))))
del y_pred_g
print(f"  Acc teste = {test_acc_g:.2f}%")

# ── Acurácia por SNR — um SNR por vez, sem acumular ──────────────────────────
snr_accs_g = {}
with h5py.File(h5_original, 'r') as f:
    for snr in ALL_SNRS:
        # Índices absolutos deste SNR, ordenados
        snr_abs = np.where(Z_orig == snr)[0]   # já são posições absolutas
        y_snr   = Y_grp[snr_abs]

        if len(snr_abs) == 0:
            snr_accs_g[snr] = None
            continue

        # Sub-amostra por SNR se ainda for grande
        if len(snr_abs) > MAX_PER_CLASS * len(GROUP_NAMES):
            chosen  = rng.choice(len(snr_abs),
                                 size=MAX_PER_CLASS * len(GROUP_NAMES),
                                 replace=False)
            chosen.sort()
            snr_abs = snr_abs[chosen]
            y_snr   = y_snr[chosen]
            del chosen

        correct_snr = 0
        total_snr   = 0
        for s in range(0, len(snr_abs), BATCH_SIZE):
            idx_b = snr_abs[s:s + BATCH_SIZE]
            xb    = f['X'][idx_b]
            yb    = y_snr[s:s + BATCH_SIZE]
            preds = predict_batch(model_geral, xb)
            correct_snr += (preds == yb).sum()
            total_snr   += len(yb)
            del xb, preds

        snr_accs_g[snr] = 100.0 * correct_snr / total_snr
        del snr_abs, y_snr
        print(f"    SNR {snr:>4} dB → {snr_accs_g[snr]:.2f}%")

model_geral.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

plot_entries.append((
    f'Geral — 6 Grupos  (Acc={test_acc_g:.2f}%)',
    cm_g, list(GROUP_NAMES.values()),
    snr_accs_g, test_acc_g,
    'Greens', 'darkorange'
))
print("  ✅ Geral concluído")

In [ ]:

model_geral = load_model("/content/cnn_GROUP_ep001_acc88.61_lr1.00e-03.pth", num_classes=6)


models_grupo = {}
for gname in GROUP_NAMES.values():
    nc    = len(CLASSES_PER_GROUP[gname])
    ckpt  = find_checkpoint(MODEL_BASE_DIR, f'cnn_{gname}') \
         or find_checkpoint('/content',      f'cnn_{gname}')
    if ckpt is None:
        print(f"  ⚠️  {gname}: checkpoint não encontrado — grupo será ignorado")
        continue
    models_grupo[gname] = load_model(ckpt, num_classes=nc)
    print(f"  ✓ {gname:<6}       : {os.path.basename(ckpt)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CÉLULA PARA GARANTIR GERAÇÃO DE TODOS OS HDF5 NECESSÁRIOS PARA O PLOT
# ══════════════════════════════════════════════════════════════════════════════

import os, json, shutil, random
import h5py
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÕES (replicadas da célula de treinamento unificada p39SJmZ61vh7)
# ══════════════════════════════════════════════════════════════════════════════

# Certifique-se de que `path` e `modulation_classes_path` estejam definidos
# de células anteriores (e.g., `wLOx9RJwg8HZ`).
# Se não estiverem, descomente e ajuste as linhas abaixo:
# import kagglehub
# path = kagglehub.dataset_download("pinxau1000/radioml2018")
# modulation_classes_path = path + '/classes-fixed.json'

# Desired SNRs for all generated HDF5 files
DESIRED_SNRS = list(range(-20, 32, 2)) # All SNRs from -20 to 30 dB

BATCH_SIZE_IO = 2048 # Batch size for HDF5 I/O operations
LOCAL_DIR    = "/content" # Directory to store generated HDF5 files locally

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS (igual ao resto do notebook)
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES  = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
ALL_MODELS   = ["GROUP", "ASK", "PSK", "APSK", "QAM", "AM", "FM"]

INPUT_FILE   = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE = modulation_classes_path

mod_classes = json.load(open(CLASSES_FILE))

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÃO DE CONSTRUÇÃO DE HDF5 (copiada da célula p39SJmZ61vh7)
# ══════════════════════════════════════════════════════════════════════════════

def _build_hdf5(input_file, output_file, target_snrs,
                group_indices, global_to_local, num_classes,
                modelo_id, batch_size=BATCH_SIZE_IO, mod_classes=mod_classes, GROUP_MAP=GROUP_MAP):
    """Filtra INPUT_FILE por SNR (e grupo se intra-grupo) e grava output_file."""
    snr_set = set(target_snrs)
    if os.path.exists(output_file):
        os.remove(output_file)

    print(f"\n  🔨 Construindo HDF5 para '{modelo_id}'...")

    original_indices = []
    with h5py.File(input_file, 'r') as src:
        N     = src['X'].shape[0]
        Z_ds  = src['Z']
        Y_ds  = src['Y']
        n_b   = int(np.ceil(N / batch_size))

        for i in range(n_b):
            s, e     = i * batch_size, min((i + 1) * batch_size, N)
            z_batch  = Z_ds[s:e, 0]
            y_batch  = np.argmax(Y_ds[s:e], axis=1)

            if modelo_id == "GROUP":
                mask = np.isin(z_batch, list(snr_set))
            else:
                mask = (np.isin(z_batch, list(snr_set)) &
                        np.isin(y_batch, group_indices))

            original_indices.extend((np.where(mask)[0] + s).tolist())
            if (i + 1) % 50 == 0 or i == n_b - 1:
                print(f"    Batch {i+1:>4}/{n_b}  |  amostras: {len(original_indices)}")

    original_indices = np.array(original_indices, dtype=np.int64)
    M = len(original_indices)
    if M == 0:
        print(f"  ⚠️  Nenhuma amostra encontrada para '{modelo_id}' com os filtros escolhidos.")
        return None

    NUM_GROUPS = 6
    lookup = np.array([GROUP_MAP[mod_classes[i]]
                       for i in range(len(mod_classes))], dtype=np.int64)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]

        with h5py.File(output_file, 'w') as out:
            # Metadados para reconstrução futura
            out.attrs['snrs']          = target_snrs
            out.attrs['modelo_id']     = modelo_id
            out.attrs['num_classes']   = num_classes
            out.attrs['group_indices'] = group_indices.tolist()
            out.attrs['total_samples'] = M
            out.attrs['seed']          = 42 # Using a fixed seed for reproducibility

            ds_X  = out.create_dataset('X', shape=(M,)+x_shape, dtype=src['X'].dtype)
            ds_Z  = out.create_dataset('Z', shape=(M,)+z_shape, dtype=src['Z'].dtype)

            if modelo_id == "GROUP":
                # Y_grouped: one-hot de 6 grupos
                ds_Y = out.create_dataset('Y_grouped',
                                          shape=(M, NUM_GROUPS), dtype=np.float32)
            else:
                # Y: one-hot local (num_classes do grupo)
                ds_Y = out.create_dataset('Y',
                                          shape=(M, num_classes), dtype=np.int32)

            n_b = int(np.ceil(M / batch_size))
            for i in range(n_b):
                s, e = i * batch_size, min((i + 1) * batch_size, M)
                idx  = original_indices[s:e]

                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]

                y_global = np.argmax(src['Y'][idx], axis=1)

                if modelo_id == "GROUP":
                    y_group = lookup[y_global]
                    yg = np.zeros((e - s, NUM_GROUPS), dtype=np.float32)
                    yg[np.arange(e - s), y_group] = 1.0
                    ds_Y[s:e] = yg
                else:
                    y_local = np.array([global_to_local[int(g)] for g in y_global])
                    yo = np.zeros((e - s, num_classes), dtype=np.int32)
                    yo[np.arange(e - s), y_local] = 1
                    ds_Y[s:e] = yo

                if (i + 1) % 20 == 0 or i == n_b - 1:
                    print(f"    Escrevendo batch {i+1:>4}/{n_b}")

    print(f"  ✅ HDF5 criado: {output_file}  ({M} amostras)")
    return output_file

# ══════════════════════════════════════════════════════════════════════════════
# GERAR HDF5 PARA TODOS OS MODELOS
# ══════════════════════════════════════════════════════════════════════════════

snr_tag_for_files = '_'.join(str(s) for s in DESIRED_SNRS)

for MODELO_ALVO in ALL_MODELS:
    print(f"\n{'='*70}")
    print(f"Processando MODELO_ALVO: {MODELO_ALVO}")
    print(f"{'='*70}")

    if MODELO_ALVO == "GROUP":
        group_indices   = np.arange(len(mod_classes), dtype=np.int64)
        num_classes     = 6
        global_to_local = {int(g): GROUP_MAP[mod_classes[g]] for g in group_indices}
        h5_name   = f'{'_'.join(sorted(GROUP_NAMES.values()))}_subset_snr_{snr_tag_for_files}.hdf5'
    else:
        group_indices = np.array(
            sorted([i for i, m in enumerate(mod_classes)
                    if GROUP_NAMES[GROUP_MAP[m]] == MODELO_ALVO]),
            dtype=np.int64
        )
        num_classes     = len(group_indices)
        global_to_local = {int(g): l for l, g in enumerate(group_indices)}
        h5_name   = f'{MODELO_ALVO}_subset_snr_{snr_tag_for_files}.hdf5'

    output_file_path = os.path.join(LOCAL_DIR, h5_name)

    _build_hdf5(INPUT_FILE, output_file_path, DESIRED_SNRS,
                group_indices, global_to_local, num_classes,
                MODELO_ALVO, batch_size=BATCH_SIZE_IO)

print("\n✅ Todos os arquivos HDF5 necessários foram gerados.")
print("Agora você pode executar a célula de plotagem novamente.")

#BUSCA ESPECTOGRAMA


In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   BLOCO UNIFICADO — SELEÇÃO DE GRUPO/SNR →                              ║
# ║                     BUSCA DE BANDA DE ESPECTROGRAMA                     ║
# ║                                                                          ║
# ║  BUGS CORRIGIDOS (em relação ao notebook original):                      ║
# ║    ① Split era feito sobre o dataset completo (24 classes / todos SNRs) ║
# ║      com índices incompatíveis com o HDF5 filtrado por grupo             ║
# ║    ② H5PyDataset usava data_Y_name='Y' fixo, ignorando h5_label_key     ║
# ║    ③ train_indices_sorted era sobrescrito por splits errados             ║
# ║                                                                          ║
# ║  Pré-requisitos:                                                         ║
# ║    • path e modulation_classes_path definidos (kagglehub)                ║
# ║    • Drive montado                                                       ║
# ║    • pip install scikit-image scipy                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, shutil, random
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.signal import spectrogram as scipy_spectrogram
from skimage.transform import resize

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO GERAL — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

MODELO_ALVO  = "ASK"
# Opções: "GROUP" | "ASK" | "PSK" | "APSK" | "QAM" | "AM" | "FM"

DESIRED_SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]

BATCH_SIZE   = 64
LR           = 5e-5
SEED         = 42

# Acurácia do modelo baseline já treinado (Conv1D sobre sinal bruto).
# Usada como referência nos gráficos e no sumário final.
# Preencha com o valor obtido pelo bloco de treino principal.
BASELINE_VAL_ACC = 95.0   # ← substitua pelo valor real

DRIVE_BASE       = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR        = "/content"
HDF5_BUILD_BATCH = 2048

# ── Configuração da busca de espectrograma ────────────────────────────────────

# Bloco A — frações absolutas da banda total testadas
BAND_FRACTIONS = [0.9]

# Bloco B — percentuais de potência acumulada testados
POWER_PCTS     = [99.9, 99.0, 97.0, 95.0, 90.0, 80.0, 60.0]

SPEC_HEIGHT    = 64        # altura do espectrograma após crop + resize
SPEC_WIDTH     = 64        # largura temporal
NPERSEG        = 128
NOVERLAP       = 120
SPEC_MODE      = 'IQ'      # 'I' | 'IQ'

EPOCHS_SEARCH  = 30        # épocas por candidato na busca (rápido)
N_CALIB        = 512       # amostras para estimar PSD média (Bloco B)

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
ALL_MODELS  = ["GROUP", "ASK", "PSK", "APSK", "QAM", "AM", "FM"]

INPUT_FILE   = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE = modulation_classes_path

assert MODELO_ALVO in ALL_MODELS, f"MODELO_ALVO inválido: {MODELO_ALVO}"

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def _set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ══════════════════════════════════════════════════════════════════════════════
# DERIVAÇÕES — group_indices, num_classes, global_to_local, h5_label_key
# ══════════════════════════════════════════════════════════════════════════════

mod_classes = json.load(open(CLASSES_FILE))

if MODELO_ALVO == "GROUP":
    group_indices   = np.arange(len(mod_classes), dtype=np.int64)
    num_classes     = 6
    global_to_local = {int(g): GROUP_MAP[mod_classes[g]] for g in group_indices}
    h5_label_key    = "Y_grouped"
    _label_note     = "6 grupos (ASK/PSK/APSK/QAM/AM/FM)"
else:
    # FIX ① — filtra apenas as classes do grupo escolhido
    group_indices = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_NAMES[GROUP_MAP[m]] == MODELO_ALVO]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}
    h5_label_key    = "Y"          # one-hot LOCAL com num_classes colunas
    _label_note     = f"{num_classes} classes intra-grupo {MODELO_ALVO}"

snr_tag   = "_".join(str(s) for s in DESIRED_SNRS)
h5_name   = (f"GROUP_subset_snr_{snr_tag}.hdf5" if MODELO_ALVO == "GROUP"
             else f"{MODELO_ALVO}_subset_snr_{snr_tag}.hdf5")
h5_local  = os.path.join(LOCAL_DIR, h5_name)
h5_drive  = os.path.join(DRIVE_BASE, MODELO_ALVO, "subset.hdf5")
ckpt_drive= os.path.join(DRIVE_BASE, MODELO_ALVO, "checkpoint.pth")
meta_drive= os.path.join(DRIVE_BASE, MODELO_ALVO, "meta.json")
idx_drive = lambda name: os.path.join(DRIVE_BASE, MODELO_ALVO, f"{name}.npy")

sep = "═" * 65
print(sep)
print(f"  Modelo      : {MODELO_ALVO}  ({_label_note})")
print(f"  num_classes : {num_classes}")
print(f"  h5_label_key: {h5_label_key}")
print(f"  SNRs        : {DESIRED_SNRS}")
print(f"  Device      : {device}")
print(sep)
print(f"\n  Classes do grupo '{MODELO_ALVO}':")
for _l, _g in enumerate(group_indices):
    print(f"    label_local={_l}  ←  global={_g}  ({mod_classes[_g]})")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1 — VERIFICAR DRIVE
# ══════════════════════════════════════════════════════════════════════════════

assert os.path.exists("/content/drive/MyDrive"), (
    "❌ Google Drive não montado! Execute:\n"
    "   from google.colab import drive\n"
    "   drive.mount('/content/drive', force_remount=True)"
)

has_h5   = os.path.exists(h5_drive)
has_ckpt = os.path.exists(ckpt_drive)
has_idx  = all(os.path.exists(idx_drive(n))
               for n in ("train_indices", "val_indices", "test_indices"))

print(f"\n📂 Drive → {os.path.join(DRIVE_BASE, MODELO_ALVO)}")
print(f"   HDF5       : {'✅' if has_h5   else '❌ será criado'}")
print(f"   Checkpoint : {'✅' if has_ckpt else '❌ treino do zero'}")
print(f"   Índices    : {'✅' if has_idx  else '❌ será gerado'}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — CRIAR / CARREGAR HDF5 FILTRADO POR GRUPO + SNR
#
# FIX ① — O HDF5 contém APENAS as classes do grupo e os SNRs desejados.
#          Y gravado com índices LOCAIS (0…num_classes-1), não globais (0-23).
#          Índices internos 0…M-1 — completamente independentes do arquivo original.
# ══════════════════════════════════════════════════════════════════════════════

def _build_hdf5(input_file, output_file, target_snrs,
                group_indices, global_to_local, num_classes,
                modelo_id, batch_size=2048):
    snr_set = set(target_snrs)
    if os.path.exists(output_file):
        os.remove(output_file)

    print(f"\n  🔨 Construindo HDF5 para '{modelo_id}'...")

    original_indices = []
    with h5py.File(input_file, 'r') as src:
        N    = src['X'].shape[0]
        n_b  = int(np.ceil(N / batch_size))
        for i in range(n_b):
            s, e    = i * batch_size, min((i + 1) * batch_size, N)
            z_batch = src['Z'][s:e, 0]
            y_batch = np.argmax(src['Y'][s:e], axis=1)
            if modelo_id == "GROUP":
                mask = np.isin(z_batch, list(snr_set))
            else:
                mask = (np.isin(z_batch, list(snr_set)) &
                        np.isin(y_batch, group_indices))   # ← FIX ①
            original_indices.extend((np.where(mask)[0] + s).tolist())
            if (i + 1) % 50 == 0 or i == n_b - 1:
                print(f"    Varredura {i+1:>4}/{n_b}  | selecionados: {len(original_indices)}")

    original_indices = np.array(original_indices, dtype=np.int64)
    M = len(original_indices)
    if M == 0:
        raise ValueError(f"Nenhuma amostra para '{modelo_id}' com SNRs={sorted(snr_set)}")
    print(f"     Total: {M} amostras")

    NUM_GROUPS   = 6
    lookup_group = np.array([GROUP_MAP[mod_classes[i]]
                             for i in range(len(mod_classes))], dtype=np.int64)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]
        with h5py.File(output_file, 'w') as out:
            out.attrs['snrs']          = target_snrs
            out.attrs['modelo_id']     = modelo_id
            out.attrs['num_classes']   = num_classes
            out.attrs['group_indices'] = group_indices.tolist()
            out.attrs['total_samples'] = M
            out.attrs['seed']          = SEED
            ds_X = out.create_dataset('X', shape=(M,) + x_shape, dtype=src['X'].dtype)
            ds_Z = out.create_dataset('Z', shape=(M,) + z_shape, dtype=src['Z'].dtype)
            if modelo_id == "GROUP":
                ds_Y = out.create_dataset('Y_grouped', shape=(M, NUM_GROUPS), dtype=np.float32)
            else:
                ds_Y = out.create_dataset('Y', shape=(M, num_classes), dtype=np.int32)  # ← FIX ①

            n_b = int(np.ceil(M / batch_size))
            for i in range(n_b):
                s, e     = i * batch_size, min((i + 1) * batch_size, M)
                idx      = original_indices[s:e]
                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]
                y_global  = np.argmax(src['Y'][idx], axis=1)
                if modelo_id == "GROUP":
                    y_grp = lookup_group[y_global]
                    yg    = np.zeros((e-s, NUM_GROUPS), dtype=np.float32)
                    yg[np.arange(e-s), y_grp] = 1.0
                    ds_Y[s:e] = yg
                else:
                    y_local = np.array([global_to_local[int(g)] for g in y_global])
                    yo      = np.zeros((e-s, num_classes), dtype=np.int32)
                    yo[np.arange(e-s), y_local] = 1
                    ds_Y[s:e] = yo
                if (i + 1) % 20 == 0 or i == n_b - 1:
                    print(f"    Escrita {i+1:>4}/{n_b}")

    print(f"  ✅ HDF5 criado: {output_file}  ({M} amostras)")
    return output_file


if has_h5:
    if not os.path.exists(h5_local):
        print(f"\n📦 Copiando HDF5 do Drive → {h5_local}…")
        shutil.copy(h5_drive, h5_local)
        print("  ✅ Cópia concluída.")
    else:
        print(f"\n📦 HDF5 já local: {h5_local}")
else:
    _build_hdf5(INPUT_FILE, h5_local, DESIRED_SNRS,
                group_indices, global_to_local, num_classes, MODELO_ALVO,
                batch_size=HDF5_BUILD_BATCH)
    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    shutil.copy(h5_local, h5_drive)
    print(f"  ✅ HDF5 salvo no Drive: {h5_drive}")

# Verificação do arquivo gerado
with h5py.File(h5_local, 'r') as _f:
    _M            = _f['X'].shape[0]
    _y_local      = np.argmax(_f[h5_label_key][:], axis=1)
    _classes_found= np.unique(_y_local).tolist()

assert _classes_found == list(range(num_classes)), \
    f"❌ Labels inesperados no HDF5! {_classes_found} ≠ {list(range(num_classes))}"
print(f"\n✅ HDF5 verificado: {_M} amostras | labels {_classes_found}")
del _y_local

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — ÍNDICES TRAIN / VAL / TEST
#
# FIX ② — split feito sobre h5_local (índices 0…M-1) com h5_label_key correto.
#          Nunca sobre o dataset original completo.
# ══════════════════════════════════════════════════════════════════════════════

def _split_deterministic(h5_path, label_key, seed=42):
    with h5py.File(h5_path, 'r') as f:
        y_all = np.argmax(f[label_key][:], axis=1)   # labels locais 0…num_classes-1
    indices = np.arange(len(y_all))                   # 0…M-1 do h5_local
    tv_idx, te_idx, tv_lbl, _ = train_test_split(
        indices, y_all, test_size=0.20, random_state=seed, stratify=y_all)
    tr_idx, va_idx, _, _ = train_test_split(
        tv_idx, tv_lbl, test_size=0.25, random_state=seed, stratify=tv_lbl)
    return np.sort(tr_idx), np.sort(va_idx), np.sort(te_idx)


if has_idx:
    print("\n📐 Carregando índices do Drive…")
    train_indices = np.load(idx_drive("train_indices"))
    val_indices   = np.load(idx_drive("val_indices"))
    test_indices  = np.load(idx_drive("test_indices"))
    assert (train_indices.max() < _M and
            val_indices.max()   < _M and
            test_indices.max()  < _M), \
        (f"❌ Índices do Drive incompatíveis com HDF5 (M={_M}). "
         "Delete os .npy do Drive e reexecute.")
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")
    print("   ✅ Índices compatíveis com o HDF5")
else:
    print("\n📐 Gerando índices sobre h5_local (0…M-1)…")
    train_indices, val_indices, test_indices = _split_deterministic(
        h5_local, h5_label_key, seed=SEED)
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")
    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    np.save(idx_drive("train_indices"), train_indices)
    np.save(idx_drive("val_indices"),   val_indices)
    np.save(idx_drive("test_indices"),  test_indices)
    print("   ✅ Índices salvos no Drive.")

tr_s, va_s, te_s = set(train_indices), set(val_indices), set(test_indices)
assert not (tr_s & va_s), "❌ Sobreposição treino/val!"
assert not (tr_s & te_s), "❌ Sobreposição treino/test!"
assert not (va_s & te_s), "❌ Sobreposição val/test!"
print("   ✅ Anti-leakage OK")

# FIX ③ — estas variáveis NÃO são sobrescritas depois daqui
train_indices_sorted = train_indices
val_indices_sorted   = val_indices
test_indices_sorted  = test_indices
h5py_path            = h5_local

# ══════════════════════════════════════════════════════════════════════════════
# CLASSES DE DATASET
# ══════════════════════════════════════════════════════════════════════════════

# ── Dataset com crop de banda absoluta (Bloco A) ──────────────────────────────

class H5PyDatasetBandCrop(Dataset):
    """
    Espectrograma real (scipy) com crop simétrico em frequência.
    band_fraction=1.0 → banda completa; 0.5 → 50% centrais.
    """
    def __init__(self, h5_filepath, data_X_name, data_Y_name, indices, formats,
                 band_fraction=1.0, spec_mode='IQ',
                 spec_height=64, spec_width=64, nperseg=128, noverlap=120):
        self.h5_filepath   = h5_filepath
        self.data_X_name   = data_X_name
        self.data_Y_name   = data_Y_name
        self.indices       = np.array(indices, dtype=np.int64)
        self.formats       = formats
        self.band_fraction = float(np.clip(band_fraction, 1e-3, 1.0))
        self.spec_mode     = spec_mode
        self.spec_height   = spec_height
        self.spec_width    = spec_width
        self.nperseg       = nperseg
        self.noverlap      = noverlap
        self.file          = None

    def __len__(self):
        return len(self.indices)

    def _compute_spec(self, signal_1d):
        _, _, Sxx = scipy_spectrogram(
            signal_1d, fs=1.0, nperseg=self.nperseg,
            noverlap=self.noverlap, window='hann', scaling='spectrum')
        Sxx_db = 10 * np.log10(Sxx + 1e-12)        # (n_freq, n_time)
        n_freq  = Sxx_db.shape[0]
        keep    = max(1, int(round(n_freq * self.band_fraction)))
        margin  = (n_freq - keep) // 2
        Sxx_db  = Sxx_db[margin: margin + keep, :]  # crop simétrico central
        return resize(Sxx_db, (self.spec_height, self.spec_width),
                      anti_aliasing=True, preserve_range=True).astype(np.float32)

    def __getitem__(self, idx):
        if self.file is None:
            self.file   = h5py.File(self.h5_filepath, 'r')
            self.X_data = self.file[self.data_X_name]
            self.Y_data = self.file[self.data_Y_name]
        real_idx = self.indices[idx]
        x = self.X_data[real_idx]
        y = self.Y_data[real_idx]
        mean, std = x.mean(), x.std()
        x = (x - mean) / (std + 1e-8)
        I_sig, Q_sig = x[:, 0], x[:, 1]
        if self.spec_mode == 'I':
            spec = np.expand_dims(self._compute_spec(I_sig), 0)
        else:
            spec = np.stack([self._compute_spec(I_sig),
                             self._compute_spec(Q_sig)], axis=0)
        s_min, s_max = spec.min(), spec.max()
        spec = (spec - s_min) / (s_max - s_min + 1e-8)
        if self.formats == 0 and hasattr(y, '__len__'):
            y = np.argmax(y)
        return torch.from_numpy(spec).float(), torch.tensor(int(y)).long()


# ── Dataset com crop por potência acumulada (Bloco B) ─────────────────────────

class H5PyDatasetPowerCrop(Dataset):
    """
    Espectrograma real com crop a partir dos bins de menor frequência,
    mantendo apenas os bins que acumulam band_fraction da potência total.
    band_fraction é calculado externamente via _power_band_fraction().
    """
    def __init__(self, h5_filepath, data_X_name, data_Y_name, indices, formats,
                 band_fraction=1.0, spec_mode='IQ',
                 spec_height=64, spec_width=64, nperseg=128, noverlap=120):
        self.h5_filepath   = h5_filepath
        self.data_X_name   = data_X_name
        self.data_Y_name   = data_Y_name
        self.indices       = np.array(indices, dtype=np.int64)
        self.formats       = formats
        self.band_fraction = float(np.clip(band_fraction, 1e-3, 1.0))
        self.spec_mode     = spec_mode
        self.spec_height   = spec_height
        self.spec_width    = spec_width
        self.nperseg       = nperseg
        self.noverlap      = noverlap
        self.file          = None

    def __len__(self):
        return len(self.indices)

    def _compute_spec(self, signal_1d):
        _, _, Sxx = scipy_spectrogram(
            signal_1d, fs=1.0, nperseg=self.nperseg,
            noverlap=self.noverlap, window='hann', scaling='spectrum')
        Sxx_db = 10 * np.log10(Sxx + 1e-12)
        n_freq  = Sxx_db.shape[0]
        n_keep  = max(1, int(round(n_freq * self.band_fraction)))
        Sxx_db  = Sxx_db[:n_keep, :]               # bins de menor frequência
        return resize(Sxx_db, (self.spec_height, self.spec_width),
                      anti_aliasing=True, preserve_range=True).astype(np.float32)

    def __getitem__(self, idx):
        if self.file is None:
            self.file   = h5py.File(self.h5_filepath, 'r')
            self.X_data = self.file[self.data_X_name]
            self.Y_data = self.file[self.data_Y_name]
        real_idx = self.indices[idx]
        x = self.X_data[real_idx]
        y = self.Y_data[real_idx]
        mean, std = x.mean(), x.std()
        x = (x - mean) / (std + 1e-8)
        I_sig, Q_sig = x[:, 0], x[:, 1]
        if self.spec_mode == 'I':
            spec = np.expand_dims(self._compute_spec(I_sig), 0)
        else:
            spec = np.stack([self._compute_spec(I_sig),
                             self._compute_spec(Q_sig)], axis=0)
        s_min, s_max = spec.min(), spec.max()
        spec = (spec - s_min) / (s_max - s_min + 1e-8)
        if self.formats == 0 and hasattr(y, '__len__'):
            y = np.argmax(y)
        return torch.from_numpy(spec).float(), torch.tensor(int(y)).long()

# ══════════════════════════════════════════════════════════════════════════════
# MODELOS
# ══════════════════════════════════════════════════════════════════════════════

class CNN2D(nn.Module):
    """Conv2D — opera sobre espectrograma (C, H, W)."""
    def __init__(self, num_classes, in_channels=2, input_size=(64, 64)):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64,  kernel_size=3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        with torch.no_grad():
            self.n_flatten = self.features(torch.zeros(1, in_channels, *input_size)).view(1,-1).size(1)
        print(f"   CNN2D flatten: {self.n_flatten}")
        self.classifier = nn.Sequential(
            nn.Linear(self.n_flatten, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES DE TREINO
# ══════════════════════════════════════════════════════════════════════════════

def _train_one_config(model, tr_loader, va_loader, epochs, lr, dev):
    """Treina e retorna lista de (train_loss, val_acc) por época."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-7)
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(dev), yb.to(dev)
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(xb), yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * len(xb)
        avg_loss = total_loss / len(tr_loader.dataset)
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(dev), yb.to(dev)
                correct += (model(xb).argmax(1) == yb).sum().item()
                total   += len(yb)
        val_acc = 100.0 * correct / total
        scheduler.step(val_acc)
        history.append((avg_loss, val_acc))
        print(f"    ep {ep:>3}/{epochs}  loss={avg_loss:.4f}  val_acc={val_acc:.2f}%")
    return history


def _make_loader_band(band_frac, DatasetClass, h5_path, label_key,
                      idx, spec_mode, spec_h, spec_w,
                      nperseg, noverlap, batch_size, seed, shuffle):
    ds = DatasetClass(
        h5_filepath=h5_path, data_X_name='X', data_Y_name=label_key,
        indices=idx, formats=0, band_fraction=band_frac,
        spec_mode=spec_mode, spec_height=spec_h, spec_width=spec_w,
        nperseg=nperseg, noverlap=noverlap)
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True,
                      persistent_workers=True,
                      **({"generator": g} if shuffle else {}))

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — SANIDADE DO DATASET DE ESPECTROGRAMA
# Verifica labels antes de iniciar a busca
# ══════════════════════════════════════════════════════════════════════════════

_ds_check = H5PyDatasetBandCrop(
    h5_local, 'X', h5_label_key, val_indices[:256], 0,
    band_fraction=1.0, spec_mode=SPEC_MODE,
    spec_height=SPEC_HEIGHT, spec_width=SPEC_WIDTH,
    nperseg=NPERSEG, noverlap=NOVERLAP)
_xb, _yb = next(iter(DataLoader(_ds_check, batch_size=64)))
assert _yb.min() >= 0 and _yb.max() < num_classes, \
    f"❌ Labels fora do range! [{_yb.min()}, {_yb.max()}] ≠ [0, {num_classes-1}]"
print(f"\n✅ Sanidade dataset espectrograma:")
print(f"   spec.shape={tuple(_xb.shape)}  y∈[0,{_yb.max().item()}]  (esperado [0,{num_classes-1}])")
del _ds_check, _xb, _yb

baseline_val_acc = BASELINE_VAL_ACC
print(f"   Referência baseline: {baseline_val_acc:.2f}%")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCO A — BUSCA: BANDA DE FREQUÊNCIA ABSOLUTA
#
# Para cada band_fraction em BAND_FRACTIONS:
#   • Gera espectrogramas com crop simétrico em torno da frequência central
#   • Treina CNN2D por EPOCHS_SEARCH épocas
#   • Registra melhor val_acc
# ══════════════════════════════════════════════════════════════════════════════

in_ch = 1 if SPEC_MODE == 'I' else 2
results_band = []

print(f"\n{sep}")
print(f"  BLOCO A — Busca de banda absoluta")
print(f"  Candidatos : {BAND_FRACTIONS}")
print(f"  Épocas/run : {EPOCHS_SEARCH}  |  spec={SPEC_HEIGHT}×{SPEC_WIDTH}  mode={SPEC_MODE}")
print(sep)

for band_frac in BAND_FRACTIONS:
    print(f"\n{'─'*65}")
    print(f"  ▶  band_fraction = {band_frac:.4f}  ({band_frac*100:.1f}% da banda)")
    print(f"{'─'*65}")
    _set_seeds(SEED)

    tr_ld = _make_loader_band(band_frac, H5PyDatasetBandCrop,
                              h5_local, h5_label_key, train_indices,
                              SPEC_MODE, SPEC_HEIGHT, SPEC_WIDTH,
                              NPERSEG, NOVERLAP, BATCH_SIZE, SEED, shuffle=True)
    va_ld = _make_loader_band(band_frac, H5PyDatasetBandCrop,
                              h5_local, h5_label_key, val_indices,
                              SPEC_MODE, SPEC_HEIGHT, SPEC_WIDTH,
                              NPERSEG, NOVERLAP, BATCH_SIZE, SEED, shuffle=False)

    model_A = CNN2D(num_classes, in_ch, (SPEC_HEIGHT, SPEC_WIDTH)).to(device)
    hist    = _train_one_config(model_A, tr_ld, va_ld, EPOCHS_SEARCH, LR, device)
    best_A  = max(h[1] for h in hist)

    results_band.append({
        "band_fraction": band_frac,
        "bandwidth_pct": band_frac * 100,
        "best_val_acc":  best_A,
        "history":       hist,
    })
    print(f"  ✅  band={band_frac:.4f}  →  val_acc={best_A:.2f}%")

# ── Resultados Bloco A ────────────────────────────────────────────────────────

print(f"\n{sep}")
print("  RESULTADOS BLOCO A")
print(f"  {'Banda (%)':>10}  {'Val Acc (%)':>12}")
print("  " + "─"*25)
for r in results_band:
    print(f"  {r['bandwidth_pct']:>9.1f}%  {r['best_val_acc']:>11.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bws  = [r['bandwidth_pct'] for r in results_band]
accs = [r['best_val_acc']  for r in results_band]

axes[0].plot(bws, accs, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].axhline(baseline_val_acc, color='gray', linestyle='--',
                label=f'Baseline Conv1D = {baseline_val_acc:.1f}%')
axes[0].set_xlabel('Banda mantida (%)'); axes[0].set_ylabel('Melhor val_acc (%)')
axes[0].set_title('Bloco A — Acurácia vs. Banda absoluta')
axes[0].legend(); axes[0].grid(True, alpha=0.4)
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter())

for r in results_band:
    vals = [h[1] for h in r['history']]
    axes[1].plot(range(1, len(vals)+1), vals, label=f"{r['bandwidth_pct']:.0f}%")
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Val acc (%)')
axes[1].set_title('Bloco A — Curvas de validação')
axes[1].legend(title='Banda', bbox_to_anchor=(1.02,1), loc='upper left')
axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('bloco_A_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

with open('bloco_A_resultados.json', 'w') as fp:
    json.dump([{k:v for k,v in r.items() if k!='history'} for r in results_band],
              fp, indent=2)
print("📄 bloco_A_resultados.json salvo.")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCO B — BUSCA: PERCENTUAL DE POTÊNCIA ACUMULADA
#
# Passo B.0 — Estima PSD média sobre N_CALIB amostras do treino.
# Para cada power_pct em POWER_PCTS:
#   • Calcula quantos bins acumulam power_pct% da potência → band_fraction
#   • Treina CNN2D mantendo apenas esses bins (a partir de DC)
#   • Registra melhor val_acc
# ══════════════════════════════════════════════════════════════════════════════

# ── B.0 — PSD média ───────────────────────────────────────────────────────────

def _estimate_mean_psd(h5_path, label_key, indices, nperseg, noverlap, n_samples):
    rng  = np.random.default_rng(SEED)
    samp = rng.choice(indices, size=min(n_samples, len(indices)), replace=False)
    acc  = None
    with h5py.File(h5_path, 'r') as f:
        for idx in samp:
            x    = f['X'][int(idx)]
            x    = (x - x.mean()) / (x.std() + 1e-8)
            _, _, Sxx = scipy_spectrogram(
                x[:, 0], fs=1.0, nperseg=nperseg, noverlap=noverlap,
                window='hann', scaling='spectrum')
            pf = Sxx.mean(axis=1)
            acc = pf if acc is None else acc + pf
    return acc / len(samp)


def _power_band_fraction(mean_psd, power_pct):
    """Retorna (fração de bins, n_bins) que acumulam power_pct% da potência."""
    total  = mean_psd.sum()
    target = total * (power_pct / 100.0)
    cumsum = np.cumsum(mean_psd)
    n_keep = int(np.clip(np.searchsorted(cumsum, target) + 1,
                         1, len(mean_psd)))
    return n_keep / len(mean_psd), n_keep


print(f"\n{sep}")
print(f"  BLOCO B — Estimando PSD média ({N_CALIB} amostras)…")
print(sep)

mean_psd = _estimate_mean_psd(
    h5_local, h5_label_key, train_indices,
    NPERSEG, NOVERLAP, N_CALIB)

# Plot PSD média
fig, ax = plt.subplots(figsize=(7, 3))
freqs = np.linspace(0, 0.5, len(mean_psd))
ax.plot(freqs, 10 * np.log10(mean_psd + 1e-12), color='darkorange')
ax.set_xlabel('Frequência normalizada (0…fs/2)')
ax.set_ylabel('Potência média (dB)')
ax.set_title(f'Bloco B — PSD média do treino  ({MODELO_ALVO})')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('bloco_B_psd_media.png', dpi=150)
plt.show()

# Mapeia power_pct → band_fraction
power_to_frac = {}
print(f"\n  {'Potência (%)':>13}  {'Bins':>6}  {'Fração':>8}")
print("  " + "─"*32)
for pct in POWER_PCTS:
    frac, n_keep = _power_band_fraction(mean_psd, pct)
    power_to_frac[pct] = frac
    print(f"  {pct:>12.1f}%  {n_keep:>6d}  {frac:>7.4f}")

# ── Busca Bloco B ─────────────────────────────────────────────────────────────

results_power = []

print(f"\n{sep}")
print(f"  BLOCO B — Busca por percentual de potência")
print(f"  Candidatos : {POWER_PCTS}")
print(f"  Épocas/run : {EPOCHS_SEARCH}")
print(sep)

for pct in POWER_PCTS:
    band_frac = power_to_frac[pct]
    print(f"\n{'─'*65}")
    print(f"  ▶  power_pct={pct:.1f}%  →  band_fraction={band_frac:.4f}")
    print(f"{'─'*65}")
    _set_seeds(SEED)

    tr_ld = _make_loader_band(band_frac, H5PyDatasetPowerCrop,
                              h5_local, h5_label_key, train_indices,
                              SPEC_MODE, SPEC_HEIGHT, SPEC_WIDTH,
                              NPERSEG, NOVERLAP, BATCH_SIZE, SEED, shuffle=True)
    va_ld = _make_loader_band(band_frac, H5PyDatasetPowerCrop,
                              h5_local, h5_label_key, val_indices,
                              SPEC_MODE, SPEC_HEIGHT, SPEC_WIDTH,
                              NPERSEG, NOVERLAP, BATCH_SIZE, SEED, shuffle=False)

    model_B = CNN2D(num_classes, in_ch, (SPEC_HEIGHT, SPEC_WIDTH)).to(device)
    hist_B  = _train_one_config(model_B, tr_ld, va_ld, EPOCHS_SEARCH, LR, device)
    best_B  = max(h[1] for h in hist_B)

    results_power.append({
        "power_pct":     pct,
        "band_fraction": band_frac,
        "best_val_acc":  best_B,
        "history":       hist_B,
    })
    print(f"  ✅  power={pct:.1f}%  →  val_acc={best_B:.2f}%")

# ── Resultados Bloco B ────────────────────────────────────────────────────────

print(f"\n{sep}")
print("  RESULTADOS BLOCO B")
print(f"  {'Potência (%)':>13}  {'Banda (%)':>10}  {'Val Acc (%)':>12}")
print("  " + "─"*40)
for r in results_power:
    print(f"  {r['power_pct']:>12.1f}%  {r['band_fraction']*100:>9.1f}%  {r['best_val_acc']:>11.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico 1 — Acurácia × % potência
pcts_B  = [r['power_pct']    for r in results_power]
accs_B  = [r['best_val_acc'] for r in results_power]
axes[0].plot(pcts_B, accs_B, 's-', color='darkorange', linewidth=2, markersize=7)
axes[0].axhline(baseline_val_acc, color='gray', linestyle='--',
                label=f'Baseline Conv1D = {baseline_val_acc:.1f}%')
axes[0].set_xlabel('Potência acumulada mantida (%)')
axes[0].set_ylabel('Melhor val_acc (%)')
axes[0].set_title('Bloco B — Acurácia vs. Percentual de Potência')
axes[0].legend(); axes[0].grid(True, alpha=0.4)
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter())

# Gráfico 2 — Comparação A × B (eixo: fração de banda)
axes[1].plot([r['band_fraction']*100 for r in results_band],
             [r['best_val_acc']      for r in results_band],
             'o-', color='steelblue',  label='Bloco A — Banda absoluta')
axes[1].plot([r['band_fraction']*100 for r in results_power],
             [r['best_val_acc']      for r in results_power],
             's-', color='darkorange', label='Bloco B — % Potência')
axes[1].axhline(baseline_val_acc, color='gray', linestyle='--',
                label=f'Baseline = {baseline_val_acc:.1f}%')
axes[1].set_xlabel('Fração de banda mantida (%)')
axes[1].set_ylabel('Melhor val_acc (%)')
axes[1].set_title('Comparação A × B')
axes[1].legend(); axes[1].grid(True, alpha=0.4)
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig('bloco_AB_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

with open('bloco_B_resultados.json', 'w') as fp:
    json.dump([{k:v for k,v in r.items() if k!='history'} for r in results_power],
              fp, indent=2)
print("📄 bloco_B_resultados.json salvo.")

# ══════════════════════════════════════════════════════════════════════════════
# SUMÁRIO FINAL
# ══════════════════════════════════════════════════════════════════════════════

thr = 0.97 * baseline_val_acc

print(f"\n{sep}")
print(f"  SUMÁRIO FINAL  —  referência baseline = {baseline_val_acc:.2f}%  (limiar 97% = {thr:.2f}%)")
print(sep)

cands_A = [r for r in results_band  if r['best_val_acc'] >= thr]
cands_B = [r for r in results_power if r['best_val_acc'] >= thr]

print(f"\n  Bloco A — menor banda que preserva ≥97% do baseline:")
if cands_A:
    best = min(cands_A, key=lambda r: r['band_fraction'])
    print(f"    → band_fraction={best['band_fraction']:.4f}  "
          f"({best['bandwidth_pct']:.1f}%)  |  val_acc={best['best_val_acc']:.2f}%")
else:
    print("    → Nenhuma configuração atingiu o limiar.")

print(f"\n  Bloco B — menor % de potência que preserva ≥97% do baseline:")
if cands_B:
    best = min(cands_B, key=lambda r: r['power_pct'])
    print(f"    → power_pct={best['power_pct']:.1f}%  "
          f"(band_fraction={best['band_fraction']:.4f})  |  val_acc={best['best_val_acc']:.2f}%")
else:
    print("    → Nenhuma configuração atingiu o limiar.")

print(f"\n{sep}\n")

#Resnet


In [ ]:
# -*- coding: utf-8 -*-
"""
ResNet 1D para classificação de modulações de rádio (RadioML 2018)
==================================================================
Baseada em:  O'Shea, Roy & Clancy — "Over the Air Deep Learning Based
             Radio Signal Classification", arXiv:1712.04578

Arquitetura (Tabela IV do artigo, L=6 residual stacks):
  Input        →  [2, 1024]
  1×1 Conv Lin →  [32, 1024]    (projeção de canais sem não-linearidade)
  Res Stack ×6 →  [32,  512 → 256 → 128 → 64 → 32 → 16]
  FC/SeLU      →  128
  FC/SeLU      →  128
  FC/Softmax   →  num_classes

Cada Residual Stack contém 2 Residual Units + MaxPool1d(2).
Cada Residual Unit:  Conv1d → ReLU → Conv1d (linear) + skip connection.

Compatibilidade com o restante do notebook:
  • Mesma interface de __init__(num_classes) da classe CNN existente.
  • Entrada esperada: tensor [batch, 2, 1024]  (igual ao H5PyDataset).
  • Saída: logits [batch, num_classes]  (sem softmax — use CrossEntropyLoss).
  • Usa SELU + AlphaDropout nas camadas FC (igual ao artigo).
"""

import torch
import torch.nn as nn


# ══════════════════════════════════════════════════════════════════════════════
# Bloco residual básico (Figura 5 do artigo)
# ══════════════════════════════════════════════════════════════════════════════

class ResidualUnit(nn.Module):
    """
    Um Residual Unit conforme a Figura 5 do artigo:
        x  →  Conv1d/ReLU  →  Conv1d (linear)  →  (+x)  →  saída

    O skip connection é adicionado ANTES de qualquer ativação posterior,
    exatamente como descrito: "Conv ReLU | Conv Linear" com soma "+".
    """

    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2          # mantém o comprimento temporal

        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(channels)
        self.relu  = nn.ReLU(inplace=True)

        # Segunda conv é "linear" (sem ativação antes da soma)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity            # skip connection
        out = self.relu(out)            # ativação pós-soma
        return out


# ══════════════════════════════════════════════════════════════════════════════
# Residual Stack  (= 2 × ResidualUnit  +  MaxPool1d(2))
# ══════════════════════════════════════════════════════════════════════════════

class ResidualStack(nn.Module):
    """
    Residual Stack = 2 Residual Units seguidos de MaxPool1d(2).
    Reduz o comprimento temporal pela metade a cada stack.
    """

    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()
        self.unit1 = ResidualUnit(channels, kernel_size)
        self.unit2 = ResidualUnit(channels, kernel_size)
        self.pool  = nn.MaxPool1d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.unit1(x)
        x = self.unit2(x)
        x = self.pool(x)
        return x


# ══════════════════════════════════════════════════════════════════════════════
# ResNet principal
# ══════════════════════════════════════════════════════════════════════════════

class ResNet(nn.Module):
    """
    ResNet 1D para classificação de modulação de rádio.

    Parâmetros
    ----------
    num_classes : int
        Número de classes de modulação (ex.: 24 para o dataset "Difícil",
        6 para o agrupamento GROUP_MAP usado no notebook).
    n_stacks : int
        Número de Residual Stacks L (padrão 6, conforme Tabela IV).
        Com L=6 e entrada de 1024 amostras:
            comprimento após stacks = 1024 / 2^6 = 16.
    channels : int
        Número de canais internos (padrão 32, conforme Tabela IV).
    kernel_size : int
        Tamanho do kernel nas convoluções residuais (padrão 3).
    fc_hidden : int
        Tamanho das camadas FC ocultas (padrão 128, conforme Tabela IV).
    alpha_drop : float
        Taxa de AlphaDropout nas camadas FC (artigo usa regularização com
        AlphaDropout para redes auto-normalizadas com SELU).
    """

    def __init__(
        self,
        num_classes: int,
        n_stacks:    int   = 6,
        channels:    int   = 32,
        kernel_size: int   = 3,
        fc_hidden:   int   = 128,
        alpha_drop:  float = 0.2,
    ):
        super().__init__()

        # ── Projeção inicial: 1×1 Conv (sem bias, sem ativação) ───────────────
        # Equivale à camada "1x1 Conv Linear" do diagrama (Figura 5).
        # Mapeia 2 canais I/Q → 'channels' canais internos.
        self.input_proj = nn.Sequential(
            nn.Conv1d(2, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
        )

        # ── L Residual Stacks ─────────────────────────────────────────────────
        self.stacks = nn.Sequential(
            *[ResidualStack(channels, kernel_size) for _ in range(n_stacks)]
        )

        # ── Detecta automaticamente o tamanho do flatten ──────────────────────
        # (mesmo padrão usado na classe CNN do notebook)
        with torch.no_grad():
            dummy = torch.zeros(1, 2, 1024)
            dummy = self.input_proj(dummy)
            dummy = self.stacks(dummy)
            self.n_flatten = dummy.view(1, -1).size(1)

        print(f"[ResNet] Tamanho detectado para o Flatten: {self.n_flatten}")
        print(f"[ResNet] Parâmetros treináveis: "
              f"{sum(p.numel() for p in self.parameters() if p.requires_grad):,}")

        # ── Classificador FC com SELU + AlphaDropout ─────────────────────────
        # O artigo usa redes auto-normalizadas (SNN) com SELU e AlphaDropout
        # nas camadas totalmente conectadas (seção IV-C).
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(self.n_flatten, fc_hidden),
            nn.SELU(inplace=True),
            nn.AlphaDropout(alpha_drop),

            nn.Linear(fc_hidden, fc_hidden),
            nn.SELU(inplace=True),
            nn.AlphaDropout(alpha_drop),

            nn.Linear(fc_hidden, num_classes),   # logits (sem Softmax)
        )

        # ── Inicialização MRSA para camadas Linear (seção IV-C) ───────────────
        # Mean-Response Scaled Activation init recomendada para redes SELU.
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # Kaiming normal com modo 'fan_in', compatível com SELU/MRSA
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, 2, 1024]
        x = self.input_proj(x)   # [batch, channels, 1024]
        x = self.stacks(x)       # [batch, channels, 1024 / 2^L]
        x = self.classifier(x)   # [batch, num_classes]
        return x


# ══════════════════════════════════════════════════════════════════════════════
# Exemplo de uso — substitua CNN(...) por ResNet(...) no notebook
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Instanciação idêntica à CNN do notebook ────────────────────────────────
    # num_classes=24  →  dataset "Difícil" (todas as modulações)
    # num_classes=6   →  agrupamento GROUP_MAP do notebook (ASK/PSK/APSK/QAM/AM/FM)
    num_classes = 24
    model = ResNet(num_classes=num_classes)

    # ── Teste com um batch fictício ────────────────────────────────────────────
    dummy_batch = torch.zeros(8, 2, 1024)   # [batch=8, canais=2, amostras=1024]
    logits = model(dummy_batch)
    print(f"Shape da saída: {logits.shape}")  # esperado: [8, 24]

    # ── Uso no lugar da CNN ────────────────────────────────────────────────────
    #
    # Substitua no notebook:
    #   model = CNN(num_classes)
    # por:
    #   model = ResNet(num_classes)
    #
    # O restante (train_and_validate, criterion, optimizer, DataLoaders)
    # permanece idêntico, pois a interface de entrada/saída é a mesma.
    #
    # Configuração recomendada (artigo, seção V):
    #   criterion = nn.CrossEntropyLoss()
    #   optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    #   scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    #       optimizer, mode='max', patience=5, factor=0.5
    #   )

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║         TREINAMENTO — ResNet 1D  |  SNR ≥ 0 dB                             ║
# ║                                                                             ║
# ║  Pré-requisitos (células anteriores já executadas):                         ║
# ║    • kagglehub.dataset_download() → variável  path                          ║
# ║    • modulation_classes_path definido                                       ║
# ║    • Classes H5PyDataset e ResNet definidas                                 ║
# ║    • Funções split() / train_and_validate() definidas                       ║
# ║    • Drive montado:                                                         ║
# ║        from google.colab import drive                                       ║
# ║        drive.mount('/content/drive', force_remount=True)                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, shutil, random
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

MODELO_ALVO = "GROUP"
# Opções: "GROUP" | "ASK" | "PSK" | "APSK" | "QAM" | "AM" | "FM"

DESIRED_SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
# SNR ≥ 0 dB — faixa de alta confiabilidade de classificação (artigo seção V)

EPOCHS       = 120
BATCH_SIZE   = 64
LR           = 1e-3
SEED         = 42

# Hiperparâmetros da ResNet (Tabela IV do artigo)
RN_N_STACKS  = 6      # L = número de Residual Stacks (paper: 6)
RN_CHANNELS  = 32     # canais internos
RN_KERNEL    = 3      # kernel das convoluções residuais
RN_FC_HIDDEN = 128    # neurônios nas camadas FC
RN_ALPHA_DROP= 0.2    # AlphaDropout nas camadas FC

DRIVE_BASE      = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR       = "/content"
HDF5_BUILD_BATCH= 2048

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS (idêntico ao resto do notebook)
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES  = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}
ALL_MODELS   = ["GROUP", "ASK", "PSK", "APSK", "QAM", "AM", "FM"]

INPUT_FILE   = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE = modulation_classes_path

assert MODELO_ALVO in ALL_MODELS, f"MODELO_ALVO inválido: {MODELO_ALVO}"

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _seed_worker(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

# ══════════════════════════════════════════════════════════════════════════════
# DERIVAÇÕES — group_indices, num_classes, global_to_local, caminhos
# ══════════════════════════════════════════════════════════════════════════════

mod_classes = json.load(open(CLASSES_FILE))

if MODELO_ALVO == "GROUP":
    group_indices   = np.arange(len(mod_classes), dtype=np.int64)
    num_classes     = 6
    global_to_local = {int(g): GROUP_MAP[mod_classes[g]] for g in group_indices}
    h5_label_key    = "Y_grouped"
    _label_note     = "6 grupos (ASK/PSK/APSK/QAM/AM/FM)"
else:
    group_indices = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_NAMES[GROUP_MAP[m]] == MODELO_ALVO]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}
    h5_label_key    = "Y"
    _label_note     = f"{num_classes} classes intra-grupo {MODELO_ALVO}"

# Caminhos com prefixo "resnet_" para não colidir com checkpoints da CNN
snr_tag    = "_".join(str(s) for s in DESIRED_SNRS)
h5_name    = (f"GROUP_subset_snr_{snr_tag}.hdf5" if MODELO_ALVO == "GROUP"
              else f"{MODELO_ALVO}_subset_snr_{snr_tag}.hdf5")
h5_local   = os.path.join(LOCAL_DIR, h5_name)
h5_drive   = os.path.join(DRIVE_BASE, MODELO_ALVO, "subset.hdf5")

# Checkpoint e meta separados por prefixo "resnet_"
ckpt_drive  = os.path.join(DRIVE_BASE, MODELO_ALVO, "resnet_checkpoint.pth")
meta_drive  = os.path.join(DRIVE_BASE, MODELO_ALVO, "resnet_meta.json")
idx_drive   = lambda name: os.path.join(DRIVE_BASE, MODELO_ALVO, f"{name}.npy")

sep = "═" * 65
print(sep)
print(f"  Modelo  : ResNet — {MODELO_ALVO}  ({_label_note})")
print(f"  SNRs    : {DESIRED_SNRS}")
print(f"  Device  : {device}")
print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1 — VERIFICAR DRIVE
# ══════════════════════════════════════════════════════════════════════════════

assert os.path.exists("/content/drive/MyDrive"), (
    "\n\n❌ Google Drive não montado!\n"
    "   Execute antes:\n"
    "       from google.colab import drive\n"
    "       drive.mount('/content/drive', force_remount=True)\n"
)

has_h5   = os.path.exists(h5_drive)
has_ckpt = os.path.exists(ckpt_drive)
has_idx  = all(os.path.exists(idx_drive(n))
               for n in ("train_indices", "val_indices", "test_indices"))

print(f"\n📂 Drive  →  {os.path.join(DRIVE_BASE, MODELO_ALVO)}")
print(f"   HDF5              : {'✅ encontrado' if has_h5   else '❌ não existe — será criado'}")
print(f"   ResNet Checkpoint : {'✅ encontrado' if has_ckpt else '❌ não existe — treino do zero'}")
print(f"   Índices           : {'✅ encontrado' if has_idx  else '❌ não existe — será gerado'}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — OBTER HDF5 LOCAL (do Drive ou criar do zero)
# ══════════════════════════════════════════════════════════════════════════════

def _build_hdf5(input_file, output_file, target_snrs,
                group_indices, global_to_local, num_classes,
                modelo_id, batch_size=2048):
    """Filtra INPUT_FILE por SNR (e grupo se intra-grupo) e grava output_file."""
    snr_set = set(target_snrs)
    if os.path.exists(output_file):
        os.remove(output_file)

    print(f"\n  🔨 Construindo HDF5 para '{modelo_id}' (SNR ≥ 0 dB)…")

    original_indices = []
    with h5py.File(input_file, 'r') as src:
        N     = src['X'].shape[0]
        Z_ds  = src['Z']
        Y_ds  = src['Y']
        n_b   = int(np.ceil(N / batch_size))

        for i in range(n_b):
            s, e     = i * batch_size, min((i + 1) * batch_size, N)
            z_batch  = Z_ds[s:e, 0]
            y_batch  = np.argmax(Y_ds[s:e], axis=1)

            if modelo_id == "GROUP":
                mask = np.isin(z_batch, list(snr_set))
            else:
                mask = (np.isin(z_batch, list(snr_set)) &
                        np.isin(y_batch, group_indices))

            original_indices.extend((np.where(mask)[0] + s).tolist())
            if (i + 1) % 50 == 0 or i == n_b - 1:
                print(f"    Batch {i+1:>4}/{n_b}  |  amostras: {len(original_indices)}")

    original_indices = np.array(original_indices, dtype=np.int64)
    M = len(original_indices)
    if M == 0:
        raise ValueError("Nenhuma amostra encontrada com os filtros escolhidos.")

    NUM_GROUPS = 6
    lookup = np.array([GROUP_MAP[mod_classes[i]]
                       for i in range(len(mod_classes))], dtype=np.int64)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]

        with h5py.File(output_file, 'w') as out:
            out.attrs['snrs']          = target_snrs
            out.attrs['modelo_id']     = modelo_id
            out.attrs['num_classes']   = num_classes
            out.attrs['group_indices'] = group_indices.tolist()
            out.attrs['total_samples'] = M
            out.attrs['seed']          = SEED

            ds_X = out.create_dataset('X', shape=(M,)+x_shape, dtype=src['X'].dtype)
            ds_Z = out.create_dataset('Z', shape=(M,)+z_shape, dtype=src['Z'].dtype)

            if modelo_id == "GROUP":
                ds_Y = out.create_dataset('Y_grouped',
                                          shape=(M, NUM_GROUPS), dtype=np.float32)
            else:
                ds_Y = out.create_dataset('Y',
                                          shape=(M, num_classes), dtype=np.int32)

            n_b = int(np.ceil(M / batch_size))
            for i in range(n_b):
                s, e = i * batch_size, min((i + 1) * batch_size, M)
                idx  = original_indices[s:e]

                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]

                y_global = np.argmax(src['Y'][idx], axis=1)

                if modelo_id == "GROUP":
                    y_group = lookup[y_global]
                    yg = np.zeros((e - s, NUM_GROUPS), dtype=np.float32)
                    yg[np.arange(e - s), y_group] = 1.0
                    ds_Y[s:e] = yg
                else:
                    y_local = np.array([global_to_local[int(g)] for g in y_global])
                    yo = np.zeros((e - s, num_classes), dtype=np.int32)
                    yo[np.arange(e - s), y_local] = 1
                    ds_Y[s:e] = yo

                if (i + 1) % 20 == 0 or i == n_b - 1:
                    print(f"    Escrevendo batch {i+1:>4}/{n_b}")

    print(f"  ✅ HDF5 criado: {output_file}  ({M} amostras)")
    return output_file


if has_h5:
    if not os.path.exists(h5_local):
        print(f"\n📦 Copiando HDF5 do Drive → {h5_local}  (pode demorar)…")
        shutil.copy(h5_drive, h5_local)
        print("  ✅ Cópia concluída.")
    else:
        print(f"\n📦 HDF5 já disponível localmente: {h5_local}")
else:
    _build_hdf5(INPUT_FILE, h5_local, DESIRED_SNRS,
                group_indices, global_to_local, num_classes, MODELO_ALVO,
                batch_size=HDF5_BUILD_BATCH)

    print(f"\n  💾 Salvando HDF5 no Drive…")
    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    shutil.copy(h5_local, h5_drive)
    print(f"  ✅ {h5_drive}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — ÍNDICES TRAIN / VAL / TEST  (carrega do Drive ou gera)
# ══════════════════════════════════════════════════════════════════════════════

def _split_deterministic(h5_path, label_key, seed=42):
    """Split 60/20/20 estratificado por label."""
    with h5py.File(h5_path, 'r') as f:
        Y_all = f[label_key][:]
    y_all   = np.argmax(Y_all, axis=1)
    indices = np.arange(len(y_all))

    tv_idx, te_idx, tv_lbl, _ = train_test_split(
        indices, y_all, test_size=0.20, random_state=seed, stratify=y_all
    )
    tr_idx, va_idx, _, _ = train_test_split(
        tv_idx, tv_lbl, test_size=0.25, random_state=seed, stratify=tv_lbl
    )
    return np.sort(tr_idx), np.sort(va_idx), np.sort(te_idx)


if has_idx:
    print("\n📐 Carregando índices do Drive…")
    train_indices = np.load(idx_drive("train_indices"))
    val_indices   = np.load(idx_drive("val_indices"))
    test_indices  = np.load(idx_drive("test_indices"))
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")
else:
    print("\n📐 Gerando índices (split 60/20/20 determinístico)…")
    train_indices, val_indices, test_indices = _split_deterministic(
        h5_local, h5_label_key, seed=SEED
    )
    print(f"   train={len(train_indices)} | val={len(val_indices)} | test={len(test_indices)}")

    os.makedirs(os.path.join(DRIVE_BASE, MODELO_ALVO), exist_ok=True)
    np.save(idx_drive("train_indices"), train_indices)
    np.save(idx_drive("val_indices"),   val_indices)
    np.save(idx_drive("test_indices"),  test_indices)
    print("   ✅ Índices salvos no Drive.")

# Verificação anti-leakage
tr_s, va_s, te_s = set(train_indices), set(val_indices), set(test_indices)
assert not (tr_s & va_s), "❌ Sobreposição treino/val!"
assert not (tr_s & te_s), "❌ Sobreposição treino/test!"
assert not (va_s & te_s), "❌ Sobreposição val/test!"
print("   ✅ Sem sobreposição entre splits (anti-leakage OK)")

train_indices_sorted = train_indices
val_indices_sorted   = val_indices
test_indices_sorted  = test_indices
h5py_path            = h5_local

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

g = torch.Generator(); g.manual_seed(SEED)

train_dataset = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name=h5_label_key, data_Z_name='Z',
    indices=train_indices, formats=0
)
val_dataset = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name=h5_label_key, data_Z_name='Z',
    indices=val_indices, formats=0
)
test_dataset = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name=h5_label_key, data_Z_name='Z',
    indices=test_indices, formats=0
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=False, num_workers=0,
                          generator=g, worker_init_fn=_seed_worker)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, num_workers=0)

print(f"\n📊 DataLoaders prontos:")
print(f"   Treino     : {len(train_dataset):>8} amostras")
print(f"   Validação  : {len(val_dataset):>8} amostras")
print(f"   Teste      : {len(test_dataset):>8} amostras")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5 — INSTANCIAR RESNET  (carrega checkpoint se existir)
# ══════════════════════════════════════════════════════════════════════════════

model     = ResNet(
    num_classes = num_classes,
    n_stacks    = RN_N_STACKS,
    channels    = RN_CHANNELS,
    kernel_size = RN_KERNEL,
    fc_hidden   = RN_FC_HIDDEN,
    alpha_drop  = RN_ALPHA_DROP,
).to(device)

epoch_ini = 0
best_acc  = 0.0

if has_ckpt:
    print(f"\n🔄 Carregando checkpoint ResNet do Drive…")
    ckpt = torch.load(ckpt_drive, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    epoch_ini = ckpt.get("epoch",   0)
    best_acc  = ckpt.get("val_acc", 0.0)
    print(f"   Retomando da época {epoch_ini}  |  melhor val_acc = {best_acc:.2f}%")
else:
    print(f"\n⚙️  Nenhum checkpoint ResNet encontrado — treinando do zero.")

print(f"   ResNet  |  {num_classes} classes  |  "
      f"L={RN_N_STACKS} stacks  |  flatten={model.n_flatten}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 6 — OTIMIZADOR E SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-8
)

if has_ckpt:
    try:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        print("   ✅ Estado do otimizador restaurado.")
    except Exception:
        print("   ⚠️  Estado do otimizador não restaurado — usando valores padrão.")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 7 — TREINAR
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print(f"  🚀 Iniciando treino — ResNet '{MODELO_ALVO}'  |  "
      f"épocas: {EPOCHS}  |  LR: {LR}")
print(sep)

best_model_path = train_and_validate(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler  = scheduler,
    epochs     = EPOCHS,
    save_name  = f"resnet_{MODELO_ALVO}",
)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 8 — SALVAR NO DRIVE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print("  💾 Salvando sessão ResNet no Drive…")
print(sep)

save_dir = os.path.join(DRIVE_BASE, MODELO_ALVO)
os.makedirs(save_dir, exist_ok=True)

# Checkpoint
shutil.copy(best_model_path, ckpt_drive)
print(f"  ✅ Checkpoint  → {ckpt_drive}")

# Índices (re-salva para garantir)
np.save(idx_drive("train_indices"), train_indices_sorted)
np.save(idx_drive("val_indices"),   val_indices_sorted)
np.save(idx_drive("test_indices"),  test_indices_sorted)
print(f"  ✅ Índices     → {save_dir}/[train|val|test]_indices.npy")
print(f"  ✅ HDF5        → {h5_drive}  (já persistido)")

# meta.json
ckpt_final = torch.load(best_model_path, map_location="cpu")
meta = {
    "modelo_id":    f"ResNet_{MODELO_ALVO}",
    "seed":         SEED,
    "desired_snrs": DESIRED_SNRS,
    "num_classes":  num_classes,
    "group_indices": group_indices.tolist(),
    "resnet_config": {
        "n_stacks":   RN_N_STACKS,
        "channels":   RN_CHANNELS,
        "kernel_size":RN_KERNEL,
        "fc_hidden":  RN_FC_HIDDEN,
        "alpha_drop": RN_ALPHA_DROP,
    },
    "train_size":   int(len(train_indices_sorted)),
    "val_size":     int(len(val_indices_sorted)),
    "test_size":    int(len(test_indices_sorted)),
    "best_epoch":   int(ckpt_final.get("epoch",   -1)),
    "best_val_acc": float(ckpt_final.get("val_acc", -1)),
}
with open(meta_drive, "w") as fp:
    json.dump(meta, fp, indent=2, ensure_ascii=False)
print(f"  ✅ meta.json   → {meta_drive}")

print(f"\n{sep}")
print(f"  ✅ Sessão ResNet '{MODELO_ALVO}' completa e persistida no Drive.")
print(f"  📁 {save_dir}")
print(sep)

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   TREINAMENTO — ResNet 1D  |  24 CLASSES DIRETAS  |  SNR ≥ 0 dB            ║
# ║                                                                             ║
# ║  Classifica as 24 modulações do RadioML 2018 em um único modelo,            ║
# ║  sem estágio hierárquico de grupos.                                         ║
# ║                                                                             ║
# ║  Pré-requisitos (células anteriores já executadas):                         ║
# ║    • kagglehub.dataset_download() → variável  path                          ║
# ║    • modulation_classes_path definido                                       ║
# ║    • Classes H5PyDataset e ResNet definidas                                 ║
# ║    • Função train_and_validate() definida                                   ║
# ║    • Drive montado:                                                         ║
# ║        from google.colab import drive                                       ║
# ║        drive.mount('/content/drive', force_remount=True)                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, shutil, random
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

# SNR ≥ 0 dB — faixa de alta confiabilidade de classificação (artigo seção V-A)
DESIRED_SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]

EPOCHS     = 120
BATCH_SIZE = 64
LR         = 1e-3
SEED       = 42

# Hiperparâmetros da ResNet (Tabela IV do artigo — configuração L=6)
RN_N_STACKS   = 6      # número de Residual Stacks
RN_CHANNELS   = 32     # canais internos
RN_KERNEL     = 3      # kernel das convoluções residuais
RN_FC_HIDDEN  = 128    # neurônios das camadas FC
RN_ALPHA_DROP = 0.2    # AlphaDropout nas camadas FC

DRIVE_BASE       = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR        = "/content"
HDF5_BUILD_BATCH = 2048

# ── Identificador fixo para este experimento ──────────────────────────────────
EXPERIMENTO   = "24classes"
NUM_CLASSES   = 24      # todas as modulações do dataset

INPUT_FILE    = path + "/GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_FILE  = modulation_classes_path

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _seed_worker(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

# ══════════════════════════════════════════════════════════════════════════════
# CAMINHOS NO DRIVE
# ══════════════════════════════════════════════════════════════════════════════

mod_classes = json.load(open(CLASSES_FILE))   # lista de 24 strings, ordenada

snr_tag      = "_".join(str(s) for s in DESIRED_SNRS)
h5_name      = f"{EXPERIMENTO}_snr_{snr_tag}.hdf5"
h5_local     = os.path.join(LOCAL_DIR,  h5_name)
h5_drive     = os.path.join(DRIVE_BASE, EXPERIMENTO, "subset.hdf5")
ckpt_drive   = os.path.join(DRIVE_BASE, EXPERIMENTO, "resnet_checkpoint.pth")
meta_drive   = os.path.join(DRIVE_BASE, EXPERIMENTO, "resnet_meta.json")
idx_drive    = lambda n: os.path.join(DRIVE_BASE, EXPERIMENTO, f"{n}.npy")

sep = "═" * 65
print(sep)
print(f"  Experimento : {EXPERIMENTO}  —  {NUM_CLASSES} modulações diretas")
print(f"  SNRs        : {DESIRED_SNRS}")
print(f"  Device      : {device}")
print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 1 — VERIFICAR DRIVE
# ══════════════════════════════════════════════════════════════════════════════

assert os.path.exists("/content/drive/MyDrive"), (
    "\n\n❌ Google Drive não montado!\n"
    "   Execute antes:\n"
    "       from google.colab import drive\n"
    "       drive.mount('/content/drive', force_remount=True)\n"
)

has_h5   = os.path.exists(h5_drive)
has_ckpt = os.path.exists(ckpt_drive)
has_idx  = all(os.path.exists(idx_drive(n))
               for n in ("train_indices", "val_indices", "test_indices"))

print(f"\n📂 Drive  →  {os.path.join(DRIVE_BASE, EXPERIMENTO)}")
print(f"   HDF5       : {'✅ encontrado' if has_h5   else '❌ não existe — será criado'}")
print(f"   Checkpoint : {'✅ encontrado' if has_ckpt else '❌ não existe — treino do zero'}")
print(f"   Índices    : {'✅ encontrado' if has_idx  else '❌ não existe — será gerado'}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — OBTER HDF5 LOCAL
# Filtra o dataset original mantendo apenas amostras com SNR ≥ 0 dB.
# O label gravado é o índice global 0-23 da modulação (one-hot de 24 posições).
# ══════════════════════════════════════════════════════════════════════════════

def _build_hdf5_24classes(input_file, output_file, target_snrs, batch_size=2048):
    """
    Filtra input_file pelos SNRs desejados e grava um novo HDF5 com:
      X          : [N, 1024, 2]   — amostras I/Q
      Y          : [N, 24]        — one-hot das 24 modulações (índice global)
      Z          : [N, 1]         — SNR de cada amostra
    """
    snr_set = set(target_snrs)
    if os.path.exists(output_file):
        os.remove(output_file)

    print(f"\n  🔨 Construindo HDF5 — 24 classes  (SNR ≥ 0 dB)…")

    # ── Passo 1: identificar índices ─────────────────────────────────────────
    original_indices = []
    with h5py.File(input_file, 'r') as src:
        N   = src['X'].shape[0]
        Z_d = src['Z']
        n_b = int(np.ceil(N / batch_size))

        for i in range(n_b):
            s, e    = i * batch_size, min((i + 1) * batch_size, N)
            z_batch = Z_d[s:e, 0]
            mask    = np.isin(z_batch, list(snr_set))
            original_indices.extend((np.where(mask)[0] + s).tolist())

            if (i + 1) % 50 == 0 or i == n_b - 1:
                print(f"    Varredura  batch {i+1:>4}/{n_b}  "
                      f"|  selecionados: {len(original_indices)}")

    original_indices = np.array(original_indices, dtype=np.int64)
    M = len(original_indices)
    if M == 0:
        raise ValueError("Nenhuma amostra encontrada para os SNRs escolhidos.")
    print(f"    Total selecionado: {M} amostras")

    # ── Passo 2: gravar novo HDF5 ─────────────────────────────────────────────
    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]   # (1024, 2)
        z_shape = src['Z'].shape[1:]   # (1,)

        with h5py.File(output_file, 'w') as out:
            out.attrs['snrs']          = target_snrs
            out.attrs['num_classes']   = 24
            out.attrs['total_samples'] = M
            out.attrs['seed']          = SEED

            ds_X = out.create_dataset('X', shape=(M,) + x_shape, dtype=src['X'].dtype)
            ds_Y = out.create_dataset('Y', shape=(M, 24),         dtype=np.float32)
            ds_Z = out.create_dataset('Z', shape=(M,) + z_shape,  dtype=src['Z'].dtype)

            n_b = int(np.ceil(M / batch_size))
            for i in range(n_b):
                s, e = i * batch_size, min((i + 1) * batch_size, M)
                idx  = original_indices[s:e]

                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]
                ds_Y[s:e] = src['Y'][idx]   # one-hot 24 já no dataset original

                if (i + 1) % 20 == 0 or i == n_b - 1:
                    print(f"    Escrevendo batch {i+1:>4}/{n_b}")

    print(f"  ✅ HDF5 criado: {output_file}  ({M} amostras)")


if has_h5:
    if not os.path.exists(h5_local):
        print(f"\n📦 Copiando HDF5 do Drive → {h5_local}…")
        shutil.copy(h5_drive, h5_local)
        print("  ✅ Cópia concluída.")
    else:
        print(f"\n📦 HDF5 já disponível localmente: {h5_local}")
else:
    _build_hdf5_24classes(INPUT_FILE, h5_local, DESIRED_SNRS,
                          batch_size=HDF5_BUILD_BATCH)
    os.makedirs(os.path.join(DRIVE_BASE, EXPERIMENTO), exist_ok=True)
    print(f"\n  💾 Salvando HDF5 no Drive…")
    shutil.copy(h5_local, h5_drive)
    print(f"  ✅ {h5_drive}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — ÍNDICES TRAIN / VAL / TEST  (60 / 20 / 20)
# Estratificado pelo índice da modulação (label 0-23).
# ══════════════════════════════════════════════════════════════════════════════

def _split_24classes(h5_path, seed=42):
    with h5py.File(h5_path, 'r') as f:
        y_all = np.argmax(f['Y'][:], axis=1)   # 0-23
    indices = np.arange(len(y_all))

    tv_idx, te_idx, tv_lbl, _ = train_test_split(
        indices, y_all, test_size=0.20, random_state=seed, stratify=y_all
    )
    tr_idx, va_idx, _, _ = train_test_split(
        tv_idx, tv_lbl, test_size=0.25, random_state=seed, stratify=tv_lbl
    )
    return np.sort(tr_idx), np.sort(va_idx), np.sort(te_idx)


if has_idx:
    print("\n📐 Carregando índices do Drive…")
    train_indices = np.load(idx_drive("train_indices"))
    val_indices   = np.load(idx_drive("val_indices"))
    test_indices  = np.load(idx_drive("test_indices"))
else:
    print("\n📐 Gerando índices (split 60/20/20 estratificado)…")
    train_indices, val_indices, test_indices = _split_24classes(h5_local, seed=SEED)
    os.makedirs(os.path.join(DRIVE_BASE, EXPERIMENTO), exist_ok=True)
    np.save(idx_drive("train_indices"), train_indices)
    np.save(idx_drive("val_indices"),   val_indices)
    np.save(idx_drive("test_indices"),  test_indices)
    print("   ✅ Índices salvos no Drive.")

print(f"   train={len(train_indices):>7} | "
      f"val={len(val_indices):>7} | "
      f"test={len(test_indices):>7}")

# Verificação anti-leakage
tr_s, va_s, te_s = set(train_indices), set(val_indices), set(test_indices)
assert not (tr_s & va_s), "❌ Sobreposição treino/val!"
assert not (tr_s & te_s), "❌ Sobreposição treino/test!"
assert not (va_s & te_s), "❌ Sobreposição val/test!"
print("   ✅ Sem sobreposição entre splits (anti-leakage OK)")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

g = torch.Generator(); g.manual_seed(SEED)

train_dataset_24 = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name='Y',      data_Z_name='Z',
    indices=train_indices, formats=0
)
val_dataset_24 = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name='Y',      data_Z_name='Z',
    indices=val_indices,   formats=0
)
test_dataset_24 = H5PyDataset(
    h5_filepath=h5_local, data_X_name='X',
    data_Y_name='Y',      data_Z_name='Z',
    indices=test_indices,  formats=0
)

train_loader_24 = DataLoader(train_dataset_24, batch_size=BATCH_SIZE, shuffle=True,
                             drop_last=False, num_workers=0,
                             generator=g, worker_init_fn=_seed_worker)
val_loader_24   = DataLoader(val_dataset_24,   batch_size=BATCH_SIZE, shuffle=False,
                             drop_last=False, num_workers=0)
test_loader_24  = DataLoader(test_dataset_24,  batch_size=BATCH_SIZE, shuffle=False,
                             drop_last=False, num_workers=0)

print(f"\n📊 DataLoaders prontos:")
print(f"   Treino     : {len(train_dataset_24):>8} amostras")
print(f"   Validação  : {len(val_dataset_24):>8} amostras")
print(f"   Teste      : {len(test_dataset_24):>8} amostras")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5 — INSTANCIAR RESNET  (carrega checkpoint se existir)
# ══════════════════════════════════════════════════════════════════════════════

model_24 = ResNet(
    num_classes = NUM_CLASSES,
    n_stacks    = RN_N_STACKS,
    channels    = RN_CHANNELS,
    kernel_size = RN_KERNEL,
    fc_hidden   = RN_FC_HIDDEN,
    alpha_drop  = RN_ALPHA_DROP,
).to(device)

epoch_ini = 0
best_acc  = 0.0

if has_ckpt:
    print(f"\n🔄 Carregando checkpoint do Drive…")
    ckpt = torch.load(ckpt_drive, map_location=device)
    model_24.load_state_dict(ckpt["model_state_dict"])
    epoch_ini = ckpt.get("epoch",   0)
    best_acc  = ckpt.get("val_acc", 0.0)
    print(f"   Retomando da época {epoch_ini}  |  melhor val_acc = {best_acc:.2f}%")
else:
    print(f"\n⚙️  Nenhum checkpoint encontrado — treinando do zero.")

print(f"   ResNet  |  {NUM_CLASSES} classes  |  "
      f"L={RN_N_STACKS} stacks  |  flatten={model_24.n_flatten}")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 6 — OTIMIZADOR E SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════

criterion_24 = nn.CrossEntropyLoss()
optimizer_24 = optim.Adam(model_24.parameters(), lr=LR)
scheduler_24 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_24, mode='max', factor=0.5, patience=5, min_lr=1e-8
)

if has_ckpt:
    try:
        optimizer_24.load_state_dict(ckpt["optimizer_state_dict"])
        print("   ✅ Estado do otimizador restaurado.")
    except Exception:
        print("   ⚠️  Estado do otimizador não restaurado — usando valores padrão.")

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 7 — TREINAR
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print(f"  🚀 Iniciando treino — ResNet 24 classes  |  "
      f"épocas: {EPOCHS}  |  LR: {LR}")
print(sep)

best_model_path_24 = train_and_validate(
    model_24,
    train_loader_24,
    val_loader_24,
    criterion_24,
    optimizer_24,
    scheduler  = scheduler_24,
    epochs     = EPOCHS,
    save_name  = f"resnet_{EXPERIMENTO}",
)

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 8 — AVALIAÇÃO NO CONJUNTO DE TESTE
# Calcula acurácia global, acurácia por SNR e matriz de confusão.
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print("  📐 Avaliação no conjunto de teste…")
print(sep)

# Carrega o melhor modelo salvo
ckpt_best = torch.load(best_model_path_24, map_location=device)
model_24.load_state_dict(ckpt_best["model_state_dict"])
model_24.eval()

y_true_test = []
y_pred_test = []
snr_test    = []

with h5py.File(h5_local, 'r') as f:
    Z_all = f['Z'][:, 0]

    with torch.no_grad():
        for inputs, labels in test_loader_24:
            inputs = inputs.to(device)
            logits = model_24(inputs)
            preds  = logits.argmax(dim=1).cpu().numpy()
            y_pred_test.append(preds)
            y_true_test.append(labels.numpy())

    # SNR de cada amostra do test set (para acurácia por SNR)
    snr_test = Z_all[test_indices]

y_true_test = np.concatenate(y_true_test)
y_pred_test = np.concatenate(y_pred_test)

test_acc = 100.0 * (y_true_test == y_pred_test).mean()
print(f"\n  Acurácia global no teste : {test_acc:.2f}%")

# ── Acurácia por SNR ──────────────────────────────────────────────────────────
snrs_uniq = sorted(np.unique(snr_test).tolist())
acc_por_snr = {}
for snr in snrs_uniq:
    mask = snr_test == snr
    if mask.sum() == 0:
        continue
    acc_por_snr[snr] = 100.0 * (y_true_test[mask] == y_pred_test[mask]).mean()

print(f"\n  {'SNR (dB)':>10}  {'Acurácia (%)':>13}")
print("  " + "─" * 26)
for snr, acc in acc_por_snr.items():
    print(f"  {snr:>10.0f}  {acc:>12.2f}%")

# ── Figura 1 — Acurácia por SNR ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(acc_por_snr.keys()), list(acc_por_snr.values()),
        'o-', color='steelblue', linewidth=2, markersize=6)
ax.set_xlabel("SNR — $E_s/N_0$ (dB)")
ax.set_ylabel("Acurácia (%)")
ax.set_title(f"ResNet 24 classes — Acurácia por SNR  (teste)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("resnet_24classes_acc_por_snr.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figura 2 — Matriz de confusão ─────────────────────────────────────────────
cm = confusion_matrix(y_true_test, y_pred_test, labels=list(range(NUM_CLASSES)),
                      normalize='true')

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=mod_classes)
disp.plot(ax=ax, colorbar=True, xticks_rotation=90, values_format=".2f",
          cmap="Blues")
ax.set_title(f"ResNet 24 classes — Matriz de Confusão  "
             f"(teste, SNR ≥ 0 dB,  acc={test_acc:.1f}%)",
             fontsize=13)
plt.tight_layout()
plt.savefig("resnet_24classes_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 9 — SALVAR NO DRIVE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{sep}")
print("  💾 Salvando sessão no Drive…")
print(sep)

save_dir = os.path.join(DRIVE_BASE, EXPERIMENTO)
os.makedirs(save_dir, exist_ok=True)

shutil.copy(best_model_path_24, ckpt_drive)
print(f"  ✅ Checkpoint  → {ckpt_drive}")

np.save(idx_drive("train_indices"), train_indices)
np.save(idx_drive("val_indices"),   val_indices)
np.save(idx_drive("test_indices"),  test_indices)
print(f"  ✅ Índices     → {save_dir}/[train|val|test]_indices.npy")
print(f"  ✅ HDF5        → {h5_drive}  (já persistido)")

# Gráficos
for fname in ("resnet_24classes_acc_por_snr.png",
              "resnet_24classes_confusion_matrix.png"):
    shutil.copy(fname, os.path.join(save_dir, fname))
print(f"  ✅ Gráficos    → {save_dir}/resnet_24classes_*.png")

# meta.json
ckpt_final = torch.load(best_model_path_24, map_location="cpu")
meta = {
    "experimento":   EXPERIMENTO,
    "seed":          SEED,
    "desired_snrs":  DESIRED_SNRS,
    "num_classes":   NUM_CLASSES,
    "class_names":   mod_classes,
    "resnet_config": {
        "n_stacks":    RN_N_STACKS,
        "channels":    RN_CHANNELS,
        "kernel_size": RN_KERNEL,
        "fc_hidden":   RN_FC_HIDDEN,
        "alpha_drop":  RN_ALPHA_DROP,
    },
    "train_size":    int(len(train_indices)),
    "val_size":      int(len(val_indices)),
    "test_size":     int(len(test_indices)),
    "best_epoch":    int(ckpt_final.get("epoch",   -1)),
    "best_val_acc":  float(ckpt_final.get("val_acc", -1)),
    "test_acc":      float(test_acc),
    "acc_por_snr":   {str(k): float(v) for k, v in acc_por_snr.items()},
}
with open(meta_drive, "w") as fp:
    json.dump(meta, fp, indent=2, ensure_ascii=False)
print(f"  ✅ meta.json   → {meta_drive}")

print(f"\n{sep}")
print(f"  ✅ Experimento '{EXPERIMENTO}' completo e persistido no Drive.")
print(f"  📁 {save_dir}")
print(f"  🏆 Melhor val_acc  : {ckpt_final.get('val_acc', -1):.2f}%")
print(f"  🏆 Test  acc       : {test_acc:.2f}%")
print(sep)

#HIPERPARAMETROS GLOBAL


In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   BUSCA DE ARQUITETURA — K-Fold CV com retomada automática entre grupos     ║
# ║                                                                              ║
# ║  Comportamento:                                                              ║
# ║    • Detecta salvamentos no Drive e continua de onde parou                  ║
# ║    • Ao terminar um grupo segue automaticamente para o próximo              ║
# ║    • LR cai pela metade a cada 2 épocas sem melhora                        ║
# ║    • Early stopping após 10 épocas sem melhora                             ║
# ║    • Divisão de dados invariante entre sessões (seed fixo + salvo no Drive) ║
# ║    • Labels remapeados automaticamente para [0, num_classes-1]              ║
# ║                                                                              ║
# ║  Pré-requisitos (células anteriores já executadas):                         ║
# ║    • kagglehub.dataset_download() → variável path                           ║
# ║    • modulation_classes_path definido                                       ║
# ║    • Classes H5PyDataset, CNN, FlexCNN definidas                            ║
# ║    • Função split() definida                                                 ║
# ║    • Drive montado                                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, copy, random, shutil
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold, train_test_split
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

# Ordem dos grupos a processar — o script segue esta sequência automaticamente
# Grupos já completos são pulados; retoma no grupo interrompido
GRUPOS_ALVO = ["ASK","PSK","APSK","QAM","AM","FM"]
# FM tem 1 classe — não precisa de busca de arquitetura

DESIRED_SNRS     = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]

K_FOLDS          = 5
EPOCHS_PER_FOLD  = 120      # máximo de épocas por fold
LR_PATIENCE      = 5       # épocas sem melhora → LR cai pela metade
EARLY_STOP       = 15      # épocas sem melhora → encerra o fold
LR_MIN           = 1e-9    # LR mínimo (não cai abaixo disso)
SEED             = 42
DROPOUT          = 0.5
LEARNING_RATE    = 1e-3
BATCH_SIZE       = 64
CLASSIFIER_HEAD  = [512]

DRIVE_BASE       = "/content/drive/MyDrive/radioml_sessions"
LOCAL_DIR        = "/content"
HDF5_BUILD_BATCH = 2048

ARCHITECTURES = [
    {
        "label": "2L_32-64",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
        ],
    },
    {
        "label": "2L_64-128",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
        ],
    },
    {
        "label": "3L_32-64-128",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
            {"out_channels": 128, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_64-128-256",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
            {"out_channels": 256, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_128-256-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 7, "pool": True},
            {"out_channels": 256, "kernel_size": 5, "pool": True},
            {"out_channels": 512, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "4L_32-64-128-256",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": True},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_64-128-256-512",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 128, "kernel_size": 7,  "pool": True},
            {"out_channels": 256, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_128-256-512-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 11, "pool": True},
            {"out_channels": 256, "kernel_size": 7,  "pool": True},
            {"out_channels": 512, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "5L_32-64-128-256-512",
        "arch": [
            {"out_channels": 32,   "kernel_size": 11, "pool": True},
            {"out_channels": 64,   "kernel_size": 7,  "pool": True},
            {"out_channels": 128,  "kernel_size": 5,  "pool": True},
            {"out_channels": 256,  "kernel_size": 3,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "5L_64-128-256-512-1024",
        "arch": [
            {"out_channels": 64,   "kernel_size": 11, "pool": True},
            {"out_channels": 128,  "kernel_size": 7,  "pool": True},
            {"out_channels": 256,  "kernel_size": 5,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            {"out_channels": 1024, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "6L_64-64-128-128-256-512_mixpool",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "6L_32-64-64-128-256-512_mixpool",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 64,  "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
]

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}

sep = "═" * 65

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ══════════════════════════════════════════════════════════════════════════════
# FlexCNN
# ══════════════════════════════════════════════════════════════════════════════

class FlexCNN(nn.Module):
    def __init__(self, num_classes, arch, classifier=None,
                 dropout=0.5, in_channels=2, input_length=1024):
        super().__init__()
        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels
        for block in arch:
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=ks // 2),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))
            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, input_length)
            n_flat = self.features(dummy).view(1, -1).size(1)

        head = []
        prev = n_flat
        for units in classifier:
            head += [nn.Linear(prev, units), nn.ReLU(inplace=True),
                     nn.Dropout(dropout)]
            prev  = units
        head.append(nn.Linear(prev, num_classes))
        self.classifier = nn.Sequential(*head)

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ══════════════════════════════════════════════════════════════════════════════

def _drive_dir(modelo):
    """Retorna e cria o diretório do modelo no Drive."""
    d = os.path.join(DRIVE_BASE, modelo)
    os.makedirs(d, exist_ok=True)
    return d


def _idx_path(modelo, name):
    return os.path.join(_drive_dir(modelo), f"{name}.npy")


def _h5_local(modelo):
    snr_tag = '_'.join(str(s) for s in DESIRED_SNRS)
    return os.path.join(LOCAL_DIR, f"{modelo}_subset_snr_{snr_tag}.hdf5")


def _h5_drive(modelo):
    snr_tag = '_'.join(str(s) for s in DESIRED_SNRS)
    return os.path.join(_drive_dir(modelo),
                        f"{modelo}_subset_snr_{snr_tag}.hdf5")


def _results_path(modelo):
    return os.path.join(_drive_dir(modelo), "kfold_fold_results.json")


def _summary_path(modelo):
    return os.path.join(_drive_dir(modelo), "kfold_summary.json")


def _folds_path(modelo):
    return os.path.join(_drive_dir(modelo),
                        f"kfold_folds_k{K_FOLDS}.json")


# ══════════════════════════════════════════════════════════════════════════════
# PASSO 1 — HDF5 FILTRADO COM LABELS REMAPEADOS
# ══════════════════════════════════════════════════════════════════════════════

def ensure_hdf5(modelo, mod_classes):
    """
    Garante que o HDF5 filtrado existe localmente com labels remapeados.
    Tenta carregar do Drive; se não existir, constrói do zero.
    Sempre verifica e corrige labels globais → locais.
    """
    h5_loc  = _h5_local(modelo)
    h5_drv  = _h5_drive(modelo)
    snr_set = set(DESIRED_SNRS)

    target_id       = {v: k for k, v in GROUP_NAMES.items()}[modelo]
    group_indices   = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_MAP[m] == target_id]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}

    # ── Carrega do Drive se disponível ───────────────────────────────────────
    if not os.path.exists(h5_loc):
        if os.path.exists(h5_drv):
            print(f"  📥 Copiando HDF5 do Drive → {h5_loc}")
            shutil.copy(h5_drv, h5_loc)
        else:
            # Constrói do zero
            print(f"  🔨 Construindo HDF5 para '{modelo}'...")
            _build_hdf5(h5_loc, mod_classes, group_indices,
                        global_to_local, num_classes, snr_set)
            # Persiste no Drive
            shutil.copy(h5_loc, h5_drv)
            print(f"  💾 HDF5 salvo no Drive: {h5_drv}")

    # ── Verifica/corrige labels ───────────────────────────────────────────────
    _fix_labels(h5_loc, global_to_local, num_classes)

    return h5_loc, num_classes, group_indices, global_to_local


def _build_hdf5(h5_out, mod_classes, group_indices,
                global_to_local, num_classes, snr_set):
    """Filtra INPUT_FILE por grupo e SNR, grava h5_out com labels locais."""
    input_file = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'
    selected   = []

    with h5py.File(input_file, 'r') as src:
        N    = src['X'].shape[0]
        n_b  = int(np.ceil(N / HDF5_BUILD_BATCH))
        for i in range(n_b):
            s, e    = i * HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, N)
            z_batch = src['Z'][s:e, 0]
            y_batch = np.argmax(src['Y'][s:e], axis=1)
            mask    = (np.isin(z_batch, list(snr_set)) &
                       np.isin(y_batch, group_indices))
            selected.extend((np.where(mask)[0] + s).tolist())
            if (i+1) % 20 == 0 or i == n_b-1:
                print(f"    Batch {i+1}/{n_b} | selecionados: {len(selected)}")

    selected = np.array(selected, dtype=np.int64)
    M        = len(selected)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]

        with h5py.File(h5_out, 'w') as out:
            out.attrs['modelo_id']      = str(group_indices[0])  # placeholder
            out.attrs['num_classes']    = num_classes
            out.attrs['group_indices']  = group_indices.tolist()
            out.attrs['snrs']           = DESIRED_SNRS
            out.attrs['total_samples']  = M
            out.attrs['labels_remapped']= True

            ds_X = out.create_dataset('X', shape=(M,)+x_shape,
                                      dtype=src['X'].dtype)
            ds_Z = out.create_dataset('Z', shape=(M,)+z_shape,
                                      dtype=src['Z'].dtype)
            ds_Y = out.create_dataset('Y', shape=(M, num_classes),
                                      dtype=np.int32)

            n_b2 = int(np.ceil(M / HDF5_BUILD_BATCH))
            for i in range(n_b2):
                s, e  = i*HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, M)
                idx   = selected[s:e]
                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]

                y_global   = np.argmax(src['Y'][idx], axis=1)
                y_local    = np.array([global_to_local[int(g)]
                                       for g in y_global])
                y_onehot   = np.zeros((len(y_local), num_classes), dtype=np.int32)
                y_onehot[np.arange(len(y_local)), y_local] = 1
                ds_Y[s:e]  = y_onehot

                if (i+1) % 10 == 0 or i == n_b2-1:
                    print(f"    Gravando batch {i+1}/{n_b2}")

    print(f"  ✅ HDF5 construído: {M} amostras, {num_classes} classes")


def _fix_labels(h5_path, global_to_local, num_classes):

    # Abre somente para leitura
    with h5py.File(h5_path, "r") as f:

        if f.attrs.get("labels_remapped", False):
            return

        y_sample = np.argmax(f["Y"][:100], axis=1)

    # <-- o arquivo já foi fechado aqui

    if y_sample.max() < num_classes:
        with h5py.File(h5_path, "a") as f:
            f.attrs["labels_remapped"] = True
        return

    print("Corrigindo labels...")

    with h5py.File(h5_path, "r") as f:
        y_global = np.argmax(f["Y"][:], axis=1)
        N = len(y_global)

    y_local = np.array([global_to_local[int(g)] for g in y_global])

    y_onehot = np.zeros((N, num_classes), dtype=np.int32)
    y_onehot[np.arange(N), y_local] = 1

    with h5py.File(h5_path, "a") as f:
        del f["Y"]
        f.create_dataset("Y", data=y_onehot)
        f.attrs["labels_remapped"] = True
        f.attrs["num_classes"] = num_classes
# ══════════════════════════════════════════════════════════════════════════════
# PASSO 2 — SPLIT DETERMINÍSTICO
# ══════════════════════════════════════════════════════════════════════════════

def ensure_split(modelo, h5_loc):
    """Carrega split do Drive ou gera e salva."""
    tp = _idx_path(modelo, "train_indices")
    vp = _idx_path(modelo, "val_indices")
    ep = _idx_path(modelo, "test_indices")

    if os.path.exists(tp) and os.path.exists(vp) and os.path.exists(ep):
        train_idx = np.load(tp)
        val_idx   = np.load(vp)
        test_idx  = np.load(ep)
        print(f"  ✅ Split carregado do Drive  "
              f"(treino={len(train_idx):,}  "
              f"val={len(val_idx):,}  "
              f"teste={len(test_idx):,})")
        return train_idx, val_idx, test_idx

    print(f"  Gerando split (seed={SEED})...")
    with h5py.File(h5_loc, 'r') as f:
        M       = f['X'].shape[0]
        y_lbl   = np.argmax(f['Y'][:], axis=1)

    indices = np.arange(M)
    tv_idx, te_idx, tv_y, _ = train_test_split(
        indices, y_lbl, test_size=0.2,
        random_state=SEED, stratify=y_lbl
    )
    tr_idx, vl_idx, _, _ = train_test_split(
        tv_idx, tv_y, test_size=0.25,
        random_state=SEED, stratify=tv_y
    )

    train_idx = np.sort(tr_idx)
    val_idx   = np.sort(vl_idx)
    test_idx  = np.sort(te_idx)

    np.save(tp, train_idx)
    np.save(vp, val_idx)
    np.save(ep, test_idx)
    print(f"  ✅ Split salvo no Drive  "
          f"(treino={len(train_idx):,}  "
          f"val={len(val_idx):,}  "
          f"teste={len(test_idx):,})")
    return train_idx, val_idx, test_idx

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 3 — FOLDS DETERMINÍSTICOS
# ══════════════════════════════════════════════════════════════════════════════

def ensure_folds(modelo, h5_loc, train_idx):
    """Carrega folds do Drive ou gera e salva."""
    fp = _folds_path(modelo)

    if os.path.exists(fp):
        with open(fp) as f:
            data = json.load(f)
        assert data["k_folds"]  == K_FOLDS,          "k_folds diverge!"
        assert data["seed"]     == SEED,              "seed diverge!"
        assert data["n_train"]  == len(train_idx),    "tamanho de treino mudou!"
        splits = [(np.array(d["train_idx"]),
                   np.array(d["val_idx"]))
                  for d in data["folds"]]
        print(f"  ✅ Folds carregados do Drive: {fp}")
        return splits

    print(f"  Gerando {K_FOLDS} folds (seed={SEED})...")
    with h5py.File(h5_loc, 'r') as f:
        y_train = np.argmax(f['Y'][train_idx], axis=1)

    skf    = StratifiedKFold(n_splits=K_FOLDS, shuffle=True,
                              random_state=SEED)
    splits = list(skf.split(np.arange(len(train_idx)), y_train))

    data = {
        "modelo": modelo, "k_folds": K_FOLDS,
        "seed": SEED, "n_train": len(train_idx),
        "folds": [
            {"fold": i, "n_train": len(tr), "n_val": len(vl),
             "train_idx": tr.tolist(), "val_idx": vl.tolist()}
            for i, (tr, vl) in enumerate(splits)
        ]
    }
    with open(fp, "w") as f:
        json.dump(data, f)

    print(f"  ✅ Folds salvos: {fp}")
    for i, (tr, vl) in enumerate(splits):
        print(f"     Fold {i+1}: treino={len(tr):,}  val={len(vl):,}")
    return splits

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 4 — CARREGAR / SALVAR RESULTADOS
# ══════════════════════════════════════════════════════════════════════════════

def load_results(modelo):
    rp = _results_path(modelo)
    if os.path.exists(rp):
        with open(rp) as f:
            data = json.load(f)
        done = {(r["label"], r["fold"]) for r in data}
        print(f"  ✅ {len(data)} resultados anteriores  "
              f"({len(done)} combinações concluídas)")
        return data, done
    return [], set()


def save_fold_result(modelo, result, all_results):
    all_results.append(result)
    with open(_results_path(modelo), "w") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)


def update_summary(modelo, all_results, num_classes):
    from collections import defaultdict
    stats = defaultdict(list)
    meta  = {}
    for r in all_results:
        stats[r["label"]].append(r["best_val_acc"])
        if r["label"] not in meta:
            meta[r["label"]] = {
                k: r[k] for k in ["arch", "n_layers", "filters",
                                   "classifier", "dropout", "lr"]
                if k in r
            }

    summary = []
    for label, accs in stats.items():
        e = {
            "modelo"       : modelo,
            "num_classes"  : num_classes,
            "label"        : label,
            "folds_done"   : len(accs),
            "complete"     : len(accs) == K_FOLDS,
            "accs_per_fold": [round(a, 4) for a in accs],
            "mean_val_acc" : round(float(np.mean(accs)), 4),
            "std_val_acc"  : round(float(np.std(accs)),  4),
            "var_val_acc"  : round(float(np.var(accs)),  4),
            "min_val_acc"  : round(float(np.min(accs)),  4),
            "max_val_acc"  : round(float(np.max(accs)),  4),
        }
        e.update(meta.get(label, {}))
        summary.append(e)

    summary.sort(key=lambda x: (x["complete"], x["mean_val_acc"]),
                 reverse=True)
    with open(_summary_path(modelo), "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    return summary


def grupo_completo(modelo):
    """Retorna True se todos os folds de todas as arquiteturas estão prontos."""
    rp = _results_path(modelo)
    if not os.path.exists(rp):
        return False
    with open(rp) as f:
        data = json.load(f)
    done = {(r["label"], r["fold"]) for r in data}
    total = len(ARCHITECTURES) * K_FOLDS
    return len(done) == total

# ══════════════════════════════════════════════════════════════════════════════
# TREINO DE UM FOLD
# ══════════════════════════════════════════════════════════════════════════════

def train_one_fold(model, tr_loader, vl_loader, device,
                   lr, epochs, lr_patience, early_stop, lr_min):
    """
    Treina um fold com:
      • LR cai pela metade a cada `lr_patience` épocas sem melhora
      • Early stop após `early_stop` épocas sem melhora
    Retorna (best_val_acc, history)
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Scheduler manual — mais granular que ReduceLROnPlateau padrão
    best_val_acc      = 0.0
    best_state        = None
    epochs_no_improve = 0
    history           = []
    model.to(device)

    for epoch in range(epochs):

        # ── Treino ────────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_correct = tr_total = 0
        pbar = tqdm(tr_loader,
                    desc=f"    Ép {epoch+1:>3}/{epochs} [tr]",
                    leave=False, ncols=80)
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += (out.argmax(1) == yb).sum().item()
            tr_total   += yb.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # ── Validação ─────────────────────────────────────────────────────────
        model.eval()
        vl_loss = vl_correct = vl_total = 0
        with torch.no_grad():
            for xb, yb in vl_loader:
                xb, yb = xb.to(device), yb.to(device)
                out     = model(xb)
                loss    = criterion(out, yb)
                vl_loss    += loss.item()
                vl_correct += (out.argmax(1) == yb).sum().item()
                vl_total   += yb.size(0)

        tr_acc  = 100.0 * tr_correct / tr_total
        vl_acc  = 100.0 * vl_correct / vl_total
        tr_loss /= len(tr_loader)
        vl_loss /= len(vl_loader)
        cur_lr   = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch"     : epoch + 1,
            "train_loss": round(tr_loss, 4),
            "train_acc" : round(tr_acc,  2),
            "val_loss"  : round(vl_loss, 4),
            "val_acc"   : round(vl_acc,  2),
            "lr"        : cur_lr,
        })

        print(f"    Ép {epoch+1:>3}/{epochs} │ "
              f"tr={tr_loss:.4f}/{tr_acc:.2f}% │ "
              f"vl={vl_loss:.4f}/{vl_acc:.2f}% │ "
              f"lr={cur_lr:.2e}  "
              f"{'★' if vl_acc > best_val_acc else ''}")

        if vl_acc > best_val_acc:
            best_val_acc      = vl_acc
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            # LR cai pela metade a cada lr_patience épocas sem melhora
            if epochs_no_improve % lr_patience == 0:
                new_lr = max(cur_lr * 0.5, lr_min)
                if new_lr < cur_lr:
                    for pg in optimizer.param_groups:
                        pg["lr"] = new_lr
                    print(f"    ↘  LR reduzido: {cur_lr:.2e} → {new_lr:.2e}")

            # Early stop após early_stop épocas sem melhora
            if epochs_no_improve >= early_stop:
                print(f"    🛑 Early stopping na época {epoch+1} "
                      f"({early_stop} épocas sem melhora)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_acc, history

# ══════════════════════════════════════════════════════════════════════════════
# BUSCA PARA UM ÚNICO GRUPO
# ══════════════════════════════════════════════════════════════════════════════

def search_grupo(modelo, mod_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{sep}")
    print(f"  🔍  GRUPO: {modelo}  │  device: {device}")
    print(sep)

    # 1. HDF5
    print(f"\n[1/4] HDF5...")
    h5_loc, num_classes, group_indices, global_to_local = \
        ensure_hdf5(modelo, mod_classes)

    # 2. Split
    print(f"\n[2/4] Split de índices...")
    train_idx, val_idx, test_idx = ensure_split(modelo, h5_loc)

    # 3. Folds
    print(f"\n[3/4] Folds...")
    fold_splits = ensure_folds(modelo, h5_loc, train_idx)

    # 4. Busca
    print(f"\n[4/4] Busca de arquitetura  "
          f"({len(ARCHITECTURES)} arquiteturas × {K_FOLDS} folds)")

    all_results, done_set = load_results(modelo)

    total = len(ARCHITECTURES) * K_FOLDS
    done  = len(done_set)
    print(f"  Total jobs: {total}  │  Concluídos: {done}"
          f"  │  Restantes: {total - done}\n")

    best_mean = max(
        (r["mean_val_acc"] for r in update_summary(
            modelo, all_results, num_classes)
         if r.get("complete")),
        default=-1.0
    )

    for arch_idx, arch_entry in enumerate(ARCHITECTURES, 1):
        label = arch_entry["label"]
        arch  = arch_entry["arch"]

        folds_done = [f for f in range(K_FOLDS) if (label, f) in done_set]
        if len(folds_done) == K_FOLDS:
            accs = [r["best_val_acc"] for r in all_results
                    if r["label"] == label]
            print(f"  ⏭  [{arch_idx}/{len(ARCHITECTURES)}] "
                  f"{label}  (média={np.mean(accs):.2f}%)")
            continue

        print(f"\n{sep}")
        print(f"  [{arch_idx}/{len(ARCHITECTURES)}]  "
              f"{modelo}  │  {label}")
        print(f"  Filtros : " +
              " → ".join(str(b["out_channels"]) for b in arch))
        restantes = [f+1 for f in range(K_FOLDS)
                     if (label, f) not in done_set]
        print(f"  Folds restantes: {restantes}")
        print(sep)

        t0_arch = time.time()

        for fold_i, (tr_pos, vl_pos) in enumerate(fold_splits):

            if (label, fold_i) in done_set:
                acc = next(r["best_val_acc"] for r in all_results
                           if r["label"] == label and r["fold"] == fold_i)
                print(f"\n  Fold {fold_i+1}/{K_FOLDS} — ⏭  "
                      f"(val_acc={acc:.2f}%)")
                continue

            print(f"\n  ── Fold {fold_i+1}/{K_FOLDS} "
                  f"(treino={len(tr_pos):,}  val={len(vl_pos):,}) ──")

            # Seed determinístico e único por (arch, fold)
            fold_seed = SEED + arch_idx * 100 + fold_i
            random.seed(fold_seed)
            np.random.seed(fold_seed)
            torch.manual_seed(fold_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(fold_seed)

            g = torch.Generator()
            g.manual_seed(fold_seed)

            tr_abs = train_idx[tr_pos]
            vl_abs = train_idx[vl_pos]

            tr_loader = DataLoader(
                H5PyDataset(h5_loc, 'X', 'Y', 'Z', tr_abs, formats=0),
                batch_size=BATCH_SIZE, shuffle=True,
                num_workers=0, generator=g
            )
            vl_loader = DataLoader(
                H5PyDataset(h5_loc, 'X', 'Y', 'Z', vl_abs, formats=0),
                batch_size=BATCH_SIZE, shuffle=False, num_workers=0
            )

            model    = FlexCNN(num_classes=num_classes, arch=arch,
                               classifier=CLASSIFIER_HEAD, dropout=DROPOUT)
            n_params = sum(p.numel() for p in model.parameters()
                          if p.requires_grad)
            print(f"  Parâmetros: {n_params:,}")

            t0_fold = time.time()

            best_val_acc, history = train_one_fold(
                model      = model,
                tr_loader  = tr_loader,
                vl_loader  = vl_loader,
                device     = device,
                lr         = LEARNING_RATE,
                epochs     = EPOCHS_PER_FOLD,
                lr_patience= LR_PATIENCE,
                early_stop = EARLY_STOP,
                lr_min     = LR_MIN,
            )

            elapsed = time.time() - t0_fold
            print(f"\n  ✅ Fold {fold_i+1}/{K_FOLDS}  "
                  f"val_acc={best_val_acc:.2f}%  ({elapsed:.0f}s)")

            fold_result = {
                "modelo"      : modelo,
                "num_classes" : num_classes,
                "label"       : label,
                "arch_idx"    : arch_idx,
                "fold"        : fold_i,
                "best_val_acc": round(best_val_acc, 4),
                "elapsed_s"   : round(elapsed, 1),
                "n_params"    : n_params,
                "fold_seed"   : fold_seed,
                "arch"        : arch,
                "n_layers"    : len(arch),
                "filters"     : [b["out_channels"] for b in arch],
                "classifier"  : CLASSIFIER_HEAD,
                "dropout"     : DROPOUT,
                "lr"          : LEARNING_RATE,
                "lr_patience" : LR_PATIENCE,
                "early_stop"  : EARLY_STOP,
                "history"     : history,
            }

            # Salva imediatamente — não perde progresso
            save_fold_result(modelo, fold_result, all_results)
            done_set.add((label, fold_i))
            update_summary(modelo, all_results, num_classes)

            model.cpu()
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Estatísticas ao completar todos os folds da arquitetura
        accs_arch = [r["best_val_acc"] for r in all_results
                     if r["label"] == label]
        if len(accs_arch) == K_FOLDS:
            print(f"\n  📊 {modelo} │ {label}")
            print(f"     Folds     : {[round(a, 2) for a in accs_arch]}")
            print(f"     Média     : {np.mean(accs_arch):.2f}%")
            print(f"     Std       : {np.std(accs_arch):.2f}%")
            print(f"     Variância : {np.var(accs_arch):.4f}")
            print(f"     Tempo     : {time.time()-t0_arch:.0f}s")
            if np.mean(accs_arch) > best_mean:
                best_mean = np.mean(accs_arch)
                print(f"  🏆 Novo melhor: {label}  "
                      f"(média={best_mean:.2f}%)")

    # Sumário final do grupo
    summary = update_summary(modelo, all_results, num_classes)
    _print_summary(modelo, summary)
    return summary


def _print_summary(modelo, summary):
    print(f"\n{sep}")
    print(f"  🏆  RANKING — {modelo}")
    print(f"{'─'*65}")
    print(f"  {'Arquitetura':<42} {'Folds':>6} "
          f"{'Média':>8} {'Std':>7} {'Var':>8}")
    print(f"{'─'*65}")
    for s in summary[:5]:
        flag   = "✅" if s["complete"] else "⏳"
        status = f"{s['folds_done']}/{K_FOLDS}"
        print(f"  {flag} {s['label'][:40]:<40} {status:>6}"
              f" {s['mean_val_acc']:>7.2f}%"
              f" {s['std_val_acc']:>6.2f}%"
              f" {s['var_val_acc']:>8.4f}")
    print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# LOOP PRINCIPAL — percorre todos os grupos automaticamente
# ══════════════════════════════════════════════════════════════════════════════

def run_all():
    mod_classes = json.load(open(modulation_classes_path))

    print(f"\n{'#'*65}")
    print(f"  BUSCA AUTOMÁTICA — {len(GRUPOS_ALVO)} grupos")
    print(f"  Grupos: {GRUPOS_ALVO}")
    print(f"  LR patience: {LR_PATIENCE} épocas  │  "
          f"Early stop: {EARLY_STOP} épocas")
    print(f"{'#'*65}")

    for gi, modelo in enumerate(GRUPOS_ALVO, 1):

        # Verifica se o grupo já está completamente processado
        if grupo_completo(modelo):
            print(f"\n  ⏭  [{gi}/{len(GRUPOS_ALVO)}] {modelo} — "
                  f"já completo, pulando")

            # Mostra sumário do grupo já concluído
            sp = _summary_path(modelo)
            if os.path.exists(sp):
                with open(sp) as f:
                    summary = json.load(f)
                if summary:
                    best = next((s for s in summary if s.get("complete")),
                                summary[0])
                    print(f"     Melhor: {best['label']}"
                          f"  (média={best['mean_val_acc']:.2f}%)")
            continue

        print(f"\n{'#'*65}")
        print(f"  [{gi}/{len(GRUPOS_ALVO)}] Iniciando grupo: {modelo}")
        print(f"{'#'*65}")

        t0 = time.time()
        search_grupo(modelo, mod_classes)
        elapsed = time.time() - t0
        print(f"\n  ⏱  Grupo '{modelo}' concluído em "
              f"{elapsed/60:.1f} min")

    # Sumário global
    print(f"\n{'#'*65}")
    print(f"  ✅  BUSCA COMPLETA — todos os grupos processados")
    print(f"{'#'*65}")
    for modelo in GRUPOS_ALVO:
        sp = _summary_path(modelo)
        if os.path.exists(sp):
            with open(sp) as f:
                summary = json.load(f)
            best = next((s for s in summary if s.get("complete")),
                        None)
            if best:
                print(f"  {modelo:<6} → {best['label']:<45}"
                      f"  {best['mean_val_acc']:.2f}% "
                      f"± {best['std_val_acc']:.2f}%")


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ══════════════════════════════════════════════════════════════════════════════

run_all()


#################################################################
  BUSCA AUTOMÁTICA — 6 grupos
  Grupos: ['ASK', 'PSK', 'APSK', 'QAM', 'AM', 'FM']
  LR patience: 5 épocas  │  Early stop: 15 épocas
#################################################################

#################################################################
  [1/6] Iniciando grupo: ASK
#################################################################

═════════════════════════════════════════════════════════════════
  🔍  GRUPO: ASK  │  device: cpu
═════════════════════════════════════════════════════════════════

[1/4] HDF5...
  📥 Copiando HDF5 do Drive → /content/ASK_subset_snr_0_2_4_6_8_10_12_14_16_18_20_22_24_26_28_30.hdf5

[2/4] Split de índices...
  ✅ Split carregado do Drive  (treino=117,964  val=39,322  teste=39,322)

[3/4] Folds...
  ✅ Folds carregados do Drive: /content/drive/MyDrive/radioml_sessions/ASK/kfold_folds_k5.json

[4/4] Busca de arquitetura  (12 arquiteturas × 5 folds)
  ✅ 22 resultados anteri

    Ép   1/120 │ tr=0.8883/65.59% │ vl=0.4976/79.63% │ lr=1.00e-03  ★


KeyboardInterrupt: 